# gsplat test zemini: araştırma yöntemleri aynı kareler üzerinde

gsplat'in bakımlı referans eğiticisi (`examples/simple_trainer.py`, v1.5.3) her seferinde **tek bir ayar**
değiştirilerek eğitilir; hepsi önceki testlerle **aynı 93 saklanan fotoğrafta** (her 10. kare) aynı kodla ölçülür.

| varyant | ne değişir |
|---|---|
| `default` | gsplat varsayılanı (orijinal 3DGS yoğunlaştırma), 1920x1080 |
| `mcmc` | 3DGS-MCMC yoğunlaştırma, 1M splat sınırı |
| `absgrad` | AbsGS: mutlak gradyanla bölme (ince detay) |
| `antialiased` | Mip-Splatting'in 2D anti-aliasing filtresi |
| `pose_opt` | eğitim sırasında kamera pozlarını iyileştirme |
| `default_4k` | 4K tam çözünürlükte eğitim (en yavaşı) |
| `with_ut` | 3DGUT + MCMC (`mcmc` satırıyla karşılaştır); standart görüntüleyicideki hâli `with_ut (klasik)` satırında |
| `bilateral_grid` | kare başına renk/pozlama düzeltmesi eğitimde; ölçüm grid'siz PLY ile (PLY'ye taşınıyor mu?) |
| `depth_loss` | SfM noktalarından derinlik kaybı |
| `mcmc_pose_opt` | MCMC + poz iyileştirme |
| `absgrad_aa` | AbsGS + anti-aliasing |
| `exclude_bad_frames` | varsayılan ayarlar, ama poz hatalı bölümlerdeki **eğitim** kareleri eğitimden çıkarılmış |

Ayrıca: yakınındaki **eğitim** kareleri bile çöken (yanlış SfM pozu) saklanan kareler otomatik bulunur ve
"poz hatasız" ikinci bir ortalama raporlanır; app eğiticisiyle yapılan A/B sonuçları da aynı tabloya eklenir.

**Kullanım:** GPU (A100 önerilir) → **Run all**. 1080p varyant başına ~15-25 dk, 4K ~45-90 dk; 12 varyant toplam ~5-6 saat.
Her varyant bitince Drive'a yazılır; kopma olursa Run all biten varyantları atlar.

In [ ]:
# @title Ayarlar
DATASET_ZIP = "/content/drive/MyDrive/GaussianTests/inputs/IMG_5966_dataset.zip"  # @param {type:"string"}
OUT_DIR = "/content/drive/MyDrive/GaussianTests/gsplat_testbed/IMG_5966"  # @param {type:"string"}
APP_AB_DIR = "/content/drive/MyDrive/GaussianTests/training_ab/IMG_5966"  # @param {type:"string"}
VARIANTS = "default,mcmc,absgrad,antialiased,pose_opt,default_4k,with_ut,bilateral_grid,depth_loss,mcmc_pose_opt,absgrad_aa,exclude_bad_frames"  # @param {type:"string"}
STEPS = 30000          # @param {type:"integer"}
TEST_EVERY = 10        # @param {type:"integer"}
SAVE_PLY = True        # @param {type:"boolean"}
FACTOR_OVERRIDE = 0    # @param {type:"integer"}
# FACTOR_OVERRIDE > 0: hızlı deneme için bütün varyantlarda çözünürlük bölücüsü (ör. 4).
# APP_AB_DIR: app eğiticisinin A/B sonuçları (varsa tabloya 'app:' satırları olarak eklenir; yoksa boş bırak).
VARIANTS = [v.strip() for v in VARIANTS.split(",") if v.strip()]

In [ ]:
# @title Kurulum (gsplat eğiticisi + bağımlılıklar, ~5-10 dk)
import os, subprocess, sys
os.environ["MAX_JOBS"] = "4"
if not os.path.exists("/content/gsplat_src"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "v1.5.3",
                    "https://github.com/nerfstudio-project/gsplat.git", "/content/gsplat_src"], check=True)
# requirements.txt'teki numpy<2 pini kullanılmıyor (Colab'da kernel yeniden başlatma ister).
!pip install -q "gsplat==1.5.3" ninja tyro viser "imageio[ffmpeg]" scikit-learn tqdm "torchmetrics[image]" tensorboard tensorly pyyaml matplotlib splines lpips "git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e" "git+https://github.com/nerfstudio-project/nerfview@4538024fe0d15fd1a0e4d760f3695fc44ca72787"
# fused-ssim derlenirken torch'u görmeli: build isolation kapalı.
!pip install -q --no-build-isolation "git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# @title Kodu yaz (gömülü kaynaklar)
import json
from pathlib import Path

SOURCES = json.loads('{"lib/splat_restorer/__init__.py": "\\"\\"\\"Splat render restorer pilot (Difix3D-style). See README.md.\\"\\"\\"\\n", "lib/splat_restorer/baseline.py": "\\"\\"\\"Cheap baselines: identity, unsharp mask, and a small residual U-Net.\\n\\nIf the diffusion restorer cannot beat the small U-Net trained on the same\\npairs, its pretrained image prior is not earning its cost.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport math\\nimport random\\nimport time\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\n\\nfrom .data import PairDataset, collect_items, split_holdout\\nfrom .workspace import Workspace, atomic_copy, atomic_write_json\\n\\n\\ndef unsharp(img: torch.Tensor, amount: float = 0.6, sigma: float = 1.2) -> torch.Tensor:\\n    \\"\\"\\"img: (B, 3, H, W) in [0, 1].\\"\\"\\"\\n    k = int(2 * math.ceil(3 * sigma) + 1)\\n    x = torch.arange(k, device=img.device, dtype=img.dtype) - k // 2\\n    g = torch.exp(-(x**2) / (2 * sigma**2))\\n    g = g / g.sum()\\n    blur = F.conv2d(F.pad(img, (k // 2,) * 4, mode=\\"reflect\\"), g.view(1, 1, 1, k).repeat(3, 1, 1, 1), groups=3)\\n    blur = F.conv2d(blur, g.view(1, 1, k, 1).repeat(3, 1, 1, 1), groups=3)\\n    return (img + amount * (img - blur)).clamp(0, 1)\\n\\n\\nclass _Block(nn.Module):\\n    def __init__(self, c_in, c_out):\\n        super().__init__()\\n        self.net = nn.Sequential(\\n            nn.Conv2d(c_in, c_out, 3, padding=1), nn.GELU(), nn.Conv2d(c_out, c_out, 3, padding=1), nn.GELU()\\n        )\\n        self.skip = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()\\n\\n    def forward(self, x):\\n        return self.net(x) + self.skip(x)\\n\\n\\nclass SmallUNet(nn.Module):\\n    \\"\\"\\"~8M-parameter residual U-Net; optional reference image as extra input.\\"\\"\\"\\n\\n    def __init__(self, use_ref: bool = True, width: int = 48):\\n        super().__init__()\\n        c_in = 6 if use_ref else 3\\n        w = width\\n        self.use_ref = use_ref\\n        self.inc = _Block(c_in, w)\\n        self.d1 = _Block(w, 2 * w)\\n        self.d2 = _Block(2 * w, 4 * w)\\n        self.d3 = _Block(4 * w, 8 * w)\\n        self.u2 = _Block(8 * w + 4 * w, 4 * w)\\n        self.u1 = _Block(4 * w + 2 * w, 2 * w)\\n        self.u0 = _Block(2 * w + w, w)\\n        self.out = nn.Conv2d(w, 3, 3, padding=1)\\n        nn.init.zeros_(self.out.weight)\\n        nn.init.zeros_(self.out.bias)\\n\\n    def forward(self, deg, ref=None):\\n        x = torch.cat([deg, ref], 1) if self.use_ref else deg\\n        e0 = self.inc(x)\\n        e1 = self.d1(F.avg_pool2d(e0, 2))\\n        e2 = self.d2(F.avg_pool2d(e1, 2))\\n        e3 = self.d3(F.avg_pool2d(e2, 2))\\n        up = lambda t, ref_t: F.interpolate(t, size=ref_t.shape[-2:], mode=\\"bilinear\\", align_corners=False)\\n        d2 = self.u2(torch.cat([up(e3, e2), e2], 1))\\n        d1 = self.u1(torch.cat([up(d2, e1), e1], 1))\\n        d0 = self.u0(torch.cat([up(d1, e0), e0], 1))\\n        return (deg + self.out(d0)).clamp(0, 1)\\n\\n    @torch.no_grad()\\n    def restore(self, deg, ref=None):\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n            return self.forward(deg, ref).float().clamp(0, 1)\\n\\n\\n@dataclass\\nclass BaselineConfig:\\n    run_name: str\\n    scenes: list[str]\\n    use_ref: bool = True\\n    steps: int = 20_000\\n    batch_size: int = 16\\n    lr: float = 3e-4\\n    crop: int = 256\\n    lambda_lpips: float = 0.2\\n    num_workers: int = 8\\n    seed: int = 0\\n\\n\\ndef train_baseline(ws: Workspace, cfg: BaselineConfig, pairs_root: Path, device=\\"cuda\\", log=print) -> Path:\\n    import lpips\\n\\n    run_dir = ws.runs_dir / cfg.run_name\\n    run_dir.mkdir(parents=True, exist_ok=True)\\n    final_path = run_dir / \\"model_final.pt\\"\\n    if final_path.exists():\\n        log(f\\"[{cfg.run_name}] already finished -> {final_path}\\")\\n        return final_path\\n    atomic_write_json(run_dir / \\"config.json\\", asdict(cfg))\\n    random.seed(cfg.seed)\\n    torch.manual_seed(cfg.seed)\\n\\n    items = collect_items(pairs_root, cfg.scenes)\\n    train_items, _ = split_holdout(items, frac=0.02, seed=cfg.seed)\\n    loader = torch.utils.data.DataLoader(\\n        PairDataset(train_items, train=True, crop=cfg.crop), batch_size=cfg.batch_size, shuffle=True,\\n        num_workers=cfg.num_workers, drop_last=True, pin_memory=True, persistent_workers=cfg.num_workers > 0,\\n    )\\n    net = SmallUNet(use_ref=cfg.use_ref).to(device)\\n    opt = torch.optim.AdamW(net.parameters(), lr=cfg.lr, weight_decay=1e-4)\\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, cfg.steps)\\n    net_lpips = lpips.LPIPS(net=\\"vgg\\", verbose=False).to(device).eval().requires_grad_(False)\\n    log(f\\"[{cfg.run_name}] {sum(p.numel() for p in net.parameters()) / 1e6:.1f}M params, {len(train_items)} pairs\\")\\n\\n    log_f = open(run_dir / \\"log.csv\\", \\"w\\", newline=\\"\\")\\n    writer = csv.writer(log_f)\\n    writer.writerow([\\"step\\", \\"loss\\", \\"sec_per_step\\"])\\n    step, t_last = 0, time.time()\\n    while step < cfg.steps:\\n        for batch in loader:\\n            deg, gt, ref = (batch[k].to(device, non_blocking=True) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n            with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n                out = net(deg, ref)\\n                loss = (out.float() - gt).abs().mean()\\n                loss = loss + cfg.lambda_lpips * net_lpips(out.float() * 2 - 1, gt * 2 - 1).float().mean()\\n            opt.zero_grad(set_to_none=True)\\n            loss.backward()\\n            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)\\n            opt.step()\\n            sched.step()\\n            step += 1\\n            if step % 200 == 0:\\n                dt = (time.time() - t_last) / 200\\n                t_last = time.time()\\n                writer.writerow([step, f\\"{loss.item():.5f}\\", f\\"{dt:.3f}\\"])\\n                log_f.flush()\\n                if step % 2000 == 0:\\n                    log(f\\"[{cfg.run_name}] step {step}/{cfg.steps} loss {loss.item():.4f}\\")\\n            if step >= cfg.steps:\\n                break\\n    log_f.close()\\n    tmp = ws.local / f\\"{cfg.run_name}_final.pt\\"\\n    torch.save({\\"state_dict\\": net.state_dict(), \\"use_ref\\": cfg.use_ref}, tmp)\\n    atomic_copy(tmp, final_path)\\n    atomic_write_json(run_dir / \\"DONE.json\\", {\\"steps\\": step})\\n    log(f\\"[{cfg.run_name}] finished -> {final_path}\\")\\n    return final_path\\n\\n\\ndef load_baseline(path: Path, device=\\"cuda\\") -> SmallUNet:\\n    sd = torch.load(path, map_location=\\"cpu\\", weights_only=False)\\n    net = SmallUNet(use_ref=sd[\\"use_ref\\"]).to(device)\\n    net.load_state_dict(sd[\\"state_dict\\"])\\n    return net.eval()\\n", "lib/splat_restorer/colmap_io.py": "\\"\\"\\"Minimal COLMAP sparse-model reader (binary and text).\\n\\nOnly what the pair generator needs: cameras, registered images (pose + name)\\nand the 3D points used to initialise the Gaussians.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport struct\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\n# model_id -> (name, number of params)\\nCAMERA_MODELS = {\\n    0: (\\"SIMPLE_PINHOLE\\", 3),\\n    1: (\\"PINHOLE\\", 4),\\n    2: (\\"SIMPLE_RADIAL\\", 4),\\n    3: (\\"RADIAL\\", 5),\\n    4: (\\"OPENCV\\", 8),\\n    5: (\\"OPENCV_FISHEYE\\", 8),\\n    6: (\\"FULL_OPENCV\\", 12),\\n    7: (\\"FOV\\", 5),\\n    8: (\\"SIMPLE_RADIAL_FISHEYE\\", 4),\\n    9: (\\"RADIAL_FISHEYE\\", 5),\\n    10: (\\"THIN_PRISM_FISHEYE\\", 12),\\n}\\n_NAME_TO_COUNT = {name: count for name, count in CAMERA_MODELS.values()}\\n\\n\\n@dataclass\\nclass Camera:\\n    id: int\\n    model: str\\n    width: int\\n    height: int\\n    params: np.ndarray\\n\\n    def intrinsics(self) -> tuple[float, float, float, float]:\\n        \\"\\"\\"Return (fx, fy, cx, cy); distortion terms are ignored.\\"\\"\\"\\n        p = self.params\\n        if self.model in (\\"SIMPLE_PINHOLE\\", \\"SIMPLE_RADIAL\\", \\"RADIAL\\", \\"SIMPLE_RADIAL_FISHEYE\\", \\"RADIAL_FISHEYE\\", \\"FOV\\"):\\n            return float(p[0]), float(p[0]), float(p[1]), float(p[2])\\n        return float(p[0]), float(p[1]), float(p[2]), float(p[3])\\n\\n    def has_distortion(self) -> bool:\\n        if self.model in (\\"SIMPLE_PINHOLE\\", \\"PINHOLE\\"):\\n            return False\\n        extra = self.params[3:] if self.model.startswith(\\"SIMPLE\\") or self.model in (\\"RADIAL\\", \\"FOV\\") else self.params[4:]\\n        return bool(np.any(np.abs(extra) > 1e-9))\\n\\n\\n@dataclass\\nclass Image:\\n    id: int\\n    qvec: np.ndarray  # w, x, y, z (world -> camera)\\n    tvec: np.ndarray\\n    camera_id: int\\n    name: str\\n\\n    def world_to_camera(self) -> np.ndarray:\\n        m = np.eye(4)\\n        m[:3, :3] = qvec_to_rotmat(self.qvec)\\n        m[:3, 3] = self.tvec\\n        return m\\n\\n\\n@dataclass\\nclass SparseModel:\\n    cameras: dict[int, Camera]\\n    images: dict[int, Image]\\n    points_xyz: np.ndarray  # (N, 3) float64\\n    points_rgb: np.ndarray  # (N, 3) uint8\\n\\n\\ndef qvec_to_rotmat(q: np.ndarray) -> np.ndarray:\\n    w, x, y, z = (float(v) for v in q)\\n    return np.array(\\n        [\\n            [1 - 2 * y * y - 2 * z * z, 2 * x * y - 2 * w * z, 2 * z * x + 2 * w * y],\\n            [2 * x * y + 2 * w * z, 1 - 2 * x * x - 2 * z * z, 2 * y * z - 2 * w * x],\\n            [2 * z * x - 2 * w * y, 2 * y * z + 2 * w * x, 1 - 2 * x * x - 2 * y * y],\\n        ]\\n    )\\n\\n\\ndef _read(f, fmt: str):\\n    size = struct.calcsize(\\"<\\" + fmt)\\n    data = f.read(size)\\n    if len(data) != size:\\n        raise ValueError(\\"Truncated COLMAP binary file\\")\\n    return struct.unpack(\\"<\\" + fmt, data)\\n\\n\\ndef _read_cameras_bin(path: Path) -> dict[int, Camera]:\\n    cams = {}\\n    with open(path, \\"rb\\") as f:\\n        (n,) = _read(f, \\"Q\\")\\n        for _ in range(n):\\n            cam_id, model_id, width, height = _read(f, \\"iiQQ\\")\\n            name, count = CAMERA_MODELS[model_id]\\n            params = np.array(_read(f, \\"d\\" * count))\\n            cams[cam_id] = Camera(cam_id, name, int(width), int(height), params)\\n    return cams\\n\\n\\ndef _read_images_bin(path: Path) -> dict[int, Image]:\\n    imgs = {}\\n    with open(path, \\"rb\\") as f:\\n        (n,) = _read(f, \\"Q\\")\\n        for _ in range(n):\\n            props = _read(f, \\"idddddddi\\")\\n            image_id = props[0]\\n            qvec = np.array(props[1:5])\\n            tvec = np.array(props[5:8])\\n            camera_id = props[8]\\n            name = b\\"\\"\\n            while True:\\n                c = f.read(1)\\n                if c == b\\"\\\\x00\\":\\n                    break\\n                if not c:\\n                    raise ValueError(\\"Truncated image name\\")\\n                name += c\\n            (n2d,) = _read(f, \\"Q\\")\\n            f.seek(24 * n2d, 1)  # x, y (double) + point3D id (int64)\\n            imgs[image_id] = Image(image_id, qvec, tvec, camera_id, name.decode(\\"utf-8\\"))\\n    return imgs\\n\\n\\ndef _read_points_bin(path: Path) -> tuple[np.ndarray, np.ndarray]:\\n    with open(path, \\"rb\\") as f:\\n        (n,) = _read(f, \\"Q\\")\\n        xyz = np.empty((n, 3), np.float64)\\n        rgb = np.empty((n, 3), np.uint8)\\n        for i in range(n):\\n            props = _read(f, \\"QdddBBBd\\")\\n            xyz[i] = props[1:4]\\n            rgb[i] = props[4:7]\\n            (track_len,) = _read(f, \\"Q\\")\\n            f.seek(8 * track_len, 1)\\n    return xyz, rgb\\n\\n\\ndef _read_cameras_txt(path: Path) -> dict[int, Camera]:\\n    cams = {}\\n    for line in path.read_text(encoding=\\"utf-8\\").splitlines():\\n        if not line.strip() or line.startswith(\\"#\\"):\\n            continue\\n        parts = line.split()\\n        cam_id, model, w, h = int(parts[0]), parts[1], int(parts[2]), int(parts[3])\\n        params = np.array([float(v) for v in parts[4 : 4 + _NAME_TO_COUNT[model]]])\\n        cams[cam_id] = Camera(cam_id, model, w, h, params)\\n    return cams\\n\\n\\ndef _read_images_txt(path: Path) -> dict[int, Image]:\\n    imgs = {}\\n    lines = [l for l in path.read_text(encoding=\\"utf-8\\").splitlines() if not l.startswith(\\"#\\")]\\n    # Pose lines alternate with 2D-point lines (which may be empty).\\n    for i in range(0, len(lines), 2):\\n        parts = lines[i].split()\\n        if len(parts) < 10:\\n            continue\\n        image_id = int(parts[0])\\n        qvec = np.array([float(v) for v in parts[1:5]])\\n        tvec = np.array([float(v) for v in parts[5:8]])\\n        imgs[image_id] = Image(image_id, qvec, tvec, int(parts[8]), \\" \\".join(parts[9:]))\\n    return imgs\\n\\n\\ndef _read_points_txt(path: Path) -> tuple[np.ndarray, np.ndarray]:\\n    xyz, rgb = [], []\\n    for line in path.read_text(encoding=\\"utf-8\\").splitlines():\\n        if not line.strip() or line.startswith(\\"#\\"):\\n            continue\\n        parts = line.split()\\n        xyz.append([float(v) for v in parts[1:4]])\\n        rgb.append([int(v) for v in parts[4:7]])\\n    return np.array(xyz, np.float64).reshape(-1, 3), np.array(rgb, np.uint8).reshape(-1, 3)\\n\\n\\ndef read_sparse_model(sparse_dir: Path) -> SparseModel:\\n    sparse_dir = Path(sparse_dir)\\n    if (sparse_dir / \\"cameras.bin\\").exists():\\n        cams = _read_cameras_bin(sparse_dir / \\"cameras.bin\\")\\n        imgs = _read_images_bin(sparse_dir / \\"images.bin\\")\\n        xyz, rgb = _read_points_bin(sparse_dir / \\"points3D.bin\\")\\n    elif (sparse_dir / \\"cameras.txt\\").exists():\\n        cams = _read_cameras_txt(sparse_dir / \\"cameras.txt\\")\\n        imgs = _read_images_txt(sparse_dir / \\"images.txt\\")\\n        xyz, rgb = _read_points_txt(sparse_dir / \\"points3D.txt\\")\\n    else:\\n        raise FileNotFoundError(f\\"No COLMAP model in {sparse_dir}\\")\\n    return SparseModel(cams, imgs, xyz, rgb)\\n", "lib/splat_restorer/data.py": "\\"\\"\\"Pair datasets read from extracted pair tars (see pairs.py).\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport random\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom PIL import Image\\n\\n\\ndef collect_items(pairs_root: Path, scenes: list[str], recipes: list[str] | None = None) -> list[dict]:\\n    items = []\\n    for s in scenes:\\n        index = json.loads((pairs_root / s / \\"index.json\\").read_text(encoding=\\"utf-8\\"))\\n        for rname, rec in index[\\"recipes\\"].items():\\n            if recipes and rname not in recipes:\\n                continue\\n            for it in rec[\\"items\\"]:\\n                items.append(\\n                    {\\n                        \\"scene\\": s,\\n                        \\"recipe\\": rname,\\n                        \\"frame\\": it[\\"frame\\"],\\n                        \\"deg\\": str(pairs_root / s / it[\\"deg\\"]),\\n                        \\"gt\\": str(pairs_root / s / it[\\"gt\\"]),\\n                        \\"ref\\": str(pairs_root / s / it[\\"ref_path\\"]),\\n                    }\\n                )\\n    return items\\n\\n\\ndef _load(path: str) -> torch.Tensor:\\n    arr = np.array(Image.open(path).convert(\\"RGB\\"), dtype=np.uint8)\\n    return torch.from_numpy(arr).permute(2, 0, 1).float() / 255.0\\n\\n\\ndef eval_crop_box(h: int, w: int, multiple: int = 64) -> tuple[int, int, int, int]:\\n    \\"\\"\\"Centred crop whose sides are multiples of ``multiple`` (top, left, h, w).\\"\\"\\"\\n    ch, cw = h - h % multiple, w - w % multiple\\n    return (h - ch) // 2, (w - cw) // 2, ch, cw\\n\\n\\nclass PairDataset(torch.utils.data.Dataset):\\n    \\"\\"\\"Returns deg / gt / ref tensors in [0, 1], shape (3, H, W).\\n\\n    Training: one random ``crop`` x ``crop`` window (shared by all three images,\\n    the reference is a nearby view so the window roughly corresponds) plus a\\n    random horizontal flip. Evaluation: deterministic centred crop to multiples\\n    of 64 (540x960 -> 512x960).\\n    \\"\\"\\"\\n\\n    def __init__(self, items: list[dict], train: bool, crop: int = 512):\\n        self.items = items\\n        self.train = train\\n        self.crop = crop\\n\\n    def __len__(self):\\n        return len(self.items)\\n\\n    def __getitem__(self, idx):\\n        it = self.items[idx]\\n        deg, gt, ref = _load(it[\\"deg\\"]), _load(it[\\"gt\\"]), _load(it[\\"ref\\"])\\n        _, h, w = deg.shape\\n        if self.train:\\n            c = min(self.crop, h - h % 8, w - w % 8)\\n            top = random.randint(0, h - c)\\n            left = random.randint(0, w - c)\\n            sl = (slice(None), slice(top, top + c), slice(left, left + c))\\n            deg, gt = deg[sl], gt[sl]\\n            rh, rw = ref.shape[1:]\\n            rt, rl = min(top, rh - c), min(left, rw - c)\\n            ref = ref[:, rt : rt + c, rl : rl + c]\\n            if random.random() < 0.5:\\n                deg, gt, ref = deg.flip(-1), gt.flip(-1), ref.flip(-1)\\n        else:\\n            top, left, ch, cw = eval_crop_box(h, w)\\n            sl = (slice(None), slice(top, top + ch), slice(left, left + cw))\\n            deg, gt, ref = deg[sl], gt[sl], ref[sl]\\n        return {\\"deg\\": deg, \\"gt\\": gt, \\"ref\\": ref, \\"idx\\": idx}\\n\\n\\ndef split_holdout(items: list[dict], frac: float = 0.02, seed: int = 0) -> tuple[list[dict], list[dict]]:\\n    \\"\\"\\"Small validation slice for loss curves (same scenes; not the real test).\\"\\"\\"\\n    rng = random.Random(seed)\\n    order = list(range(len(items)))\\n    rng.shuffle(order)\\n    n_val = max(1, int(len(items) * frac))\\n    val = [items[i] for i in order[:n_val]]\\n    train = [items[i] for i in order[n_val:]]\\n    return train, val\\n", "lib/splat_restorer/distill.py": "\\"\\"\\"Distill-back test: does restoring novel views actually improve the *splat*?\\n\\nFor each test scene and recipe, starting from the same weak splat:\\n\\n* control: fine-tune on the real training photos only;\\n* <method>: fine-tune on the training photos plus restored renders of the\\n  pair-frame poses (pseudo ground truth, Difix3D-style).\\n\\nBoth are scored on the scene\'s eval frames, which no splat ever trained on.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image\\n\\nfrom .gs_trainer import TrainConfig, View, params_from_state, render, ssim, train_gaussians\\nfrom .splits import subsample\\nfrom .workspace import Workspace\\n\\n\\ndef _load_photo(path: Path, device) -> torch.Tensor:\\n    return torch.from_numpy(np.asarray(Image.open(path).convert(\\"RGB\\"), dtype=np.float32) / 255.0).to(device)\\n\\n\\ndef pad_to_multiple(x: torch.Tensor, m: int = 64) -> tuple[torch.Tensor, tuple[int, int]]:\\n    \\"\\"\\"x: (B, 3, H, W). Reflect-pad bottom/right to multiples of m.\\"\\"\\"\\n    h, w = x.shape[-2:]\\n    ph, pw = (-h) % m, (-w) % m\\n    return F.pad(x, (0, pw, 0, ph), mode=\\"reflect\\"), (h, w)\\n\\n\\ndef restore_full(fn, deg_hwc: torch.Tensor, ref_hwc: torch.Tensor) -> torch.Tensor:\\n    deg = deg_hwc.permute(2, 0, 1)[None]\\n    ref = ref_hwc.permute(2, 0, 1)[None]\\n    deg_p, (h, w) = pad_to_multiple(deg)\\n    ref_p, _ = pad_to_multiple(ref)\\n    out = fn(deg_p, ref_p).float()[..., :h, :w]\\n    return out[0].permute(1, 2, 0).clamp(0, 1)\\n\\n\\n@torch.no_grad()\\ndef _score(params, frames, photos, Ks, w2c, width, height, net_lpips) -> dict:\\n    ps, ss, ls = [], [], []\\n    for i in frames:\\n        img, _ = render(params, w2c[i], Ks[i], width, height, 3)\\n        gt = photos[i]\\n        ps.append(10 * math.log10(1 / max(F.mse_loss(img, gt).item(), 1e-12)))\\n        ss.append(ssim(img, gt).item())\\n        ls.append(net_lpips(img.permute(2, 0, 1)[None] * 2 - 1, gt.permute(2, 0, 1)[None] * 2 - 1).item())\\n    return {\\"psnr\\": float(np.mean(ps)), \\"ssim\\": float(np.mean(ss)), \\"lpips\\": float(np.mean(ls))}\\n\\n\\ndef distill_scene(\\n    ws: Workspace,\\n    pairs_root: Path,\\n    scene: str,\\n    recipe: str,\\n    methods: dict,\\n    finetune_steps: int = 1500,\\n    max_pseudo: int = 60,\\n    pseudo_share: float = 0.5,\\n    device=\\"cuda\\",\\n    net_lpips=None,\\n    log=print,\\n) -> dict:\\n    index = json.loads((pairs_root / scene / \\"index.json\\").read_text(encoding=\\"utf-8\\"))\\n    rec = index[\\"recipes\\"][recipe]\\n    width, height = index[\\"width\\"], index[\\"height\\"]\\n    Ks = torch.tensor(index[\\"cameras\\"][\\"K\\"], device=device)\\n    w2c = torch.tensor(index[\\"cameras\\"][\\"w2c\\"], device=device)\\n    root = pairs_root / scene\\n    eval_photos = {i: _load_photo(root / \\"eval\\" / f\\"{i:05d}.jpg\\", device) for i in rec[\\"eval\\"]}\\n    train_photos = {i: _load_photo(root / recipe / \\"ref\\" / f\\"{i:05d}.jpg\\", device) for i in rec[\\"train\\"]}\\n    start = torch.load(ws.splat_path(scene, recipe), map_location=\\"cpu\\", weights_only=False)\\n    ft_cfg = TrainConfig(steps=finetune_steps, densify=False, lr_means=1.6e-5, lr_means_final_frac=0.1, seed=0)\\n    score = lambda p: _score(p, rec[\\"eval\\"], eval_photos, Ks, w2c, width, height, net_lpips)\\n\\n    results = {\\"start\\": score(params_from_state(start, device))}\\n    train_views = [View(train_photos[i], w2c[i], Ks[i]) for i in rec[\\"train\\"]]\\n\\n    params = train_gaussians(train_views, width, height, ft_cfg, params=params_from_state(start, device))\\n    results[\\"control\\"] = score(params)\\n    del params\\n\\n    pseudo_frames = subsample(rec[\\"pair\\"], max_pseudo)\\n    from .splits import nearest_reference\\n\\n    w2c_np = np.array(index[\\"cameras\\"][\\"w2c\\"])\\n    centers = -np.einsum(\\"fij,fi->fj\\", w2c_np[:, :3, :3], w2c_np[:, :3, 3])\\n    forwards = w2c_np[:, 2, :3]\\n    base = params_from_state(start, device)\\n    with torch.no_grad():\\n        renders = {i: render(base, w2c[i], Ks[i], width, height, 3)[0] for i in pseudo_frames}\\n    del base\\n\\n    for name, loader in methods.items():\\n        fn = loader()\\n        with torch.no_grad():\\n            pseudo = {}\\n            for i in pseudo_frames:\\n                ref = train_photos[nearest_reference(i, rec[\\"train\\"], centers, forwards)]\\n                pseudo[i] = restore_full(fn, renders[i], ref)\\n        del fn\\n        torch.cuda.empty_cache()\\n        # Pseudo views together get ``pseudo_share`` of the samples.\\n        w_pseudo = pseudo_share / (1 - pseudo_share) * len(train_views) / max(len(pseudo), 1)\\n        views = train_views + [View(pseudo[i], w2c[i], Ks[i], weight=w_pseudo) for i in pseudo_frames]\\n        params = train_gaussians(views, width, height, ft_cfg, params=params_from_state(start, device))\\n        results[name] = score(params)\\n        del params, pseudo\\n        torch.cuda.empty_cache()\\n    log(f\\"  {scene[:12]} {recipe}: \\" + \\", \\".join(f\\"{k} {v[\'psnr\']:.2f}/{v[\'lpips\']:.3f}\\" for k, v in results.items()))\\n    return results\\n\\n\\ndef run_distill(\\n    ws: Workspace, pairs_root: Path, test_scenes: list[str], recipes: list[str], methods: dict, tag: str,\\n    finetune_steps: int = 1500, device=\\"cuda\\", log=print,\\n) -> Path:\\n    import lpips\\n\\n    net_lpips = lpips.LPIPS(net=\\"alex\\", verbose=False).to(device).eval()\\n    out_dir = ws.results_dir / tag\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    partial = out_dir / \\"per_scene.json\\"\\n    all_res = json.loads(partial.read_text(encoding=\\"utf-8\\")) if partial.exists() else {}\\n    for scene in test_scenes:\\n        for recipe in recipes:\\n            key = f\\"{scene}/{recipe}\\"\\n            if key in all_res:\\n                continue\\n            if not ws.splat_path(scene, recipe).exists():\\n                log(f\\"  skip {key}: no saved splat\\")\\n                continue\\n            all_res[key] = distill_scene(\\n                ws, pairs_root, scene, recipe, methods, finetune_steps=finetune_steps, device=device,\\n                net_lpips=net_lpips, log=log,\\n            )\\n            partial.write_text(json.dumps(all_res, indent=2), encoding=\\"utf-8\\")\\n\\n    lines = []\\n    for recipe in recipes:\\n        keys = [k for k in all_res if k.endswith(\\"/\\" + recipe)]\\n        if not keys:\\n            continue\\n        names = list(all_res[keys[0]])\\n        lines += [f\\"\\\\n### {recipe} ({len(keys)} scenes)\\\\n\\", \\"| condition | PSNR ↑ | Δ vs control | SSIM ↑ | LPIPS ↓ | Δ vs control | wins vs control |\\", \\"|---|---|---|---|---|---|---|\\"]\\n        ctrl = {m: np.mean([all_res[k][\\"control\\"][m] for k in keys]) for m in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n        for n in names:\\n            mean = {m: np.mean([all_res[k][n][m] for k in keys]) for m in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n            wins = sum(all_res[k][n][\\"psnr\\"] > all_res[k][\\"control\\"][\\"psnr\\"] for k in keys)\\n            lines.append(\\n                f\\"| {n} | {mean[\'psnr\']:.2f} | {mean[\'psnr\'] - ctrl[\'psnr\']:+.2f} | {mean[\'ssim\']:.4f} | \\"\\n                f\\"{mean[\'lpips\']:.4f} | {mean[\'lpips\'] - ctrl[\'lpips\']:+.4f} | {wins}/{len(keys)} |\\"\\n            )\\n    md = \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return out_dir\\n", "lib/splat_restorer/dl3dv.py": "\\"\\"\\"DL3DV-Benchmark access: scene listing, deterministic split, per-scene download.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport io\\nimport random\\nimport shutil\\nfrom pathlib import Path\\n\\nREPO = \\"DL3DV/DL3DV-Benchmark\\"\\nSCENE_SUBDIR = \\"gaussian_splat\\"  # undistorted images + COLMAP model\\nIMAGES = \\"images_4\\"  # 960x540\\n\\n\\ndef list_scenes() -> list[str]:\\n    from huggingface_hub import HfApi\\n\\n    entries = HfApi().list_repo_tree(REPO, repo_type=\\"dataset\\", recursive=False)\\n    return sorted(e.path for e in entries if len(e.path) == 64 and all(c in \\"0123456789abcdef\\" for c in e.path))\\n\\n\\ndef scene_labels() -> dict[str, str]:\\n    \\"\\"\\"scene hash -> coarse label (e.g. indoor/outdoor) from benchmark-meta.csv, if available.\\"\\"\\"\\n    from huggingface_hub import hf_hub_download\\n\\n    try:\\n        path = hf_hub_download(REPO, \\"benchmark-meta.csv\\", repo_type=\\"dataset\\")\\n    except Exception:\\n        return {}\\n    text = Path(path).read_text(encoding=\\"utf-8\\", errors=\\"replace\\")\\n    rows = list(csv.DictReader(io.StringIO(text)))\\n    if not rows:\\n        return {}\\n    hash_key = next((k for k in rows[0] if \\"hash\\" in k.lower()), None)\\n    # \\"environment\\" is bounded / unbounded (roughly indoor-ish vs. open scenes).\\n    label_key = next((k for k in rows[0] if k.lower() in (\\"environment\\", \\"env\\", \\"label\\")), None)\\n    if hash_key is None:\\n        return {}\\n    return {r[hash_key].strip(): (r.get(label_key) or \\"\\").strip() if label_key else \\"\\" for r in rows}\\n\\n\\ndef make_split(scenes: list[str], n_test: int, seed: int = 1234, labels: dict[str, str] | None = None) -> dict:\\n    \\"\\"\\"Shuffle once; n_test scenes become test, the rest an ordered train pool.\\n\\n    Test scenes are drawn round-robin over ``labels`` (e.g. bounded/unbounded)\\n    so both kinds are represented. Train runs with N scenes always take the\\n    first N of the pool, so a 10-scene run is a subset of the 25-scene run\\n    (clean scaling curve).\\n    \\"\\"\\"\\n    rng = random.Random(seed)\\n    order = list(scenes)\\n    rng.shuffle(order)\\n    labels = labels or {}\\n    groups: dict[str, list[str]] = {}\\n    for s in order:\\n        groups.setdefault(labels.get(s, \\"\\"), []).append(s)\\n    test: list[str] = []\\n    keys = sorted(groups)\\n    while len(test) < min(n_test, len(order)):\\n        for k in keys:\\n            if groups[k] and len(test) < n_test:\\n                test.append(groups[k].pop(0))\\n    test_set = set(test)\\n    return {\\n        \\"seed\\": seed,\\n        \\"test\\": test,\\n        \\"train_pool\\": [s for s in order if s not in test_set],\\n        \\"labels\\": {s: labels.get(s, \\"\\") for s in order},\\n    }\\n\\n\\ndef download_scene(scene: str, dest: Path) -> Path:\\n    \\"\\"\\"Download images_4 + sparse model of one scene. Returns the scene dir.\\"\\"\\"\\n    from huggingface_hub import snapshot_download\\n\\n    dest.mkdir(parents=True, exist_ok=True)\\n    snapshot_download(\\n        REPO,\\n        repo_type=\\"dataset\\",\\n        local_dir=str(dest),\\n        allow_patterns=[f\\"{scene}/{SCENE_SUBDIR}/sparse/0/*\\", f\\"{scene}/{SCENE_SUBDIR}/{IMAGES}/*\\"],\\n        max_workers=16,\\n    )\\n    scene_dir = dest / scene / SCENE_SUBDIR\\n    if not (scene_dir / IMAGES).is_dir():\\n        raise FileNotFoundError(f\\"Download of {scene} produced no {IMAGES} folder\\")\\n    return scene_dir\\n\\n\\ndef remove_download(dest: Path) -> None:\\n    shutil.rmtree(dest, ignore_errors=True)\\n", "lib/splat_restorer/evaluate.py": "\\"\\"\\"Image-level evaluation on held-out *test scenes* (never seen in training).\\n\\nMetrics: PSNR, SSIM, LPIPS-Alex (training used LPIPS-VGG, so the evaluation\\nmetric is not the one the models optimised). A method only \\"works\\" if it\\nimproves LPIPS *without* lowering PSNR much: diffusion models can hallucinate\\nplausible texture that looks sharper yet is wrong.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport json\\nimport math\\nimport random\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\n\\nfrom .data import PairDataset, collect_items\\nfrom .gs_trainer import ssim\\nfrom .workspace import Workspace\\n\\n\\ndef _metrics(out: torch.Tensor, gt: torch.Tensor, net_lpips) -> dict:\\n    mse = F.mse_loss(out, gt).item()\\n    return {\\n        \\"psnr\\": 10 * math.log10(1 / max(mse, 1e-12)),\\n        \\"ssim\\": ssim(out[0].permute(1, 2, 0), gt[0].permute(1, 2, 0)).item(),\\n        \\"lpips\\": net_lpips(out * 2 - 1, gt * 2 - 1).mean().item(),\\n    }\\n\\n\\ndef build_methods(ws: Workspace, restorer_runs: list[str], baseline_runs: list[str], device=\\"cuda\\") -> dict:\\n    \\"\\"\\"name -> zero-arg loader returning callable(deg, ref) -> out (all [0, 1], Bx3xHxW).\\"\\"\\"\\n    from .baseline import load_baseline, unsharp\\n\\n    methods = {\\n        \\"input\\": lambda: (lambda d, r: d),\\n        \\"unsharp\\": lambda: (lambda d, r: unsharp(d)),\\n    }\\n    for run in baseline_runs:\\n        path = ws.runs_dir / run / \\"model_final.pt\\"\\n        if path.exists():\\n            methods[run] = (lambda p=path: (lambda net: (lambda d, r: net.restore(d, r if net.use_ref else None)))(load_baseline(p, device)))\\n    for run in restorer_runs:\\n        path = ws.runs_dir / run / \\"model_final.pt\\"\\n        if path.exists():\\n            def loader(p=path):\\n                from .train_restorer import load_restorer\\n\\n                model, use_ref = load_restorer(p, device)\\n                return lambda d, r: model.restore(d, r if use_ref else None)\\n\\n            methods[run] = loader\\n    return methods\\n\\n\\n@torch.no_grad()\\ndef evaluate(\\n    ws: Workspace,\\n    pairs_root: Path,\\n    test_scenes: list[str],\\n    methods: dict,\\n    tag: str,\\n    max_items_per_scene: int = 40,\\n    n_sheets: int = 8,\\n    device=\\"cuda\\",\\n    log=print,\\n) -> Path:\\n    import lpips\\n\\n    net_lpips = lpips.LPIPS(net=\\"alex\\", verbose=False).to(device).eval()\\n    items = []\\n    for s in test_scenes:\\n        its = collect_items(pairs_root, [s])\\n        random.Random(0).shuffle(its)\\n        items += its[:max_items_per_scene]\\n    ds = PairDataset(items, train=False)\\n    log(f\\"[eval {tag}] {len(items)} pairs from {len(test_scenes)} test scenes, methods: {list(methods)}\\")\\n\\n    out_dir = ws.results_dir / tag\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    sheet_idx = sorted(random.Random(1).sample(range(len(ds)), min(n_sheets, len(ds))))\\n    sheet_cols: dict[int, list] = defaultdict(list)\\n    rows = []\\n    for name, loader in methods.items():\\n        fn = loader()\\n        for i in range(len(ds)):\\n            s = ds[i]\\n            deg, gt, ref = (s[k][None].to(device) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n            out = fn(deg, ref).float().clamp(0, 1)\\n            m = _metrics(out, gt, net_lpips)\\n            it = items[i]\\n            rows.append({\\"method\\": name, \\"scene\\": it[\\"scene\\"], \\"recipe\\": it[\\"recipe\\"], \\"frame\\": it[\\"frame\\"], **m})\\n            if i in sheet_idx:\\n                if not sheet_cols[i]:\\n                    sheet_cols[i].append((\\"ref\\", ref[0].cpu()))\\n                sheet_cols[i].append((name, out[0].cpu()))\\n                if name == list(methods)[-1]:\\n                    sheet_cols[i].append((\\"gt\\", gt[0].cpu()))\\n        del fn\\n        torch.cuda.empty_cache()\\n        log(f\\"[eval {tag}] {name} done\\")\\n\\n    with open(out_dir / \\"per_item.csv\\", \\"w\\", newline=\\"\\") as f:\\n        w = csv.DictWriter(f, fieldnames=list(rows[0]))\\n        w.writeheader()\\n        w.writerows(rows)\\n\\n    summary = _summarise(rows)\\n    (out_dir / \\"summary.json\\").write_text(json.dumps(summary, indent=2), encoding=\\"utf-8\\")\\n    (out_dir / \\"summary.md\\").write_text(_markdown(summary), encoding=\\"utf-8\\")\\n    _contact_sheets(out_dir, sheet_cols)\\n    log(_markdown(summary))\\n    return out_dir\\n\\n\\ndef _summarise(rows: list[dict]) -> dict:\\n    groups = defaultdict(list)\\n    for r in rows:\\n        groups[(r[\\"method\\"], r[\\"recipe\\"])].append(r)\\n        groups[(r[\\"method\\"], \\"all\\")].append(r)\\n    out = {}\\n    for (method, recipe), rs in groups.items():\\n        out.setdefault(method, {})[recipe] = {\\n            k: float(np.mean([r[k] for r in rs])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")\\n        } | {\\"n\\": len(rs)}\\n    return out\\n\\n\\ndef _markdown(summary: dict) -> str:\\n    recipes = sorted({rec for m in summary.values() for rec in m}, key=lambda r: (r != \\"all\\", r))\\n    base = summary.get(\\"input\\", {})\\n    lines = []\\n    for rec in recipes:\\n        lines.append(f\\"\\\\n### {rec}\\\\n\\")\\n        lines.append(\\"| method | PSNR ↑ | ΔPSNR | SSIM ↑ | LPIPS ↓ | ΔLPIPS |\\")\\n        lines.append(\\"|---|---|---|---|---|---|\\")\\n        for method, by_rec in summary.items():\\n            if rec not in by_rec:\\n                continue\\n            m = by_rec[rec]\\n            b = base.get(rec, m)\\n            lines.append(\\n                f\\"| {method} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - b[\'psnr\']:+.2f} | {m[\'ssim\']:.4f} | \\"\\n                f\\"{m[\'lpips\']:.4f} | {m[\'lpips\'] - b[\'lpips\']:+.4f} |\\"\\n            )\\n    return \\"\\\\n\\".join(lines) + \\"\\\\n\\"\\n\\n\\ndef _contact_sheets(out_dir: Path, sheet_cols: dict) -> None:\\n    from PIL import Image, ImageDraw\\n\\n    for i, cols in sheet_cols.items():\\n        tiles = []\\n        for name, t in cols:\\n            arr = (t.permute(1, 2, 0).numpy() * 255 + 0.5).astype(np.uint8)\\n            img = Image.fromarray(arr)\\n            ImageDraw.Draw(img).text((8, 8), name, fill=(255, 255, 0))\\n            tiles.append(img)\\n        w, h = tiles[0].size\\n        cols_n = 3\\n        rows_n = math.ceil(len(tiles) / cols_n)\\n        sheet = Image.new(\\"RGB\\", (w * cols_n, h * rows_n))\\n        for k, tile in enumerate(tiles):\\n            sheet.paste(tile, ((k % cols_n) * w, (k // cols_n) * h))\\n        sheet.save(out_dir / f\\"sheet_{i:04d}.jpg\\", quality=88)\\n", "lib/splat_restorer/gs_trainer.py": "\\"\\"\\"Small, self-contained 3DGS trainer on top of gsplat 1.5.x.\\n\\nIt is intentionally plain: the pair generator only needs *plausible* splat\\nartefacts, and the distill-back test needs a fair, repeatable fine-tune.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport math\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\n\\nSH_C0 = 0.28209479177387814\\n\\n\\n@dataclass\\nclass View:\\n    image: torch.Tensor  # (H, W, 3) float in [0, 1] or uint8; any device\\n    w2c: torch.Tensor  # (4, 4)\\n    K: torch.Tensor  # (3, 3)\\n    weight: float = 1.0\\n\\n\\n@dataclass\\nclass TrainConfig:\\n    steps: int = 4000\\n    sh_degree: int = 3\\n    sh_increase_every: int = 1000\\n    ssim_lambda: float = 0.2\\n    densify: bool = True\\n    refine_start: int = 500\\n    refine_stop_frac: float = 0.75\\n    opacity_reset_every: int = 1_000_000  # disabled: short runs do not recover\\n    lr_means: float = 1.6e-4\\n    lr_means_final_frac: float = 0.01\\n    lr_scales: float = 5e-3\\n    lr_quats: float = 1e-3\\n    lr_opacities: float = 5e-2\\n    lr_sh0: float = 2.5e-3\\n    lr_shN: float = 2.5e-3 / 20\\n    init_opacity: float = 0.1\\n    seed: int = 0\\n    log_every: int = 0\\n    extra: dict = field(default_factory=dict)\\n\\n\\ndef _knn_mean_dist(xyz: np.ndarray, k: int = 3) -> np.ndarray:\\n    from scipy.spatial import cKDTree\\n\\n    tree = cKDTree(xyz)\\n    d, _ = tree.query(xyz, k=k + 1)\\n    return np.sqrt((d[:, 1:] ** 2).mean(axis=1))\\n\\n\\ndef init_params(xyz: np.ndarray, rgb: np.ndarray, sh_degree: int, init_opacity: float, device) -> torch.nn.ParameterDict:\\n    n = len(xyz)\\n    dist = np.clip(_knn_mean_dist(xyz), 1e-7, None)\\n    scales = np.log(np.repeat(dist[:, None], 3, axis=1))\\n    quats = np.zeros((n, 4), np.float32)\\n    quats[:, 0] = 1.0\\n    opac = np.full(n, math.log(init_opacity / (1 - init_opacity)), np.float32)\\n    sh0 = ((rgb - 0.5) / SH_C0)[:, None, :]\\n    k = (sh_degree + 1) ** 2 - 1\\n    shN = np.zeros((n, k, 3), np.float32)\\n    t = lambda a: torch.nn.Parameter(torch.as_tensor(a, dtype=torch.float32, device=device))\\n    return torch.nn.ParameterDict(\\n        {\\n            \\"means\\": t(xyz),\\n            \\"scales\\": t(scales),\\n            \\"quats\\": t(quats),\\n            \\"opacities\\": t(opac),\\n            \\"sh0\\": t(sh0),\\n            \\"shN\\": t(shN),\\n        }\\n    )\\n\\n\\ndef make_optimizers(params, cfg: TrainConfig, scene_scale: float) -> dict[str, torch.optim.Optimizer]:\\n    lrs = {\\n        \\"means\\": cfg.lr_means * scene_scale,\\n        \\"scales\\": cfg.lr_scales,\\n        \\"quats\\": cfg.lr_quats,\\n        \\"opacities\\": cfg.lr_opacities,\\n        \\"sh0\\": cfg.lr_sh0,\\n        \\"shN\\": cfg.lr_shN,\\n    }\\n    return {k: torch.optim.Adam([{\\"params\\": params[k], \\"lr\\": lr, \\"name\\": k}], eps=1e-15) for k, lr in lrs.items()}\\n\\n\\ndef render(params, w2c: torch.Tensor, K: torch.Tensor, width: int, height: int, sh_degree: int,\\n           rasterize_mode: str = \\"classic\\", **raster_kwargs):\\n    \\"\\"\\"Render one view. Returns (image HxWx3 clamped to [0,1] in the graph, info).\\n\\n    ``rasterize_mode=\\"antialiased\\"`` for splats trained with gsplat\'s antialiased mode;\\n    ``raster_kwargs`` pass through to gsplat (e.g. ``with_ut=True, with_eval3d=True`` for 3DGUT).\\"\\"\\"\\n    from gsplat import rasterization\\n\\n    colors = torch.cat([params[\\"sh0\\"], params[\\"shN\\"]], dim=1)\\n    img, _alpha, info = rasterization(\\n        means=params[\\"means\\"],\\n        quats=params[\\"quats\\"],\\n        scales=torch.exp(params[\\"scales\\"]),\\n        opacities=torch.sigmoid(params[\\"opacities\\"]),\\n        colors=colors,\\n        viewmats=w2c[None],\\n        Ks=K[None],\\n        width=width,\\n        height=height,\\n        sh_degree=sh_degree,\\n        packed=False,\\n        rasterize_mode=rasterize_mode,\\n        **raster_kwargs,\\n    )\\n    return img[0].clamp(0.0, 1.0), info\\n\\n\\ndef _gauss_window(size: int = 11, sigma: float = 1.5, device=None) -> torch.Tensor:\\n    x = torch.arange(size, dtype=torch.float32, device=device) - size // 2\\n    g = torch.exp(-(x**2) / (2 * sigma**2))\\n    g = g / g.sum()\\n    return (g[:, None] @ g[None, :])[None, None].repeat(3, 1, 1, 1)\\n\\n\\ndef ssim(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\\n    \\"\\"\\"SSIM of two (H, W, 3) images in [0, 1].\\"\\"\\"\\n    x = a.permute(2, 0, 1)[None]\\n    y = b.permute(2, 0, 1)[None]\\n    w = _gauss_window(device=a.device)\\n    pad = w.shape[-1] // 2\\n    mu_x = F.conv2d(x, w, padding=pad, groups=3)\\n    mu_y = F.conv2d(y, w, padding=pad, groups=3)\\n    sxx = F.conv2d(x * x, w, padding=pad, groups=3) - mu_x**2\\n    syy = F.conv2d(y * y, w, padding=pad, groups=3) - mu_y**2\\n    sxy = F.conv2d(x * y, w, padding=pad, groups=3) - mu_x * mu_y\\n    c1, c2 = 0.01**2, 0.03**2\\n    m = ((2 * mu_x * mu_y + c1) * (2 * sxy + c2)) / ((mu_x**2 + mu_y**2 + c1) * (sxx + syy + c2))\\n    return m.mean()\\n\\n\\ndef psnr(a: torch.Tensor, b: torch.Tensor) -> float:\\n    mse = F.mse_loss(a, b).item()\\n    return float(10 * math.log10(1.0 / max(mse, 1e-12)))\\n\\n\\ndef train_gaussians(\\n    views: list[View],\\n    width: int,\\n    height: int,\\n    cfg: TrainConfig,\\n    scene_scale: float = 1.1,\\n    params: torch.nn.ParameterDict | None = None,\\n    init_xyz: np.ndarray | None = None,\\n    init_rgb: np.ndarray | None = None,\\n    log=print,\\n) -> torch.nn.ParameterDict:\\n    \\"\\"\\"Optimise Gaussians on ``views``. Pass ``params`` to fine-tune existing ones.\\"\\"\\"\\n    from gsplat.strategy import DefaultStrategy\\n\\n    device = views[0].w2c.device\\n    gen = torch.Generator().manual_seed(cfg.seed)\\n    if params is None:\\n        assert init_xyz is not None and init_rgb is not None\\n        params = init_params(init_xyz, init_rgb, cfg.sh_degree, cfg.init_opacity, device)\\n    optimizers = make_optimizers(params, cfg, scene_scale)\\n    means_gamma = cfg.lr_means_final_frac ** (1.0 / max(cfg.steps, 1))\\n\\n    strategy = None\\n    if cfg.densify:\\n        strategy = DefaultStrategy(\\n            refine_start_iter=cfg.refine_start,\\n            refine_stop_iter=int(cfg.steps * cfg.refine_stop_frac),\\n            reset_every=cfg.opacity_reset_every,\\n            verbose=False,\\n        )\\n        strategy.check_sanity(params, optimizers)\\n        state = strategy.initialize_state(scene_scale=scene_scale)\\n\\n    weights = torch.tensor([v.weight for v in views], dtype=torch.float64)\\n    for step in range(cfg.steps):\\n        idx = int(torch.multinomial(weights, 1, generator=gen))\\n        v = views[idx]\\n        sh_deg = min(step // cfg.sh_increase_every, cfg.sh_degree) if cfg.densify else cfg.sh_degree\\n        img, info = render(params, v.w2c, v.K, width, height, sh_deg)\\n        if strategy is not None:\\n            strategy.step_pre_backward(params, optimizers, state, step, info)\\n        # Large photo sets may live on the CPU and/or as uint8 to save GPU memory.\\n        gt = v.image.to(img.device, non_blocking=True)\\n        if not gt.dtype.is_floating_point:\\n            gt = gt.float() / 255.0\\n        l1 = (img - gt).abs().mean()\\n        loss = (1 - cfg.ssim_lambda) * l1 + cfg.ssim_lambda * (1 - ssim(img, gt))\\n        loss.backward()\\n        if strategy is not None:\\n            strategy.step_post_backward(params, optimizers, state, step, info, packed=False)\\n        for opt in optimizers.values():\\n            opt.step()\\n            opt.zero_grad(set_to_none=True)\\n        for g in optimizers[\\"means\\"].param_groups:\\n            g[\\"lr\\"] *= means_gamma\\n        if cfg.log_every and (step + 1) % cfg.log_every == 0:\\n            log(f\\"    step {step + 1}/{cfg.steps} loss={loss.item():.4f} n={len(params[\'means\'])}\\")\\n    return params\\n\\n\\ndef params_to_cpu_state(params: torch.nn.ParameterDict) -> dict[str, torch.Tensor]:\\n    return {k: v.detach().cpu() for k, v in params.items()}\\n\\n\\ndef params_from_state(state: dict[str, torch.Tensor], device) -> torch.nn.ParameterDict:\\n    return torch.nn.ParameterDict({k: torch.nn.Parameter(v.to(device)) for k, v in state.items()})\\n", "lib/splat_restorer/gsplat_testbed.py": "\\"\\"\\"Train gsplat\'s reference trainer (examples/simple_trainer.py, v1.5.3) variants and\\nscore them on the same held-out photos as our other experiments.\\n\\nEach variant changes one setting of gsplat\'s maintained trainer, so research\\nfeatures (MCMC, AbsGS, antialiasing, pose refinement, full-resolution input)\\nare compared on one footing. Splats are read straight from the trainer\'s\\ncheckpoint and rendered with our cameras through ``own_splat``\'s evaluation,\\nso numbers line up with the Difix and A/B tests. Frames where even nearby\\n*training* views fail (bad SfM poses in close-up segments) are flagged and a\\nsecond average without them is reported.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nimport subprocess\\nimport sys\\nimport time\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nfrom .gs_trainer import params_from_state, render\\nfrom .own_splat import SplatData, _psnr, _w2c_t\\n\\n\\n@dataclass\\nclass GsVariant:\\n    name: str\\n    preset: str = \\"default\\"  # simple_trainer config: \\"default\\" or \\"mcmc\\"\\n    factor: int = 2  # data_factor: 2 -> 1920x1080 from 4K, 1 -> native 4K\\n    args: list[str] = field(default_factory=list)\\n    rasterize_mode: str = \\"classic\\"\\n    # Extra gsplat rasterization options needed to render this splat as it was trained (3DGUT).\\n    render: dict = field(default_factory=dict)\\n    # Also score with the plain rasterizer, i.e. how a standard 3DGS viewer would show the PLY.\\n    portable_eval: bool = False\\n    # Leave training frames in pose-failure segments out of training (held-out set unchanged).\\n    exclude_bad_frames: bool = False\\n    note: str = \\"\\"\\n\\n\\nVARIANTS = {\\n    \\"default\\": GsVariant(\\"default\\", note=\\"gsplat varsayılanı (orijinal 3DGS yoğunlaştırma)\\"),\\n    \\"mcmc\\": GsVariant(\\"mcmc\\", preset=\\"mcmc\\", args=[\\"--strategy.cap-max\\", \\"1000000\\"], note=\\"3DGS-MCMC, 1M splat\\"),\\n    \\"absgrad\\": GsVariant(\\"absgrad\\", args=[\\"--strategy.absgrad\\", \\"--strategy.grow-grad2d\\", \\"0.0006\\"],\\n                         note=\\"AbsGS (mutlak gradyan; gsplat benchmark eşiği 0.0006)\\"),\\n    \\"antialiased\\": GsVariant(\\"antialiased\\", args=[\\"--antialiased\\"], rasterize_mode=\\"antialiased\\",\\n                             note=\\"Mip-Splatting\'in 2D filtresi\\"),\\n    \\"pose_opt\\": GsVariant(\\"pose_opt\\", args=[\\"--pose-opt\\"], note=\\"eğitim kamera pozlarını iyileştirir\\"),\\n    \\"default_4k\\": GsVariant(\\"default_4k\\", factor=1, note=\\"4K tam çözünürlükte eğitim\\"),\\n    # Appended later; earlier variants keep their names so a resumed run skips them.\\n    # 3DGUT has no 2D mean gradients for the default densifier, so gsplat runs it with MCMC:\\n    # compare it against the `mcmc` row.\\n    \\"with_ut\\": GsVariant(\\"with_ut\\", preset=\\"mcmc\\", args=[\\"--strategy.cap-max\\", \\"1000000\\", \\"--with-ut\\", \\"--with-eval3d\\"],\\n                         render={\\"with_ut\\": True, \\"with_eval3d\\": True}, portable_eval=True,\\n                         note=\\"3DGUT + MCMC (unscented transform, 3D değerlendirme); `mcmc` ile karşılaştır\\"),\\n    \\"bilateral_grid\\": GsVariant(\\"bilateral_grid\\", args=[\\"--use-bilateral-grid\\"],\\n                                note=\\"kare başına renk/pozlama grid\'i (ölçüm grid\'siz PLY ile)\\"),\\n    \\"depth_loss\\": GsVariant(\\"depth_loss\\", args=[\\"--depth-loss\\"], note=\\"SfM noktalarından derinlik kaybı\\"),\\n    \\"mcmc_pose_opt\\": GsVariant(\\"mcmc_pose_opt\\", preset=\\"mcmc\\", args=[\\"--strategy.cap-max\\", \\"1000000\\", \\"--pose-opt\\"],\\n                               note=\\"MCMC + poz iyileştirme\\"),\\n    \\"absgrad_aa\\": GsVariant(\\"absgrad_aa\\", args=[\\"--strategy.absgrad\\", \\"--strategy.grow-grad2d\\", \\"0.0006\\", \\"--antialiased\\"],\\n                            rasterize_mode=\\"antialiased\\",\\n                            note=\\"AbsGS + anti-aliasing (gsplat\'in en iyi varsayılan kombinasyonu)\\"),\\n    \\"exclude_bad_frames\\": GsVariant(\\"exclude_bad_frames\\", exclude_bad_frames=True,\\n                                    note=\\"poz hatalı bölümlerdeki eğitim kareleri çıkarılmış\\"),\\n}\\n\\n# Wrapper around simple_trainer: optionally drops training images listed in GS_EXCLUDE_NAMES\\n# (a JSON list of image names) at the dataset level, so the train/val split (index % test_every)\\n# and therefore the held-out frames stay exactly the same.\\nWRAPPER = \\"\\"\\"import json, os, runpy, sys\\nimport numpy as np\\nimport datasets.colmap as dc\\n_exclude = set(json.load(open(os.environ[\\"GS_EXCLUDE_NAMES\\"]))) if os.environ.get(\\"GS_EXCLUDE_NAMES\\") else set()\\n_init = dc.Dataset.__init__\\ndef _filtered_init(self, parser, split=\\"train\\", *args, **kwargs):\\n    _init(self, parser, split, *args, **kwargs)\\n    if split == \\"train\\" and _exclude:\\n        before = len(self.indices)\\n        self.indices = np.array([i for i in self.indices if parser.image_names[i] not in _exclude],\\n                                dtype=self.indices.dtype)\\n        print(f\\"[tb_train] excluded {before - len(self.indices)} training images\\", flush=True)\\ndc.Dataset.__init__ = _filtered_init\\nsys.argv = [\\"simple_trainer.py\\", *sys.argv[1:]]\\nrunpy.run_path(\\"simple_trainer.py\\", run_name=\\"__main__\\")\\n\\"\\"\\"\\n\\n\\n# ----------------------------------------------------------------------------- data\\n\\n\\ndef prepare_factor_images(data_dir: Path, factor: int, workers: int = 8, log=print) -> None:\\n    \\"\\"\\"``images_<factor>`` as area-downsampled PNGs (gsplat requires the folder to exist;\\n    given JPGs it would re-create them with bicubic resizing instead).\\"\\"\\"\\n    import cv2\\n\\n    if factor <= 1:\\n        return\\n    src, dst = Path(data_dir) / \\"images\\", Path(data_dir) / f\\"images_{factor}\\"\\n    dst.mkdir(exist_ok=True)\\n    files = sorted(p for p in src.rglob(\\"*\\") if p.is_file())\\n\\n    def convert(p: Path):\\n        out = dst / p.relative_to(src).with_suffix(\\".png\\")\\n        if out.exists():\\n            return\\n        out.parent.mkdir(parents=True, exist_ok=True)\\n        img = cv2.imread(str(p), cv2.IMREAD_COLOR)\\n        h, w = img.shape[:2]\\n        small = cv2.resize(img, (int(round(w / factor)), int(round(h / factor))), interpolation=cv2.INTER_AREA)\\n        tmp = out.with_suffix(\\".tmp.png\\")\\n        cv2.imwrite(str(tmp), small, [cv2.IMWRITE_PNG_COMPRESSION, 1])\\n        tmp.replace(out)\\n\\n    todo = [p for p in files if not (dst / p.relative_to(src).with_suffix(\\".png\\")).exists()]\\n    if todo:\\n        log(f\\"[images_{factor}] {len(todo)} images, area downsample x{factor}\\")\\n        with ThreadPoolExecutor(workers) as pool:\\n            for n, _ in enumerate(pool.map(convert, todo), 1):\\n                if n % 200 == 0 or n == len(todo):\\n                    log(f\\"[images_{factor}] {n}/{len(todo)}\\")\\n\\n\\n# ----------------------------------------------------------------------------- training\\n\\n\\ndef trainer_command(v: GsVariant, data_dir: Path, result_dir: Path, steps: int, test_every: int) -> list[str]:\\n    return [\\n        sys.executable, \\"tb_train.py\\", v.preset,\\n        \\"--data-dir\\", str(data_dir), \\"--data-factor\\", str(v.factor), \\"--test-every\\", str(test_every),\\n        \\"--no-normalize-world-space\\", \\"--result-dir\\", str(result_dir),\\n        \\"--max-steps\\", str(steps), \\"--eval-steps\\", str(steps), \\"--save-steps\\", str(steps),\\n        \\"--disable-viewer\\", \\"--disable-video\\", *v.args,\\n    ]\\n\\n\\ndef run_variant(v: GsVariant, data_dir: Path, examples_dir: Path, result_dir: Path, steps: int,\\n                test_every: int, exclude_names: list[str] | None = None, log_every_s: float = 60.0,\\n                log=print) -> float:\\n    \\"\\"\\"Run simple_trainer (through WRAPPER) in a subprocess; full output goes to\\n    result_dir/train.log. Returns seconds.\\"\\"\\"\\n    import os\\n\\n    result_dir.mkdir(parents=True, exist_ok=True)\\n    (Path(examples_dir) / \\"tb_train.py\\").write_text(WRAPPER, encoding=\\"utf-8\\")\\n    env = dict(os.environ)\\n    env.pop(\\"GS_EXCLUDE_NAMES\\", None)\\n    if exclude_names:\\n        (result_dir / \\"exclude_names.json\\").write_text(json.dumps(exclude_names), encoding=\\"utf-8\\")\\n        env[\\"GS_EXCLUDE_NAMES\\"] = str(result_dir / \\"exclude_names.json\\")\\n    cmd = trainer_command(v, data_dir, result_dir, steps, test_every)\\n    log(\\"[train] \\" + \\" \\".join(cmd[1:]))\\n    t0, last = time.time(), 0.0\\n    with open(result_dir / \\"train.log\\", \\"w\\", encoding=\\"utf-8\\") as f, subprocess.Popen(\\n        cmd, cwd=examples_dir, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,\\n        encoding=\\"utf-8\\", errors=\\"replace\\", bufsize=1,\\n    ) as proc:\\n        for line in proc.stdout:\\n            f.write(line)\\n            line = line.rstrip()\\n            is_progress = \\"it/s]\\" in line or \\"s/it]\\" in line\\n            if not line or (is_progress and time.time() - last < log_every_s):\\n                continue\\n            if is_progress:\\n                last = time.time()\\n            log(\\"  \\" + line[-160:])\\n        code = proc.wait()\\n    if code != 0:\\n        tail = (result_dir / \\"train.log\\").read_text(encoding=\\"utf-8\\").splitlines()[-25:]\\n        raise RuntimeError(f\\"{v.name} failed (exit {code}):\\\\n\\" + \\"\\\\n\\".join(tail))\\n    return time.time() - t0\\n\\n\\ndef load_checkpoint(result_dir: Path, device=\\"cuda\\") -> tuple[torch.nn.ParameterDict, int]:\\n    \\"\\"\\"Splats from simple_trainer\'s last checkpoint; same layout as gs_trainer params.\\"\\"\\"\\n    ckpts = sorted(Path(result_dir, \\"ckpts\\").glob(\\"ckpt_*_rank0.pt\\"), key=lambda p: int(p.stem.split(\\"_\\")[1]))\\n    if not ckpts:\\n        raise FileNotFoundError(f\\"No checkpoint in {result_dir}/ckpts\\")\\n    splats = torch.load(ckpts[-1], map_location=\\"cpu\\", weights_only=False)[\\"splats\\"]\\n    state = {k: splats[k].float() for k in (\\"means\\", \\"scales\\", \\"quats\\", \\"opacities\\", \\"sh0\\", \\"shN\\")}\\n    sh_degree = int(round(math.sqrt(state[\\"shN\\"].shape[1] + 1))) - 1\\n    return params_from_state(state, device), sh_degree\\n\\n\\ndef save_ply(params, path: Path) -> None:\\n    \\"\\"\\"Standard 3DGS PLY (log scales, logit opacity, SH channel-major f_rest).\\"\\"\\"\\n    p = {k: v.detach().cpu().numpy().astype(np.float32) for k, v in params.items()}\\n    n, k = p[\\"means\\"].shape[0], p[\\"shN\\"].shape[1]\\n    names = ([\\"x\\", \\"y\\", \\"z\\", \\"nx\\", \\"ny\\", \\"nz\\"] + [f\\"f_dc_{i}\\" for i in range(3)]\\n             + [f\\"f_rest_{i}\\" for i in range(3 * k)] + [\\"opacity\\"]\\n             + [f\\"scale_{i}\\" for i in range(3)] + [f\\"rot_{i}\\" for i in range(4)])\\n    cols = [p[\\"means\\"], np.zeros((n, 3), np.float32), p[\\"sh0\\"].reshape(n, 3),\\n            p[\\"shN\\"].transpose(0, 2, 1).reshape(n, 3 * k), p[\\"opacities\\"].reshape(n, 1), p[\\"scales\\"], p[\\"quats\\"]]\\n    data = np.concatenate(cols, axis=1).astype(\\"<f4\\")\\n    header = \\"ply\\\\nformat binary_little_endian 1.0\\\\nelement vertex %d\\\\n%send_header\\\\n\\" % (\\n        n, \\"\\".join(f\\"property float {nm}\\\\n\\" for nm in names))\\n    tmp = Path(path).with_suffix(\\".tmp\\")\\n    with open(tmp, \\"wb\\") as f:\\n        f.write(header.encode(\\"ascii\\"))\\n        f.write(data.tobytes())\\n    tmp.replace(path)\\n\\n\\n# ----------------------------------------------------------------------------- pose-failure frames\\n\\n\\n@torch.no_grad()\\ndef frame_psnr(params, deg: int, data: SplatData, ids: list[int], rasterize_mode=\\"classic\\", device=\\"cuda\\",\\n               **raster_kwargs) -> dict[int, float]:\\n    w2c, K = _w2c_t(data, device)\\n    out = {}\\n    for i in ids:\\n        img, _ = render(params, w2c[i], K, data.width, data.height, deg, rasterize_mode, **raster_kwargs)\\n        out[i] = _psnr(img, data.photo(i, device))\\n    return out\\n\\n\\ndef pose_failure_holdouts(train_psnr: dict[int, float], holdout: list[int], k_mad: float = 4.0,\\n                          neighbours: int = 2) -> tuple[list[int], list[int], float]:\\n    \\"\\"\\"Held-out frames whose neighbouring *training* frames the splat also fails to reproduce.\\n\\n    Training frames far below the typical training PSNR (median - k*MAD) cannot be fitted\\n    even though they were trained on: usually wrong SfM poses (close-up segments) or\\n    heavy blur. Returns (flagged held-out frames, flagged training frames, threshold dB).\\"\\"\\"\\n    v = np.array(list(train_psnr.values()))\\n    med = float(np.median(v))\\n    thr = med - k_mad * 1.4826 * float(np.median(np.abs(v - med)))\\n    bad_train = sorted(i for i, p in train_psnr.items() if p < thr)\\n    bad_set = set(bad_train)\\n    flagged = [h for h in holdout if any((h + d) in bad_set for d in range(-neighbours, neighbours + 1) if d)]\\n    return flagged, bad_train, thr\\n\\n\\n# ----------------------------------------------------------------------------- reporting\\n\\n\\ndef summarise(results: dict, keep: list[int]) -> dict:\\n    out = {}\\n    keep_s = {str(i) for i in keep}\\n    for name, r in results.items():\\n        rows = r[\\"frames\\"]\\n        allv = list(rows.values())\\n        kept = [x for i, x in rows.items() if i in keep_s]\\n        m = {f\\"{k}\\": float(np.mean([x[k] for x in allv])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n        m.update({f\\"{k}_ok\\": float(np.mean([x[k] for x in kept])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")})\\n        info = r.get(\\"info\\", {})\\n        m.update(splats=info.get(\\"splats\\"), minutes=info.get(\\"train_seconds\\", 0) / 60, source=r.get(\\"source\\", \\"gsplat\\"),\\n                 gsplat_val_psnr=(r.get(\\"gsplat_val\\") or {}).get(\\"psnr\\"))\\n        out[name] = m\\n    return out\\n\\n\\ndef markdown(summary: dict, n_frames: int, flagged: list[int], thr: float) -> str:\\n    base = summary.get(\\"default\\") or next(iter(summary.values()))\\n    lines = [\\n        f\\"## gsplat test zemini: {n_frames} saklanan kare (hiçbiri eğitimde kullanılmadı)\\",\\n        \\"\\",\\n        f\\"\'Poz hatasız\' sütunları, yakınındaki eğitim kareleri bile {thr:.1f} dB altında kalan \\"\\n        f\\"{len(flagged)} kareyi hariç tutar (yanlış SfM pozu / yakın plan çökmesi): {flagged}\\",\\n        \\"\\",\\n        \\"| varyant | PSNR ↑ | Δ | PSNR (poz hatasız) | Δ | SSIM ↑ | LPIPS ↓ | LPIPS (poz hatasız) | gsplat val PSNR \\"\\n        \\"| splat | dk |\\",\\n        \\"|---|---|---|---|---|---|---|---|---|---|---|\\",\\n    ]\\n    for name, m in summary.items():\\n        splats = f\\"{m[\'splats\']:,}\\" if m.get(\\"splats\\") else \\"-\\"\\n        gv = f\\"{m[\'gsplat_val_psnr\']:.2f}\\" if m.get(\\"gsplat_val_psnr\\") else \\"-\\"\\n        lines.append(\\n            f\\"| {name} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - base[\'psnr\']:+.2f} | {m[\'psnr_ok\']:.2f} | \\"\\n            f\\"{m[\'psnr_ok\'] - base[\'psnr_ok\']:+.2f} | {m[\'ssim\']:.4f} | {m[\'lpips\']:.4f} | {m[\'lpips_ok\']:.4f} | \\"\\n            f\\"{gv} | {splats} | {m[\'minutes\']:.0f} |\\"\\n        )\\n    lines += [\\"\\", \\"Δ: `default` satırına göre. \'app:\' ile başlayan satırlar app\'in kendi eğiticisiyle yapılan \\"\\n              \\"A/B testinden (aynı kareler, aynı ölçüm). +0,3 dB altındaki farklar koşudan koşuya oynayabilir.\\",\\n              \\"\'gsplat val PSNR\': gsplat\'in kendi değerlendirmesi (kendi lens düzeltmesi/kırpması ve, varsa, \\"\\n              \\"bilateral grid ile); kendi içinde karşılaştırılır, bizim sütunlarla birebir aynı ölçek değildir.\\",\\n              \\"\'(klasik)\' satırları, splat\'in standart bir 3DGS görüntüleyicide (3DGUT olmadan) nasıl göründüğüdür.\\"]\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef save_json(path: Path, obj) -> None:\\n    tmp = Path(path).with_suffix(\\".tmp\\")\\n    tmp.write_text(json.dumps(obj, indent=1), encoding=\\"utf-8\\")\\n    tmp.replace(path)\\n\\n\\ndef gsplat_val(stats_dir: Path) -> dict | None:\\n    \\"\\"\\"gsplat\'s own validation metrics (last val_step*.json) from a run or Drive folder.\\"\\"\\"\\n    files = sorted(Path(stats_dir).glob(\\"*val_step*.json\\"), key=lambda p: int(\\"\\".join(c for c in p.stem.split(\\"step\\")[-1] if c.isdigit()) or 0))\\n    return json.loads(files[-1].read_text(encoding=\\"utf-8\\")) if files else None\\n\\n\\ndef fix_gsplat_install(examples_dir: Path, python: str = sys.executable) -> None:\\n    \\"\\"\\"Make gsplat v1.5.3\'s example trainer importable on Colab. Idempotent.\\n\\n    1. ``examples/datasets`` has no ``__init__.py``, so it is a namespace package and loses to\\n       the regular Hugging Face ``datasets`` package that Colab preinstalls.\\n    2. The pinned pycolmap fork defines ``np.uint64(-1)``, an OverflowError under numpy 2.\\n    Then imports both exactly as the trainer does, in a subprocess from ``examples_dir``.\\n    \\"\\"\\"\\n    import importlib.util\\n    import re\\n\\n    init = Path(examples_dir) / \\"datasets\\" / \\"__init__.py\\"\\n    if not init.exists():\\n        init.write_text(\\"\\", encoding=\\"utf-8\\")\\n    spec = importlib.util.find_spec(\\"pycolmap\\")  # locate without importing (import would fail)\\n    if spec and spec.submodule_search_locations:\\n        sm = Path(next(iter(spec.submodule_search_locations))) / \\"scene_manager.py\\"\\n        if sm.exists():\\n            text = sm.read_text(encoding=\\"utf-8\\")\\n            fixed = re.sub(r\\"np\\\\.uint64\\\\(\\\\s*-1\\\\s*\\\\)\\", \\"np.uint64(np.iinfo(np.uint64).max)\\", text)\\n            if fixed != text:\\n                sm.write_text(fixed, encoding=\\"utf-8\\")\\n    check = (\\"import numpy, pycolmap; from datasets.colmap import Parser; import datasets.colmap as d; \\"\\n             \\"print(\'ok numpy\', numpy.__version__, d.__file__)\\")\\n    out = subprocess.run([python, \\"-c\\", check], cwd=examples_dir, capture_output=True, text=True)\\n    if out.returncode != 0:\\n        raise RuntimeError(\\"gsplat trainer imports still fail:\\\\n\\" + out.stderr[-2000:])\\n    print(out.stdout.strip())\\n", "lib/splat_restorer/LICENSE-DIFIX3D.txt": "NVIDIA License\\n\\n1. Definitions\\n\\n“Licensor” means any person or entity that distributes its Work.\\n“Work” means (a) the original work of authorship made available under this license, which may include software, documentation, or other files, and (b) any additions to or derivative works  thereof  that are made available under this license.\\nThe terms “reproduce,” “reproduction,” “derivative works,” and “distribution” have the meaning as provided under U.S. copyright law; provided, however, that for the purposes of this license, derivative works shall not include works that remain separable from, or merely link (or bind by name) to the interfaces of, the Work.\\nWorks are “made available” under this license by including in or with the Work either (a) a copyright notice referencing the applicability of this license to the Work, or (b) a copy of this license.\\n\\n2. License Grant\\n\\n2.1 Copyright Grant. Subject to the terms and conditions of this license, each Licensor grants to you a perpetual, worldwide, non-exclusive, royalty-free, copyright license to use, reproduce, prepare derivative works of, publicly display, publicly perform, sublicense and distribute its Work and any resulting derivative works in any form.\\n\\n3. Limitations\\n\\n3.1 Redistribution. You may reproduce or distribute the Work only if (a) you do so under this license, (b) you include a complete copy of this license with your distribution, and (c) you retain without modification any copyright, patent, trademark, or attribution notices that are present in the Work.\\n\\n3.2 Derivative Works. You may specify that additional or different terms apply to the use, reproduction, and distribution of your derivative works of the Work (“Your Terms”) only if (a) Your Terms provide that the use limitation in Section 3.3 applies to your derivative works, and (b) you identify the specific derivative works that are subject to Your Terms. Notwithstanding Your Terms, this license (including the redistribution requirements in Section 3.1) will continue to apply to the Work itself.\\n\\n3.3 Use Limitation. The Work and any derivative works thereof only may be used or intended for use non-commercially. Notwithstanding the foregoing, NVIDIA Corporation and its affiliates may use the Work and any derivative works commercially. As used herein, “non-commercially” means for research or evaluation purposes only.\\n\\n3.4 Patent Claims. If you bring or threaten to bring a patent claim against any Licensor (including any claim, cross-claim or counterclaim in a lawsuit) to enforce any patents that you allege are infringed by any Work, then your rights under this license from such Licensor (including the grant in Section 2.1) will terminate immediately.\\n\\n3.5 Trademarks. This license does not grant any rights to use any Licensor’s or its affiliates’ names, logos, or trademarks, except as necessary to reproduce the notices described in this license.\\n\\n3.6 Termination. If you violate any term of this license, then your rights under this license (including the grant in Section 2.1) will terminate immediately.\\n\\n4. Disclaimer of Warranty.\\n\\nTHE WORK IS PROVIDED “AS IS” WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, EITHER EXPRESS OR IMPLIED, INCLUDING WARRANTIES OR CONDITIONS OF \\nMERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE, TITLE OR NON-INFRINGEMENT. YOU BEAR THE RISK OF UNDERTAKING ANY ACTIVITIES UNDER THIS LICENSE. \\n\\n5. Limitation of Liability.\\n\\nEXCEPT AS PROHIBITED BY APPLICABLE LAW, IN NO EVENT AND UNDER NO LEGAL THEORY, WHETHER IN TORT (INCLUDING NEGLIGENCE), CONTRACT, OR OTHERWISE SHALL ANY LICENSOR BE LIABLE TO YOU FOR DAMAGES, INCLUDING ANY DIRECT, INDIRECT, SPECIAL, INCIDENTAL, OR CONSEQUENTIAL DAMAGES ARISING OUT OF OR RELATED TO THIS LICENSE, THE USE OR INABILITY TO USE THE WORK (INCLUDING BUT NOT LIMITED TO LOSS OF GOODWILL, BUSINESS INTERRUPTION, LOST PROFITS OR DATA, COMPUTER FAILURE OR MALFUNCTION, OR ANY OTHER DAMAGES OR LOSSES), EVEN IF THE LICENSOR HAS BEEN ADVISED OF THE POSSIBILITY OF SUCH DAMAGES.\\n\\nSTABILITY AI COMMUNITY LICENSE AGREEMENT\\n\\nLast Updated: July 5, 2024\\n\\nINTRODUCTION\\nThis Agreement applies to any individual person or entity (“You”, “Your” or “Licensee”) that uses or distributes any portion or element of the Stability AI Materials or Derivative Works thereof for any Research & Non-Commercial or Commercial purpose. Capitalized terms not otherwise defined herein are defined in Section V below.\\n\\nThis Agreement is intended to allow research, non-commercial, and limited commercial uses of the Models free of charge. In order to ensure that certain limited commercial uses of the Models continue to be allowed, this Agreement preserves free access to the Models for people or organizations generating annual revenue of less than US $1,000,000 (or local currency equivalent).\\n\\nBy clicking “I Accept” or by using or distributing or using any portion or element of the Stability Materials or Derivative Works, You agree that You have read, understood and are bound by the terms of this Agreement. If You are acting on behalf of a company, organization or other entity, then “You” includes you and that entity, and You agree that You: (i) are an authorized representative of such entity with the authority to bind such entity to this Agreement, and (ii) You agree to the terms of this Agreement on that entity’s behalf.\\n\\nRESEARCH & NON-COMMERCIAL USE LICENSE\\nSubject to the terms of this Agreement, Stability AI grants You a non-exclusive, worldwide, non-transferable, non-sublicensable, revocable and royalty-free limited license under Stability AI’s intellectual property or other rights owned by Stability AI embodied in the Stability AI Materials to use, reproduce, distribute, and create Derivative Works of, and make modifications to, the Stability AI Materials for any Research or Non-Commercial Purpose. “Research Purpose” means academic or scientific advancement, and in each case, is not primarily intended for commercial advantage or monetary compensation to You or others. “Non-Commercial Purpose” means any purpose other than a Research Purpose that is not primarily intended for commercial advantage or monetary compensation to You or others, such as personal use (i.e., hobbyist) or evaluation and testing.\\n\\nCOMMERCIAL USE LICENSE\\nSubject to the terms of this Agreement (including the remainder of this Section III), Stability AI grants You a non-exclusive, worldwide, non-transferable, non-sublicensable, revocable and royalty-free limited license under Stability AI’s intellectual property or other rights owned by Stability AI embodied in the Stability AI Materials to use, reproduce, distribute, and create Derivative Works of, and make modifications to, the Stability AI Materials for any Commercial Purpose. “Commercial Purpose” means any purpose other than a Research Purpose or Non-Commercial Purpose that is primarily intended for commercial advantage or monetary compensation to You or others, including but not limited to, (i) creating, modifying, or distributing Your product or service, including via a hosted service or application programming interface, and (ii) for Your business’s or organization’s internal operations. If You are using or distributing the Stability AI Materials for a Commercial Purpose, You must register with Stability AI at (https://stability.ai/community-license). If at any time You or Your Affiliate(s), either individually or in aggregate, generate more than USD $1,000,000 in annual revenue (or the equivalent thereof in Your local currency), regardless of whether that revenue is generated directly or indirectly from the Stability AI Materials or Derivative Works, any licenses granted to You under this Agreement shall terminate as of such date. You must request a license from Stability AI at (https://stability.ai/enterprise) , which Stability AI may grant to You in its sole discretion. If you receive Stability AI Materials, or any Derivative Works thereof, from a Licensee as part of an integrated end user product, then Section III of this Agreement will not apply to you.\\n\\nGENERAL TERMS\\nYour Research, Non-Commercial, and Commercial License(s) under this Agreement are subject to the following terms. a. Distribution & Attribution. If You distribute or make available the Stability AI Materials or a Derivative Work to a third party, or a product or service that uses any portion of them, You shall: (i) provide a copy of this Agreement to that third party, (ii) retain the following attribution notice within a \\"Notice\\" text file distributed as a part of such copies: \\"This Stability AI Model is licensed under the Stability AI Community License, Copyright © Stability AI Ltd. All Rights Reserved”, and (iii) prominently display “Powered by Stability AI” on a related website, user interface, blogpost, about page, or product documentation. If You create a Derivative Work, You may add your own attribution notice(s) to the “Notice” text file included with that Derivative Work, provided that You clearly indicate which attributions apply to the Stability AI Materials and state in the “Notice” text file that You changed the Stability AI Materials and how it was modified. b. Use Restrictions. Your use of the Stability AI Materials and Derivative Works, including any output or results of the Stability AI Materials or Derivative Works, must comply with applicable laws and regulations (including Trade Control Laws and equivalent regulations) and adhere to the Documentation and Stability AI’s AUP, which is hereby incorporated by reference. Furthermore, You will not use the Stability AI Materials or Derivative Works, or any output or results of the Stability AI Materials or Derivative Works, to create or improve any foundational generative AI model (excluding the Models or Derivative Works). c. Intellectual Property. (i) Trademark License. No trademark licenses are granted under this Agreement, and in connection with the Stability AI Materials or Derivative Works, You may not use any name or mark owned by or associated with Stability AI or any of its Affiliates, except as required under Section IV(a) herein. (ii) Ownership of Derivative Works. As between You and Stability AI, You are the owner of Derivative Works You create, subject to Stability AI’s ownership of the Stability AI Materials and any Derivative Works made by or for Stability AI. (iii) Ownership of Outputs. As between You and Stability AI, You own any outputs generated from the Models or Derivative Works to the extent permitted by applicable law. (iv) Disputes. If You or Your Affiliate(s) institute litigation or other proceedings against Stability AI (including a cross-claim or counterclaim in a lawsuit) alleging that the Stability AI Materials, Derivative Works or associated outputs or results, or any portion of any of the foregoing, constitutes infringement of intellectual property or other rights owned or licensable by You, then any licenses granted to You under this Agreement shall terminate as of the date such litigation or claim is filed or instituted. You will indemnify and hold harmless Stability AI from and against any claim by any third party arising out of or related to Your use or distribution of the Stability AI Materials or Derivative Works in violation of this Agreement. (v) Feedback. From time to time, You may provide Stability AI with verbal and/or written suggestions, comments or other feedback related to Stability AI’s existing or prospective technology, products or services (collectively, “Feedback”). You are not obligated to provide Stability AI with Feedback, but to the extent that You do, You hereby grant Stability AI a perpetual, irrevocable, royalty-free, fully-paid, sub-licensable, transferable, non-exclusive, worldwide right and license to exploit the Feedback in any manner without restriction. Your Feedback is provided “AS IS” and You make no warranties whatsoever about any Feedback. d. Disclaimer Of Warranty. UNLESS REQUIRED BY APPLICABLE LAW, THE STABILITY AI MATERIALS AND ANY OUTPUT AND RESULTS THEREFROM ARE PROVIDED ON AN \\"AS IS\\" BASIS, WITHOUT WARRANTIES OF ANY KIND, EITHER EXPRESS OR IMPLIED, INCLUDING, WITHOUT LIMITATION, ANY WARRANTIES OF TITLE, NON-INFRINGEMENT, MERCHANTABILITY, OR FITNESS FOR A PARTICULAR PURPOSE. YOU ARE SOLELY RESPONSIBLE FOR DETERMINING THE APPROPRIATENESS OR LAWFULNESS OF USING OR REDISTRIBUTING THE STABILITY AI MATERIALS, DERIVATIVE WORKS OR ANY OUTPUT OR RESULTS AND ASSUME ANY RISKS ASSOCIATED WITH YOUR USE OF THE STABILITY AI MATERIALS, DERIVATIVE WORKS AND ANY OUTPUT AND RESULTS. e. Limitation Of Liability. IN NO EVENT WILL STABILITY AI OR ITS AFFILIATES BE LIABLE UNDER ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, TORT, NEGLIGENCE, PRODUCTS LIABILITY, OR OTHERWISE, ARISING OUT OF THIS AGREEMENT, FOR ANY LOST PROFITS OR ANY DIRECT, INDIRECT, SPECIAL, CONSEQUENTIAL, INCIDENTAL, EXEMPLARY OR PUNITIVE DAMAGES, EVEN IF STABILITY AI OR ITS AFFILIATES HAVE BEEN ADVISED OF THE POSSIBILITY OF ANY OF THE FOREGOING. f. Term And Termination. The term of this Agreement will commence upon Your acceptance of this Agreement or access to the Stability AI Materials and will continue in full force and effect until terminated in accordance with the terms and conditions herein. Stability AI may terminate this Agreement if You are in breach of any term or condition of this Agreement. Upon termination of this Agreement, You shall delete and cease use of any Stability AI Materials or Derivative Works. Section IV(d), (e), and (g) shall survive the termination of this Agreement. g. Governing Law. This Agreement will be governed by and constructed in accordance with the laws of the United States and the State of California without regard to choice of law principles, and the UN Convention on Contracts for International Sale of Goods does not apply to this Agreement.\\n\\nDEFINITIONS\\n“Affiliate(s)” means any entity that directly or indirectly controls, is controlled by, or is under common control with the subject entity; for purposes of this definition, “control” means direct or indirect ownership or control of more than 50% of the voting interests of the subject entity.\\n\\n\\"Agreement\\" means this Stability AI Community License Agreement.\\n\\n“AUP” means the Stability AI Acceptable Use Policy available at (https://stability.ai/use-policy), as may be updated from time to time.\\n\\n\\"Derivative Work(s)” means (a) any derivative work of the Stability AI Materials as recognized by U.S. copyright laws and (b) any modifications to a Model, and any other model created which is based on or derived from the Model or the Model’s output, including “fine tune” and “low-rank adaptation” models derived from a Model or a Model’s output, but do not include the output of any Model.\\n\\n“Documentation” means any specifications, manuals, documentation, and other written information provided by Stability AI related to the Software or Models.\\n\\n“Model(s)\\" means, collectively, Stability AI’s proprietary models and algorithms, including machine-learning models, trained model weights and other elements of the foregoing listed on Stability’s Core Models Webpage available at (https://stability.ai/core-models), as may be updated from time to time.\\n\\n\\"Stability AI\\" or \\"we\\" means Stability AI Ltd. and its Affiliates.\\n\\n\\"Software\\" means Stability AI’s proprietary software made available under this Agreement now or in the future.\\n\\n“Stability AI Materials” means, collectively, Stability’s proprietary Models, Software and Documentation (and any portion or combination thereof) made available under this Agreement.\\n\\n“Trade Control Laws” means any applicable U.S. and non-U.S. export control and trade sanctions laws and regulations.", "lib/splat_restorer/own_splat.py": "\\"\\"\\"Test a restorer on one of our own trained splats.\\n\\nInputs: the COLMAP dataset the splat was trained on (``images/`` +\\n``sparse/<model>/``), the exported PLY and the list of held-out frames the\\ntrainer never saw. Two questions:\\n\\n1. ``evaluate_holdout``: on held-out photos, is the restored render closer to\\n   the real photo than the raw render (PSNR / SSIM / LPIPS)?\\n2. ``distill_back``: if the splat is fine-tuned with restored renders of new\\n   viewpoints (Difix3D-style pseudo views), do the held-out frames improve,\\n   compared with fine-tuning on the real photos alone?\\n\\n``prepare_dataset`` reproduces the importer the splat was trained with\\n(undistort to PINHOLE with alpha=0, resize to the long edge, frames in sorted\\nname order), so frame indices and held-out photos match the training run.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image, ImageDraw\\n\\nfrom .colmap_io import read_sparse_model\\nfrom .distill import restore_full\\nfrom .gs_trainer import TrainConfig, View, params_from_state, params_to_cpu_state, render, ssim, train_gaussians\\nfrom .splits import nearest_reference, subsample\\n\\n# ----------------------------------------------------------------------------- data\\n\\n\\n@dataclass\\nclass SplatData:\\n    frames_dir: Path\\n    frames: list[str]  # frame file names, index = frame id\\n    names: list[str]  # original image names\\n    width: int\\n    height: int\\n    K: np.ndarray  # (3, 3)\\n    w2c: np.ndarray  # (N, 4, 4)\\n\\n    @property\\n    def num_frames(self) -> int:\\n        return len(self.frames)\\n\\n    @property\\n    def centers(self) -> np.ndarray:\\n        R, t = self.w2c[:, :3, :3], self.w2c[:, :3, 3]\\n        return -np.einsum(\\"nji,nj->ni\\", R, t)\\n\\n    @property\\n    def forwards(self) -> np.ndarray:\\n        return self.w2c[:, 2, :3]\\n\\n    def photo_u8(self, i: int) -> torch.Tensor:\\n        return torch.from_numpy(np.asarray(Image.open(self.frames_dir / self.frames[i]).convert(\\"RGB\\")).copy())\\n\\n    def photo(self, i: int, device) -> torch.Tensor:\\n        return self.photo_u8(i).to(device).float() / 255.0\\n\\n\\ndef _k_and_dist(cam) -> tuple[np.ndarray, np.ndarray]:\\n    p = cam.params\\n    if cam.model == \\"SIMPLE_PINHOLE\\":\\n        fx = fy = p[0]; cx, cy = p[1:3]; dist = [0.0] * 4\\n    elif cam.model == \\"SIMPLE_RADIAL\\":\\n        fx = fy = p[0]; cx, cy = p[1:3]; dist = [p[3], 0.0, 0.0, 0.0]\\n    elif cam.model == \\"PINHOLE\\":\\n        fx, fy, cx, cy = p[:4]; dist = [0.0] * 4\\n    elif cam.model == \\"OPENCV\\":\\n        fx, fy, cx, cy = p[:4]; dist = list(p[4:8])\\n    else:\\n        raise ValueError(f\\"Unsupported camera model {cam.model} (fisheye/360 not handled)\\")\\n    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1.0]], dtype=np.float64)\\n    return K, np.array(dist, dtype=np.float64)\\n\\n\\ndef prepare_dataset(source: Path, out: Path, long_edge: int = 1920, model_name: str = \\"0\\",\\n                    workers: int = 8, log=print) -> SplatData:\\n    \\"\\"\\"Undistorted, resized frames + cameras, like the training run\'s importer. Resumable.\\"\\"\\"\\n    import cv2\\n\\n    source, out = Path(source), Path(out)\\n    meta_path = out / \\"cameras.json\\"\\n    if meta_path.exists():\\n        return load_prepared(out)\\n    model = read_sparse_model(source / \\"sparse\\" / model_name)\\n    if len(model.cameras) != 1:\\n        raise ValueError(\\"Only single-camera datasets are supported.\\")\\n    cam = next(iter(model.cameras.values()))\\n    K, dist = _k_and_dist(cam)\\n    w, h = cam.width, cam.height\\n    s = min(1.0, long_edge / max(w, h))\\n    ow, oh = max(1, round(w * s)), max(1, round(h * s))\\n    if np.any(dist):\\n        newK, _ = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 0, (ow, oh))\\n    else:\\n        newK = K.copy(); newK[0] *= ow / w; newK[1] *= oh / h\\n    mx, my = cv2.initUndistortRectifyMap(K, dist, None, newK, (ow, oh), cv2.CV_32FC1)\\n\\n    images = sorted(model.images.values(), key=lambda im: im.name)\\n    frames_dir = out / \\"frames\\"\\n    frames_dir.mkdir(parents=True, exist_ok=True)\\n\\n    def convert(item):\\n        i, im = item\\n        dest = frames_dir / f\\"frame_{i:06d}.png\\"\\n        if dest.exists():\\n            return\\n        img = cv2.imread(str(source / \\"images\\" / im.name))\\n        if img is None or img.shape[:2] != (h, w):\\n            raise ValueError(f\\"Bad image {im.name}\\")\\n        tmp = frames_dir / f\\"frame_{i:06d}.tmp.png\\"\\n        cv2.imwrite(str(tmp), cv2.remap(img, mx, my, cv2.INTER_LINEAR), [cv2.IMWRITE_PNG_COMPRESSION, 1])\\n        tmp.replace(dest)\\n\\n    log(f\\"[data] {len(images)} images {w}x{h} {cam.model} -> {ow}x{oh} PINHOLE\\")\\n    with ThreadPoolExecutor(workers) as pool:\\n        for n, _ in enumerate(pool.map(convert, enumerate(images)), 1):\\n            if n % 100 == 0 or n == len(images):\\n                log(f\\"[data] {n}/{len(images)}\\")\\n    meta = {\\n        \\"width\\": ow, \\"height\\": oh, \\"K\\": newK.tolist(),\\n        \\"names\\": [im.name for im in images],\\n        \\"frames\\": [f\\"frame_{i:06d}.png\\" for i in range(len(images))],\\n        \\"w2c\\": [im.world_to_camera().tolist() for im in images],\\n    }\\n    meta_path.write_text(json.dumps(meta), encoding=\\"utf-8\\")\\n    return load_prepared(out)\\n\\n\\ndef load_prepared(out: Path) -> SplatData:\\n    meta = json.loads((Path(out) / \\"cameras.json\\").read_text(encoding=\\"utf-8\\"))\\n    return SplatData(Path(out) / \\"frames\\", meta[\\"frames\\"], meta[\\"names\\"], meta[\\"width\\"], meta[\\"height\\"],\\n                     np.array(meta[\\"K\\"]), np.array(meta[\\"w2c\\"]))\\n\\n\\n_PLY_TYPES = {\\"float\\": \\"<f4\\", \\"float32\\": \\"<f4\\", \\"double\\": \\"<f8\\", \\"float64\\": \\"<f8\\", \\"uchar\\": \\"u1\\", \\"uint8\\": \\"u1\\",\\n              \\"char\\": \\"i1\\", \\"int8\\": \\"i1\\", \\"short\\": \\"<i2\\", \\"ushort\\": \\"<u2\\", \\"int\\": \\"<i4\\", \\"int32\\": \\"<i4\\",\\n              \\"uint\\": \\"<u4\\", \\"uint32\\": \\"<u4\\"}\\n\\n\\ndef load_ply(path: Path, device) -> tuple[torch.nn.ParameterDict, int]:\\n    \\"\\"\\"Standard 3DGS PLY (log scales, logit opacity, SH) -> gs_trainer params, sh_degree.\\"\\"\\"\\n    with open(path, \\"rb\\") as f:\\n        props, n, in_vertex, elements = [], 0, False, []\\n        while True:\\n            line = f.readline().decode(\\"ascii\\").strip()\\n            if line.startswith(\\"format\\"):\\n                assert \\"binary_little_endian\\" in line, f\\"Unsupported PLY format: {line}\\"\\n            elif line.startswith(\\"element\\"):\\n                _, name, count = line.split()\\n                elements.append(name)\\n                in_vertex = name == \\"vertex\\"\\n                if in_vertex:\\n                    n = int(count)\\n            elif line.startswith(\\"property\\") and in_vertex:\\n                _, typ, name = line.split()\\n                props.append((name, _PLY_TYPES[typ]))\\n            elif line == \\"end_header\\":\\n                break\\n        assert elements and elements[0] == \\"vertex\\", f\\"Expected vertex data first, got {elements}\\"\\n        arr = np.fromfile(f, dtype=np.dtype(props), count=n)\\n    names = arr.dtype.names\\n    cols = lambda prefix, k: np.stack([arr[f\\"{prefix}{i}\\"] for i in range(k)], axis=1).astype(np.float32)\\n    n_rest = sum(1 for p in names if p.startswith(\\"f_rest_\\"))\\n    k = n_rest // 3\\n    sh_degree = int(round(math.sqrt(k + 1))) - 1\\n    shN = cols(\\"f_rest_\\", n_rest).reshape(n, 3, k).transpose(0, 2, 1) if k else np.zeros((n, 0, 3), np.float32)\\n    t = lambda a: torch.nn.Parameter(torch.as_tensor(np.ascontiguousarray(a), dtype=torch.float32, device=device))\\n    params = torch.nn.ParameterDict({\\n        \\"means\\": t(np.stack([arr[\\"x\\"], arr[\\"y\\"], arr[\\"z\\"]], axis=1)),\\n        \\"scales\\": t(cols(\\"scale_\\", 3)),\\n        \\"quats\\": t(cols(\\"rot_\\", 4)),\\n        \\"opacities\\": t(arr[\\"opacity\\"].astype(np.float32)),\\n        \\"sh0\\": t(cols(\\"f_dc_\\", 3)[:, None, :]),\\n        \\"shN\\": t(shN),\\n    })\\n    return params, sh_degree\\n\\n\\n# ----------------------------------------------------------------------------- metrics\\n\\n\\ndef _psnr(a: torch.Tensor, b: torch.Tensor) -> float:\\n    return 10 * math.log10(1.0 / max(F.mse_loss(a, b).item(), 1e-12))\\n\\n\\ndef _scores(img: torch.Tensor, gt: torch.Tensor, net_lpips) -> dict:\\n    \\"\\"\\"img, gt: (H, W, 3) in [0, 1] on the same device.\\"\\"\\"\\n    return {\\n        \\"psnr\\": _psnr(img, gt),\\n        \\"ssim\\": ssim(img, gt).item(),\\n        \\"lpips\\": net_lpips(img.permute(2, 0, 1)[None] * 2 - 1, gt.permute(2, 0, 1)[None] * 2 - 1).item(),\\n    }\\n\\n\\ndef _w2c_t(data: SplatData, device) -> tuple[torch.Tensor, torch.Tensor]:\\n    return (torch.tensor(data.w2c, dtype=torch.float32, device=device),\\n            torch.tensor(data.K, dtype=torch.float32, device=device))\\n\\n\\n@torch.no_grad()\\ndef score_splat(params, sh_degree: int, data: SplatData, ids: list[int], net_lpips, device=\\"cuda\\") -> dict:\\n    w2c, K = _w2c_t(data, device)\\n    rows = []\\n    for i in ids:\\n        img, _ = render(params, w2c[i], K, data.width, data.height, sh_degree)\\n        rows.append(_scores(img, data.photo(i, device), net_lpips))\\n    return {k: float(np.mean([r[k] for r in rows])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n\\n\\n@torch.no_grad()\\ndef reproduce_check(params, sh_degree: int, data: SplatData, original_metrics: dict, device=\\"cuda\\") -> list[dict]:\\n    \\"\\"\\"Re-render the frames of the training run\'s own evaluation and compare PSNR.\\"\\"\\"\\n    w2c, K = _w2c_t(data, device)\\n    index = {f: i for i, f in enumerate(data.frames)}\\n    rows = []\\n    for v in original_metrics[\\"views\\"]:\\n        i = index[v[\\"frame\\"]]\\n        img, _ = render(params, w2c[i], K, data.width, data.height, sh_degree)\\n        rows.append({\\"frame\\": v[\\"frame\\"], \\"original\\": v[\\"psnr\\"], \\"now\\": _psnr(img, data.photo(i, device))})\\n    return rows\\n\\n\\n# ----------------------------------------------------------------------------- restoration\\n\\n\\ndef make_restore_fn(model, use_ref: bool, long_edge: int = 0):\\n    \\"\\"\\"fn(deg, ref) on (1, 3, H, W) tensors padded to multiples of 64.\\n\\n    ``long_edge`` > 0 runs the model at that size and resizes back (the model\\n    was trained around 1024 px).\\"\\"\\"\\n\\n    def fn(deg, ref):\\n        h, w = deg.shape[-2:]\\n        if long_edge and max(h, w) > long_edge:\\n            s = long_edge / max(h, w)\\n            size = (max(64, round(h * s / 64) * 64), max(64, round(w * s / 64) * 64))\\n            d = F.interpolate(deg, size=size, mode=\\"area\\")\\n            r = F.interpolate(ref, size=size, mode=\\"area\\") if use_ref else None\\n            out = model.restore(d, r)\\n            return F.interpolate(out, size=(h, w), mode=\\"bicubic\\", align_corners=False).clamp(0, 1)\\n        return model.restore(deg, ref if use_ref else None)\\n\\n    return fn\\n\\n\\ndef _to_u8(img: torch.Tensor) -> np.ndarray:\\n    return (img.float().clamp(0, 1).cpu().numpy() * 255 + 0.5).astype(np.uint8)\\n\\n\\ndef _label(img: Image.Image, text: str) -> Image.Image:\\n    from PIL import ImageFont\\n\\n    try:  # Pillow >= 10.1 can size the built-in font\\n        font = ImageFont.load_default(size=max(14, img.width // 28))\\n    except TypeError:\\n        font = ImageFont.load_default()\\n    d = ImageDraw.Draw(img)\\n    pad = max(4, img.width // 120)\\n    box = d.textbbox((pad, pad), text, font=font)\\n    d.rectangle((0, 0, box[2] + pad, box[3] + pad), fill=(0, 0, 0))\\n    d.text((pad, pad), text, fill=(255, 255, 255), font=font)\\n    return img\\n\\n\\ndef _worst_box(err: np.ndarray, size: int) -> tuple[int, int]:\\n    t = torch.from_numpy(err)[None, None]\\n    step = max(1, size // 4)\\n    pooled = F.avg_pool2d(t, size, stride=step)\\n    idx = int(pooled.flatten().argmax())\\n    nx = pooled.shape[-1]\\n    return (idx % nx) * step, (idx // nx) * step\\n\\n\\ndef comparison_image(tiles: dict[str, np.ndarray], out: Path, tile_width: int = 640, box_from: str = \\"render\\") -> None:\\n    \\"\\"\\"One column per image: full frame on top, 3x zoom below where ``tiles[box_from]``\\n    is furthest from the photo (red box).\\"\\"\\"\\n    gt, rnd = tiles[\\"foto\\"].astype(np.float32), tiles[box_from].astype(np.float32)\\n    h, w = gt.shape[:2]\\n    crop = max(48, min(h, w) // 4)\\n    x, y = _worst_box(np.abs(gt - rnd).mean(-1), crop)\\n    s = tile_width / w\\n    top_h = round(h * s)\\n    canvas = Image.new(\\"RGB\\", (tile_width * len(tiles), top_h + tile_width), (20, 20, 20))\\n    for j, (name, arr) in enumerate(tiles.items()):\\n        im = Image.fromarray(arr)\\n        small = im.resize((tile_width, top_h), Image.LANCZOS)\\n        ImageDraw.Draw(small).rectangle((x * s, y * s, (x + crop) * s, (y + crop) * s), outline=(255, 0, 0), width=3)\\n        zoom = im.crop((x, y, x + crop, y + crop)).resize((tile_width, tile_width), Image.LANCZOS)\\n        canvas.paste(_label(small, name), (j * tile_width, 0))\\n        canvas.paste(_label(zoom, f\\"{name} (zoom)\\"), (j * tile_width, top_h))\\n    canvas.save(out, quality=92)\\n\\n\\ndef evaluate_holdout(\\n    params,\\n    sh_degree: int,\\n    data: SplatData,\\n    holdout: list[int],\\n    train_ids: list[int],\\n    methods: dict,\\n    out_dir: Path,\\n    net_lpips,\\n    n_images: int = 12,\\n    restore_long_edge: int = 0,\\n    device=\\"cuda\\",\\n    log=print,\\n) -> dict:\\n    \\"\\"\\"methods: name -> (loader() -> Restorer, use_ref). Scores render and each method vs the held-out photos.\\"\\"\\"\\n    out_dir = Path(out_dir)\\n    img_dir = out_dir / \\"images\\"\\n    img_dir.mkdir(parents=True, exist_ok=True)\\n    res_path = out_dir / \\"per_frame.json\\"\\n    res = json.loads(res_path.read_text(encoding=\\"utf-8\\")) if res_path.exists() else {}\\n    w2c, K = _w2c_t(data, device)\\n    centers, forwards = data.centers, data.forwards\\n\\n    with torch.no_grad():\\n        renders = {}\\n        for i in holdout:\\n            img, _ = render(params, w2c[i], K, data.width, data.height, sh_degree)\\n            renders[i] = img.half().cpu()\\n            res.setdefault(str(i), {})[\\"render\\"] = _scores(img, data.photo(i, device), net_lpips)\\n    by_render = sorted(holdout, key=lambda i: res[str(i)][\\"render\\"][\\"psnr\\"])\\n    shown = sorted(set(by_render[: n_images // 2]) | set(subsample(holdout, n_images - n_images // 2)))\\n    log(f\\"[eval] {len(holdout)} held-out frames; render PSNR \\"\\n        f\\"{np.mean([res[str(i)][\'render\'][\'psnr\'] for i in holdout]):.2f}\\")\\n    refs = {i: nearest_reference(i, train_ids, centers, forwards) for i in holdout}\\n\\n    for name, (loader, use_ref) in methods.items():\\n        todo = [i for i in holdout if name not in res[str(i)] or (i in shown and not (img_dir / f\\"{name}_{i:06d}.jpg\\").exists())]\\n        if not todo:\\n            continue\\n        model = loader()\\n        fn = make_restore_fn(model, use_ref, restore_long_edge)\\n        with torch.no_grad():\\n            for n, i in enumerate(todo, 1):\\n                deg = renders[i].to(device).float()\\n                ref = data.photo(refs[i], device)\\n                out = restore_full(fn, deg, ref)\\n                res[str(i)][name] = _scores(out, data.photo(i, device), net_lpips)\\n                if i in shown:\\n                    Image.fromarray(_to_u8(out)).save(img_dir / f\\"{name}_{i:06d}.jpg\\", quality=95)\\n                if n % 20 == 0 or n == len(todo):\\n                    log(f\\"[eval] {name}: {n}/{len(todo)}\\")\\n        res_path.write_text(json.dumps(res), encoding=\\"utf-8\\")\\n        del model, fn\\n        torch.cuda.empty_cache()\\n    res_path.write_text(json.dumps(res), encoding=\\"utf-8\\")\\n\\n    for i in shown:\\n        tiles = {\\"foto\\": _to_u8(data.photo(i, \\"cpu\\")), \\"render\\": _to_u8(renders[i])}\\n        for name in methods:\\n            p = img_dir / f\\"{name}_{i:06d}.jpg\\"\\n            if p.exists():\\n                tiles[name] = np.asarray(Image.open(p).convert(\\"RGB\\"))\\n        tiles[\\"referans\\"] = _to_u8(data.photo(refs[i], \\"cpu\\"))\\n        comparison_image(tiles, out_dir / f\\"compare_{i:06d}.jpg\\")\\n\\n    summary = summarise(res, holdout, [\\"render\\", *methods])\\n    summary[\\"shown_frames\\"] = shown\\n    (out_dir / \\"summary.json\\").write_text(json.dumps(summary, indent=2), encoding=\\"utf-8\\")\\n    md = eval_markdown(summary)\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return summary\\n\\n\\ndef summarise(res: dict, ids: list[int], names: list[str]) -> dict:\\n    out = {\\"frames\\": len(ids), \\"methods\\": {}}\\n    for name in names:\\n        rows = [res[str(i)][name] for i in ids if name in res[str(i)]]\\n        if not rows:\\n            continue\\n        base = [res[str(i)][\\"render\\"] for i in ids if name in res[str(i)]]\\n        out[\\"methods\\"][name] = {\\n            \\"psnr\\": float(np.mean([r[\\"psnr\\"] for r in rows])),\\n            \\"ssim\\": float(np.mean([r[\\"ssim\\"] for r in rows])),\\n            \\"lpips\\": float(np.mean([r[\\"lpips\\"] for r in rows])),\\n            \\"lpips_wins\\": int(sum(r[\\"lpips\\"] < b[\\"lpips\\"] for r, b in zip(rows, base))),\\n            \\"psnr_wins\\": int(sum(r[\\"psnr\\"] > b[\\"psnr\\"] for r, b in zip(rows, base))),\\n            \\"n\\": len(rows),\\n        }\\n    worst = sorted(ids, key=lambda i: res[str(i)][\\"render\\"][\\"psnr\\"])[:8]\\n    out[\\"worst\\"] = [{\\"frame\\": i, **{n: res[str(i)][n] for n in names if n in res[str(i)]}} for i in worst]\\n    return out\\n\\n\\ndef eval_markdown(s: dict) -> str:\\n    r = s[\\"methods\\"][\\"render\\"]\\n    lines = [\\n        f\\"## Saklanan kareler: render vs Difix ({s[\'frames\']} kare, splat bu fotoğrafları hiç görmedi)\\",\\n        \\"\\",\\n        \\"| yöntem | PSNR ↑ | Δ | SSIM ↑ | LPIPS ↓ | Δ | LPIPS\'i iyileşen kare | PSNR\'ı iyileşen kare |\\",\\n        \\"|---|---|---|---|---|---|---|---|\\",\\n    ]\\n    for name, m in s[\\"methods\\"].items():\\n        lines.append(\\n            f\\"| {name} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - r[\'psnr\']:+.2f} | {m[\'ssim\']:.4f} | {m[\'lpips\']:.4f} | \\"\\n            f\\"{m[\'lpips\'] - r[\'lpips\']:+.4f} | {m[\'lpips_wins\']}/{m[\'n\']} | {m[\'psnr_wins\']}/{m[\'n\']} |\\"\\n        )\\n    names = [n for n in s[\\"methods\\"] if n != \\"render\\"]\\n    lines += [\\"\\", \\"**En kötü 8 kare (render PSNR\'a göre), PSNR / LPIPS:**\\", \\"\\",\\n              \\"| kare | render | \\" + \\" | \\".join(names) + \\" |\\", \\"|---|---\\" + \\"|---\\" * len(names) + \\"|\\"]\\n    for w in s[\\"worst\\"]:\\n        cells = [f\\"{w[\'render\'][\'psnr\']:.2f} / {w[\'render\'][\'lpips\']:.3f}\\"]\\n        cells += [f\\"{w[n][\'psnr\']:.2f} / {w[n][\'lpips\']:.3f}\\" if n in w else \\"-\\" for n in names]\\n        lines.append(f\\"| {w[\'frame\']} | \\" + \\" | \\".join(cells) + \\" |\\")\\n    lines += [\\"\\", \\"Okuma: LPIPS (algısal fark) düşüyorsa ve PSNR belirgin düşmüyorsa düzeltme gerçek. \\"\\n              \\"PSNR düşüp LPIPS iyileşiyorsa model keskin ama uydurma detay ekliyor olabilir; zoom\'lara bak.\\"]\\n    return \\"\\\\n\\".join(lines)\\n\\n\\n# ----------------------------------------------------------------------------- distill back\\n\\n\\ndef pseudo_poses(data: SplatData, train_ids: list[int], n: int, shift_frac: float) -> list[tuple[int, np.ndarray]]:\\n    \\"\\"\\"New viewpoints: training cameras moved sideways / up by ``shift_frac`` of the camera path size.\\n\\n    Returns (source training frame, w2c) pairs; the source photo is the restorer\'s reference.\\"\\"\\"\\n    centers = data.centers\\n    extent = float(np.linalg.norm(centers[train_ids].max(0) - centers[train_ids].min(0)))\\n    offset = shift_frac * extent\\n    dirs = [(0, 1.0), (0, -1.0), (1, 1.0), (1, -1.0)]  # camera right / left / down / up\\n    out = []\\n    for j, i in enumerate(subsample(train_ids, n)):\\n        axis, sign = dirs[j % len(dirs)]\\n        w2c = data.w2c[i].copy()\\n        R = w2c[:3, :3]\\n        c = centers[i] + sign * offset * R[axis]\\n        w2c[:3, 3] = -R @ c\\n        out.append((i, w2c))\\n    return out\\n\\n\\ndef distill_back(\\n    params,\\n    sh_degree: int,\\n    data: SplatData,\\n    holdout: list[int],\\n    train_ids: list[int],\\n    methods: dict,\\n    out_dir: Path,\\n    net_lpips,\\n    steps: int = 3000,\\n    n_pseudo: int = 120,\\n    shift_frac: float = 0.03,\\n    pseudo_share: float = 0.3,\\n    train_stride: int = 2,\\n    restore_long_edge: int = 0,\\n    device=\\"cuda\\",\\n    log=print,\\n) -> dict:\\n    \\"\\"\\"Fine-tune the splat: real photos only (control) vs photos + restored pseudo views.\\"\\"\\"\\n    out_dir = Path(out_dir)\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    res_path = out_dir / \\"results.json\\"\\n    results = json.loads(res_path.read_text(encoding=\\"utf-8\\")) if res_path.exists() else {}\\n    start_state = params_to_cpu_state(params)\\n    w2c_t, K = _w2c_t(data, device)\\n    scene_scale = 1.1 * float(np.linalg.norm(data.centers[train_ids] - data.centers[train_ids].mean(0), axis=1).max())\\n    cfg = TrainConfig(steps=steps, sh_degree=sh_degree, densify=False, lr_means=1.6e-5, lr_means_final_frac=0.1, seed=0)\\n    score = lambda p: score_splat(p, sh_degree, data, holdout, net_lpips, device)\\n\\n    if \\"start\\" not in results:\\n        results[\\"start\\"] = score(params)\\n        res_path.write_text(json.dumps(results, indent=2), encoding=\\"utf-8\\")\\n    ft_ids = train_ids[::train_stride]\\n    log(f\\"[distill] loading {len(ft_ids)} training photos (every {train_stride}.)\\")\\n    train_views = [View(data.photo_u8(i), w2c_t[i], K) for i in ft_ids]\\n\\n    if \\"control\\" not in results:\\n        p = train_gaussians(train_views, data.width, data.height, cfg, scene_scale=scene_scale,\\n                            params=params_from_state(start_state, device), log=log)\\n        results[\\"control\\"] = score(p)\\n        del p\\n        torch.cuda.empty_cache()\\n        res_path.write_text(json.dumps(results, indent=2), encoding=\\"utf-8\\")\\n        log(f\\"[distill] control: {results[\'control\']}\\")\\n\\n    poses = pseudo_poses(data, train_ids, n_pseudo, shift_frac)\\n    base = params_from_state(start_state, device)\\n    with torch.no_grad():\\n        renders = [render(base, torch.tensor(w, dtype=torch.float32, device=device), K,\\n                          data.width, data.height, sh_degree)[0].half().cpu() for _, w in poses]\\n    del base\\n    for name, (loader, use_ref) in methods.items():\\n        if name in results:\\n            continue\\n        model = loader()\\n        fn = make_restore_fn(model, use_ref, restore_long_edge)\\n        pseudo = []\\n        with torch.no_grad():\\n            for (src, w), rnd in zip(poses, renders):\\n                out = restore_full(fn, rnd.to(device).float(), data.photo(src, device))\\n                pseudo.append(torch.from_numpy(_to_u8(out)))\\n        del model, fn\\n        torch.cuda.empty_cache()\\n        ex_dir = out_dir / f\\"pseudo_{name}\\"\\n        ex_dir.mkdir(exist_ok=True)\\n        for k in subsample(list(range(len(poses))), 6):\\n            tiles = {\\"render (yeni kamera)\\": _to_u8(renders[k]), name: pseudo[k].numpy(),\\n                     \\"kaynak foto\\": _to_u8(data.photo(poses[k][0], \\"cpu\\"))}\\n            tw = 800\\n            row = [_label(Image.fromarray(a).resize((tw, round(tw * data.height / data.width)), Image.LANCZOS), t)\\n                   for t, a in tiles.items()]\\n            canvas = Image.new(\\"RGB\\", (tw * len(row), row[0].height))\\n            for j, im in enumerate(row):\\n                canvas.paste(im, (tw * j, 0))\\n            canvas.save(ex_dir / f\\"pseudo_{k:03d}.jpg\\", quality=90)\\n        w_pseudo = pseudo_share / (1 - pseudo_share) * len(train_views) / max(len(pseudo), 1)\\n        views = train_views + [View(img, torch.tensor(w, dtype=torch.float32, device=device), K, weight=w_pseudo)\\n                               for (_, w), img in zip(poses, pseudo)]\\n        p = train_gaussians(views, data.width, data.height, cfg, scene_scale=scene_scale,\\n                            params=params_from_state(start_state, device), log=log)\\n        results[name] = score(p)\\n        del p, pseudo, views\\n        torch.cuda.empty_cache()\\n        res_path.write_text(json.dumps(results, indent=2), encoding=\\"utf-8\\")\\n        log(f\\"[distill] {name}: {results[name]}\\")\\n\\n    meta = {\\"steps\\": steps, \\"n_pseudo\\": n_pseudo, \\"shift_frac\\": shift_frac, \\"pseudo_share\\": pseudo_share,\\n            \\"train_photos\\": len(ft_ids), \\"holdout\\": len(holdout)}\\n    md = distill_markdown(results, meta)\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return results\\n\\n\\ndef distill_markdown(results: dict, meta: dict) -> str:\\n    c = results.get(\\"control\\", results[\\"start\\"])\\n    lines = [\\n        f\\"## Splat\'e geri besleme ({meta[\'steps\']} adım ince ayar, {meta[\'n_pseudo\']} yeni bakış, \\"\\n        f\\"kaydırma %{meta[\'shift_frac\'] * 100:.0f})\\",\\n        \\"\\",\\n        f\\"Ölçüm: {meta[\'holdout\']} saklanan kare (hiçbir koşulda eğitimde kullanılmadı).\\",\\n        \\"\\",\\n        \\"| koşul | PSNR ↑ | Δ control | SSIM ↑ | LPIPS ↓ | Δ control |\\",\\n        \\"|---|---|---|---|---|---|\\",\\n    ]\\n    for name, m in results.items():\\n        lines.append(f\\"| {name} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - c[\'psnr\']:+.2f} | {m[\'ssim\']:.4f} | \\"\\n                     f\\"{m[\'lpips\']:.4f} | {m[\'lpips\'] - c[\'lpips\']:+.4f} |\\")\\n    lines += [\\"\\", \\"* `start`: mevcut splat. `control`: aynı ince ayar, sadece gerçek fotoğraflar.\\",\\n              \\"* Difix koşulu `control`\'ü geçiyorsa düzeltilmiş yeni bakışlar splat\'in kendisini iyileştirmiş demektir.\\"]\\n    return \\"\\\\n\\".join(lines)\\n", "lib/splat_restorer/pairs.py": "\\"\\"\\"Generate (degraded render, real photo, reference photo) pairs for one scene.\\n\\nFor each recipe a deliberately weak splat is trained on a sparse subset of the\\nframes; every non-training frame is rendered and saved next to its real photo.\\nThe reference is the real photo of the closest training camera.\\n\\nOutput is one tar per scene so Drive sees a single large atomic file:\\n\\n    index.json\\n    <recipe>/deg/<frame>.jpg   render of the weak splat\\n    <recipe>/gt/<frame>.jpg    real photo at the same pose\\n    <recipe>/ref/<frame>.jpg   real photo of a training camera (shared)\\n    eval/<frame>.jpg           (test scenes) never-trained frames for distill-back\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport shutil\\nimport tarfile\\nimport time\\nimport zlib\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom PIL import Image\\n\\nfrom .gs_trainer import TrainConfig, View, params_to_cpu_state, psnr, render, train_gaussians\\nfrom .scene import Scene, load_colmap_scene\\nfrom .splits import RECIPES, make_split, nearest_reference, subsample\\nfrom .workspace import Workspace, atomic_copy\\n\\nJPEG_QUALITY = 95\\n\\n\\ndef _save_jpg(arr: np.ndarray, path: Path) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    Image.fromarray((np.clip(arr, 0, 1) * 255 + 0.5).astype(np.uint8)).save(path, quality=JPEG_QUALITY)\\n\\n\\ndef _fname(i: int) -> str:\\n    return f\\"{i:05d}.jpg\\"\\n\\n\\ndef generate_scene_pairs(\\n    ws: Workspace,\\n    scene_name: str,\\n    scene_dir: Path,\\n    recipes: list[str],\\n    is_test: bool,\\n    max_pairs_per_recipe: int = 100,\\n    images_subdir: str = \\"images_4\\",\\n    device: str = \\"cuda\\",\\n    log=print,\\n) -> Path:\\n    t0 = time.time()\\n    scene = load_colmap_scene(scene_dir, images_subdir=images_subdir, name=scene_name)\\n    log(f\\"  {scene_name[:12]}: {scene.num_frames} frames {scene.width}x{scene.height}, {len(scene.points_xyz)} points\\")\\n    # uint8 keeps a 300-frame 960p scene at ~0.5 GB of RAM.\\n    photos_u8 = [(scene.load_image(i) * 255 + 0.5).astype(np.uint8) for i in range(scene.num_frames)]\\n    photo = lambda i: photos_u8[i].astype(np.float32) / 255.0\\n    centers, forwards = scene.camera_centers(), scene.view_dirs()\\n    Ks = torch.as_tensor(scene.Ks, device=device)\\n    w2c = torch.as_tensor(scene.w2c, device=device)\\n\\n    work = ws.local / \\"pairs_tmp\\" / scene_name\\n    shutil.rmtree(work, ignore_errors=True)\\n    work.mkdir(parents=True)\\n    index = {\\n        \\"scene\\": scene_name,\\n        \\"width\\": scene.width,\\n        \\"height\\": scene.height,\\n        \\"is_test\\": is_test,\\n        \\"cameras\\": {\\"K\\": scene.Ks.tolist(), \\"w2c\\": scene.w2c.tolist()},\\n        \\"recipes\\": {},\\n    }\\n\\n    for rname in recipes:\\n        recipe = RECIPES[rname]\\n        split = make_split(scene.num_frames, recipe)\\n        views = [View(torch.as_tensor(photo(i), device=device), w2c[i], Ks[i]) for i in split.train]\\n        cfg = TrainConfig(steps=recipe.steps, seed=zlib.crc32(scene_name.encode()) % 10_000)\\n        t1 = time.time()\\n        params = train_gaussians(\\n            views, scene.width, scene.height, cfg, init_xyz=scene.points_xyz, init_rgb=scene.points_rgb\\n        )\\n        train_s = time.time() - t1\\n\\n        pair_frames = subsample(split.pair, max_pairs_per_recipe)\\n        items = []\\n        with torch.no_grad():\\n            for i in pair_frames:\\n                img, _ = render(params, w2c[i], Ks[i], scene.width, scene.height, cfg.sh_degree)\\n                deg = img.float().cpu().numpy()\\n                ref = nearest_reference(i, split.train, centers, forwards)\\n                _save_jpg(deg, work / rname / \\"deg\\" / _fname(i))\\n                _save_jpg(photo(i), work / rname / \\"gt\\" / _fname(i))\\n                ref_path = work / rname / \\"ref\\" / _fname(ref)\\n                if not ref_path.exists():\\n                    _save_jpg(photo(ref), ref_path)\\n                items.append(\\n                    {\\n                        \\"frame\\": i,\\n                        \\"ref\\": ref,\\n                        \\"deg\\": f\\"{rname}/deg/{_fname(i)}\\",\\n                        \\"gt\\": f\\"{rname}/gt/{_fname(i)}\\",\\n                        \\"ref_path\\": f\\"{rname}/ref/{_fname(ref)}\\",\\n                        \\"psnr\\": round(psnr(img, torch.as_tensor(photo(i), device=device)), 3),\\n                    }\\n                )\\n        if is_test:\\n            # Distill-back needs every training photo and the splat itself.\\n            for i in split.train:\\n                p = work / rname / \\"ref\\" / _fname(i)\\n                if not p.exists():\\n                    _save_jpg(photo(i), p)\\n            out = ws.splat_path(scene_name, rname)\\n            out.parent.mkdir(parents=True, exist_ok=True)\\n            tmp = ws.local / \\"splat_tmp.pt\\"\\n            torch.save(params_to_cpu_state(params), tmp)\\n            atomic_copy(tmp, out)\\n            tmp.unlink(missing_ok=True)\\n\\n        mean_psnr = float(np.mean([it[\\"psnr\\"] for it in items])) if items else float(\\"nan\\")\\n        index[\\"recipes\\"][rname] = {\\n            \\"train\\": split.train,\\n            \\"pair\\": split.pair,\\n            \\"eval\\": split.eval,\\n            \\"steps\\": recipe.steps,\\n            \\"num_gaussians\\": int(len(params[\\"means\\"])),\\n            \\"train_seconds\\": round(train_s, 1),\\n            \\"mean_psnr\\": round(mean_psnr, 3),\\n            \\"items\\": items,\\n        }\\n        log(\\n            f\\"    {rname}: train {len(split.train)} views, {len(params[\'means\'])} gaussians, \\"\\n            f\\"{train_s:.0f}s, {len(items)} pairs, render PSNR {mean_psnr:.2f}\\"\\n        )\\n        del params, views\\n        torch.cuda.empty_cache()\\n\\n    if is_test:\\n        ev = make_split(scene.num_frames, RECIPES[recipes[0]]).eval\\n        for i in ev:\\n            _save_jpg(photo(i), work / \\"eval\\" / _fname(i))\\n\\n    (work / \\"index.json\\").write_text(json.dumps(index), encoding=\\"utf-8\\")\\n    local_tar = ws.local / f\\"{scene_name}.tar\\"\\n    with tarfile.open(local_tar, \\"w\\") as tar:\\n        tar.add(work, arcname=scene_name)\\n    atomic_copy(local_tar, ws.pair_tar(scene_name))\\n    local_tar.unlink(missing_ok=True)\\n    shutil.rmtree(work, ignore_errors=True)\\n    log(f\\"  {scene_name[:12]}: done in {time.time() - t0:.0f}s\\")\\n    return ws.pair_tar(scene_name)\\n\\n\\ndef extract_pairs(ws: Workspace, scenes: list[str]) -> Path:\\n    \\"\\"\\"Extract the tars of ``scenes`` to local disk (skips already extracted ones).\\"\\"\\"\\n    out = ws.local_pairs\\n    out.mkdir(parents=True, exist_ok=True)\\n    for s in scenes:\\n        if (out / s / \\"index.json\\").exists():\\n            continue\\n        tar_path = ws.pair_tar(s)\\n        if not tar_path.exists():\\n            raise FileNotFoundError(f\\"Pairs for scene {s} are missing: {tar_path}\\")\\n        with tarfile.open(tar_path) as tar:\\n            tar.extractall(out)\\n    return out\\n\\n\\ndef load_index(pairs_root: Path, scene: str) -> dict:\\n    return json.loads((pairs_root / scene / \\"index.json\\").read_text(encoding=\\"utf-8\\"))\\n", "lib/splat_restorer/pose_repair.py": "\\"\\"\\"Pose-repair experiment: does a better SfM fix the frames the splat cannot fit?\\n\\nOn IMG_5966 a few short video segments render at 11-14 dB even on frames the\\nsplat was trained on: their camera poses are wrong, so they feed inconsistent\\nsupervision to every trainer. This module compares SfM reconstructions of the\\nsame images end to end:\\n\\n1. ``sfm_stats``: registration, reprojection error and track statistics, and\\n   whether the known-bad frames were registered at all.\\n2. Each reconstruction trains the app\'s trainer (``train_ab``) with the SAME\\n   held-out photos, matched by file name (each SfM uses its own cameras, so\\n   frame indices and coordinate frames differ between reconstructions).\\n3. ``frame_fit``: render PSNR of every registered frame, so the bad segments\\n   can be compared before/after even though most of them are training frames.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nimport shutil\\nimport struct\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image\\n\\nfrom .colmap_io import read_sparse_model\\nfrom .gs_trainer import render\\nfrom .own_splat import SplatData, _to_u8, _w2c_t, comparison_image, load_ply\\n\\n# Common `spirula sfm auto` flags: the capture is one phone video, one lens, frames in file order\\n# (`--sequence .` as the app\'s preprocessing runner passes for a single video; the current SfM used it too).\\nSFM_COMMON = [\\"--data-type\\", \\"video\\", \\"--camera-model\\", \\"opencv\\", \\"--camera-mode\\", \\"single\\",\\n              \\"--sequence\\", \\".\\", \\"--no-masks\\"]\\n\\n# Extra flags per reconstruction; None = the dataset\'s existing SfM (Spirula, quality high, SIFT).\\nSFM_VARIANTS: dict[str, list[str] | None] = {\\n    \\"current\\": None,\\n    \\"extreme_sift\\": [\\"--quality\\", \\"extreme\\"],\\n    \\"extreme_aliked\\": [\\"--quality\\", \\"extreme\\", \\"--features\\", \\"aliked-n16rot\\", \\"--matcher\\", \\"lightglue\\"],\\n    \\"high_aliked_bottomup\\": [\\"--quality\\", \\"high\\", \\"--features\\", \\"aliked-n16rot\\", \\"--matcher\\", \\"lightglue\\",\\n                             \\"--mapper\\", \\"bottom-up\\"],\\n}\\n\\n# Help-text tokens each variant needs, so an unsupported option skips the variant instead of failing the run.\\nSFM_REQUIRES = {\\n    \\"extreme_sift\\": [\\"extreme\\"],\\n    \\"extreme_aliked\\": [\\"extreme\\", \\"aliked-n16rot\\", \\"lightglue\\"],\\n    \\"high_aliked_bottomup\\": [\\"aliked-n16rot\\", \\"lightglue\\", \\"bottom-up\\"],\\n}\\n\\n\\ndef sfm_args(binary: str, images: Path, workspace: Path, extra: list[str], device: str) -> list[str]:\\n    return [str(binary), \\"sfm\\", \\"auto\\", str(images), \\"--output\\", str(workspace), *SFM_COMMON, *extra,\\n            \\"--device\\", device, \\"--lang\\", \\"en\\"]\\n\\n\\n# ----------------------------------------------------------------------------- COLMAP statistics\\n\\n\\ndef _read(f, fmt: str):\\n    size = struct.calcsize(\\"<\\" + fmt)\\n    data = f.read(size)\\n    if len(data) != size:\\n        raise ValueError(\\"Truncated COLMAP binary file\\")\\n    return struct.unpack(\\"<\\" + fmt, data)\\n\\n\\ndef _image_observations(path: Path) -> dict[str, tuple]:\\n    \\"\\"\\"Per image name: (qvec, tvec, camera_id, xy of observations that have a 3D point, their point ids).\\"\\"\\"\\n    out = {}\\n    with open(path, \\"rb\\") as f:\\n        (n,) = _read(f, \\"Q\\")\\n        for _ in range(n):\\n            props = _read(f, \\"idddddddi\\")\\n            name = bytearray()\\n            while (c := f.read(1)) != b\\"\\u0000\\":\\n                if not c:\\n                    raise ValueError(\\"Truncated image name\\")\\n                name += c\\n            (n2d,) = _read(f, \\"Q\\")\\n            obs = np.frombuffer(f.read(24 * n2d), dtype=[(\\"x\\", \\"<f8\\"), (\\"y\\", \\"<f8\\"), (\\"id\\", \\"<i8\\")])\\n            keep = obs[\\"id\\"] >= 0\\n            out[name.decode(\\"utf-8\\")] = (np.array(props[1:5]), np.array(props[5:8]), props[8],\\n                                         np.stack([obs[\\"x\\"][keep], obs[\\"y\\"][keep]], 1), obs[\\"id\\"][keep])\\n    return out\\n\\n\\ndef _point_stats(path: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\\n    \\"\\"\\"(point ids, xyz, track length) per 3D point. (Spirula leaves the stored error field at 0.)\\"\\"\\"\\n    with open(path, \\"rb\\") as f:\\n        (n,) = _read(f, \\"Q\\")\\n        ids = np.empty(n, np.int64)\\n        xyz = np.empty((n, 3), np.float64)\\n        track = np.empty(n, np.int64)\\n        for i in range(n):\\n            props = _read(f, \\"QdddBBBd\\")\\n            ids[i] = props[0]\\n            xyz[i] = props[1:4]\\n            (t,) = _read(f, \\"Q\\")\\n            track[i] = t\\n            f.seek(8 * t, 1)\\n    return ids, xyz, track\\n\\n\\ndef _project(cam, pts_cam: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Camera-frame points -> pixels with the COLMAP camera model (NaN for unsupported models).\\"\\"\\"\\n    p = cam.params\\n    x, y = pts_cam[:, 0] / pts_cam[:, 2], pts_cam[:, 1] / pts_cam[:, 2]\\n    if cam.model in (\\"SIMPLE_PINHOLE\\", \\"SIMPLE_RADIAL\\", \\"RADIAL\\"):\\n        f, cx, cy = p[0], p[1], p[2]\\n        k1 = p[3] if len(p) > 3 else 0.0\\n        k2 = p[4] if cam.model == \\"RADIAL\\" else 0.0\\n        r2 = x * x + y * y\\n        d = 1 + k1 * r2 + k2 * r2 * r2\\n        return np.stack([f * x * d + cx, f * y * d + cy], 1)\\n    if cam.model in (\\"PINHOLE\\", \\"OPENCV\\"):\\n        fx, fy, cx, cy = p[:4]\\n        if cam.model == \\"OPENCV\\":\\n            k1, k2, p1, p2 = p[4:8]\\n            r2 = x * x + y * y\\n            radial = 1 + k1 * r2 + k2 * r2 * r2\\n            x, y = (x * radial + 2 * p1 * x * y + p2 * (r2 + 2 * x * x),\\n                    y * radial + p1 * (r2 + 2 * y * y) + 2 * p2 * x * y)\\n        return np.stack([fx * x + cx, fy * y + cy], 1)\\n    return np.full((len(pts_cam), 2), np.nan)\\n\\n\\ndef reprojection_errors(sparse_dir: Path) -> tuple[dict[str, np.ndarray], np.ndarray]:\\n    \\"\\"\\"Per image name: pixel reprojection error of each observation; and track lengths.\\"\\"\\"\\n    from .colmap_io import qvec_to_rotmat\\n\\n    sparse_dir = Path(sparse_dir)\\n    cams = read_sparse_model(sparse_dir).cameras\\n    pid, xyz, track = _point_stats(sparse_dir / \\"points3D.bin\\")\\n    order = np.argsort(pid)\\n    pid, xyz = pid[order], xyz[order]\\n    errors = {}\\n    for name, (q, t, cam_id, xy, ids) in _image_observations(sparse_dir / \\"images.bin\\").items():\\n        if not len(ids):\\n            errors[name] = np.empty(0)\\n            continue\\n        pos = np.searchsorted(pid, ids)\\n        pts = xyz[np.clip(pos, 0, len(pid) - 1)] @ qvec_to_rotmat(q).T + t\\n        errors[name] = np.linalg.norm(_project(cams[cam_id], pts) - xy, axis=1)\\n    return errors, track\\n\\n\\ndef components(workspace: Path) -> list[tuple[Path, int]]:\\n    \\"\\"\\"sparse/N models in a Spirula/COLMAP workspace, largest first.\\"\\"\\"\\n    found = []\\n    for d in sorted((Path(workspace) / \\"sparse\\").iterdir()):\\n        if (d / \\"images.bin\\").is_file() and (d / \\"cameras.bin\\").is_file() and (d / \\"points3D.bin\\").is_file():\\n            with open(d / \\"images.bin\\", \\"rb\\") as f:\\n                found.append((d, _read(f, \\"Q\\")[0]))\\n    return sorted(found, key=lambda x: -x[1])\\n\\n\\ndef sfm_stats(sparse_dir: Path, bad_names: list[str], holdout_names: list[str], n_input: int) -> dict:\\n    errors, track = reprojection_errors(sparse_dir)\\n    cams = read_sparse_model(Path(sparse_dir)).cameras\\n    all_err = np.concatenate([e for e in errors.values() if len(e)]) if errors else np.empty(0)\\n    image_err = {n: float(np.median(e)) for n, e in errors.items() if len(e)}\\n    bad_in = [n for n in bad_names if n in errors]\\n    return {\\n        \\"registered\\": len(errors),\\n        \\"coverage\\": len(errors) / max(n_input, 1),\\n        \\"cameras\\": len(cams),\\n        \\"points\\": int(len(track)),\\n        \\"reproj_error_px\\": float(np.mean(all_err)) if len(all_err) else float(\\"nan\\"),\\n        \\"reproj_error_median_px\\": float(np.median(all_err)) if len(all_err) else float(\\"nan\\"),\\n        \\"track_length\\": float(track.mean()) if len(track) else float(\\"nan\\"),\\n        \\"obs_per_image_median\\": float(np.median([len(e) for e in errors.values()])) if errors else 0.0,\\n        \\"bad_registered\\": len(bad_in),\\n        \\"bad_total\\": len(bad_names),\\n        \\"bad_obs_median\\": float(np.median([len(errors[n]) for n in bad_in])) if bad_in else 0.0,\\n        \\"bad_reproj_median_px\\": float(np.median([image_err[n] for n in bad_in if n in image_err])) if bad_in else float(\\"nan\\"),\\n        \\"holdout_registered\\": sum(n in errors for n in holdout_names),\\n        \\"holdout_total\\": len(holdout_names),\\n    }\\n\\n\\ndef make_dataset(images: Path, sparse_dir: Path, out: Path) -> Path:\\n    \\"\\"\\"Folder with images/ (symlink, or copy where symlinks are unavailable) and sparse/0.\\"\\"\\"\\n    out = Path(out)\\n    (out / \\"sparse\\").mkdir(parents=True, exist_ok=True)\\n    target = out / \\"sparse\\" / \\"0\\"\\n    if not target.exists():\\n        shutil.copytree(sparse_dir, target)\\n    link = out / \\"images\\"\\n    if not link.exists():\\n        try:\\n            link.symlink_to(Path(images).resolve(), target_is_directory=True)\\n        except OSError:\\n            shutil.copytree(images, link)\\n    return out\\n\\n\\ndef holdout_indices(data: SplatData, holdout_names: list[str]) -> tuple[list[int], list[str]]:\\n    \\"\\"\\"Indices of the fixed held-out photos in this reconstruction, and the names it did not register.\\"\\"\\"\\n    index = {n: i for i, n in enumerate(data.names)}\\n    return [index[n] for n in holdout_names if n in index], [n for n in holdout_names if n not in index]\\n\\n\\n# ----------------------------------------------------------------------------- per-frame fit\\n\\n\\n@torch.no_grad()\\ndef frame_fit(ply: Path, data: SplatData, keep_renders: set[str], render_dir: Path, device=\\"cuda\\") -> dict[str, float]:\\n    \\"\\"\\"Render PSNR of every registered frame (train and held-out), keyed by image name.\\"\\"\\"\\n    params, deg = load_ply(ply, device)\\n    w2c, K = _w2c_t(data, device)\\n    render_dir.mkdir(parents=True, exist_ok=True)\\n    out = {}\\n    for i, name in enumerate(data.names):\\n        img, _ = render(params, w2c[i], K, data.width, data.height, deg)\\n        out[name] = 10 * math.log10(1.0 / max(F.mse_loss(img, data.photo(i, device)).item(), 1e-12))\\n        if name in keep_renders:\\n            Image.fromarray(_to_u8(img)).save(render_dir / f\\"{Path(name).stem}.jpg\\", quality=92)\\n    del params\\n    torch.cuda.empty_cache()\\n    return out\\n\\n\\ndef summarise_variant(name: str, stats: dict | None, frames: dict, fit: dict[str, float], data: SplatData,\\n                      holdout_names: list[str], bad_names: list[str], info: dict) -> dict:\\n    \\"\\"\\"frames: held-out metrics keyed by this reconstruction\'s frame index (as train_ab returns them).\\"\\"\\"\\n    by_name = {data.names[int(i)]: m for i, m in frames.items()}\\n    bad = set(bad_names)\\n\\n    def mean(rows, key):\\n        return float(np.mean([r[key] for r in rows])) if rows else float(\\"nan\\")\\n\\n    all_rows = [by_name[n] for n in holdout_names if n in by_name]\\n    good_rows = [by_name[n] for n in holdout_names if n in by_name and n not in bad]\\n    bad_rows = [by_name[n] for n in holdout_names if n in by_name and n in bad]\\n    holdout = set(holdout_names)\\n    train_fit = [v for n, v in fit.items() if n not in holdout]\\n    bad_fit = [fit[n] for n in bad_names if n in fit]\\n    return {\\n        \\"name\\": name,\\n        \\"sfm\\": stats,\\n        \\"holdout_n\\": len(all_rows),\\n        \\"psnr\\": mean(all_rows, \\"psnr\\"), \\"ssim\\": mean(all_rows, \\"ssim\\"), \\"lpips\\": mean(all_rows, \\"lpips\\"),\\n        \\"psnr_posefree\\": mean(good_rows, \\"psnr\\"), \\"lpips_posefree\\": mean(good_rows, \\"lpips\\"),\\n        \\"psnr_bad_holdout\\": mean(bad_rows, \\"psnr\\"), \\"bad_holdout_n\\": len(bad_rows),\\n        \\"train_fit_psnr\\": float(np.mean(train_fit)) if train_fit else float(\\"nan\\"),\\n        \\"bad_fit_psnr\\": float(np.mean(bad_fit)) if bad_fit else float(\\"nan\\"),\\n        \\"bad_fit_n\\": len(bad_fit),\\n        \\"frames_below_18db\\": int(sum(v < 18 for v in fit.values())),\\n        \\"splats\\": info.get(\\"splats\\"),\\n        \\"minutes\\": info.get(\\"train_seconds\\", 0) / 60,\\n    }\\n\\n\\ndef markdown(rows: list[dict], n_holdout: int, n_bad: int) -> str:\\n    base = rows[0]\\n    lines = [\\n        f\\"## Poz onarımı: aynı {n_holdout} saklanan fotoğraf (isimle eşlenir), app eğiticisi + lr_decay, 30k\\",\\n        \\"\\",\\n        \\"| SfM | kayıtlı | yeniden proj. hata (px) | iz uzunluğu | PSNR ↑ | Δ | LPIPS ↓ | PSNR (poz hatası hariç) | \\"\\n        f\\"kötü bölüm eğitim-karesi PSNR ({n_bad} kare) | <18 dB kare | splat |\\",\\n        \\"|---|---|---|---|---|---|---|---|---|---|---|\\",\\n    ]\\n    for r in rows:\\n        s = r[\\"sfm\\"] or {}\\n        reg = f\\"{s.get(\'registered\', \'?\')}\\" + (f\\" ({s[\'bad_registered\']}/{s[\'bad_total\']} kötü)\\" if s else \\"\\")\\n        lines.append(\\n            f\\"| {r[\'name\']} | {reg} | {s.get(\'reproj_error_px\', float(\'nan\')):.3f} | {s.get(\'track_length\', float(\'nan\')):.2f} | \\"\\n            f\\"{r[\'psnr\']:.2f} | {r[\'psnr\'] - base[\'psnr\']:+.2f} | {r[\'lpips\']:.4f} | {r[\'psnr_posefree\']:.2f} | \\"\\n            f\\"{r[\'bad_fit_psnr\']:.2f} ({r[\'bad_fit_n\']}) | {r[\'frames_below_18db\']} | {r[\'splats\']:,} |\\"\\n        )\\n    missing = [f\\"{r[\'name\']}: {n_holdout - r[\'holdout_n\']}\\" for r in rows if r[\\"holdout_n\\"] < n_holdout]\\n    lines += [\\n        \\"\\",\\n        \\"* **kötü bölüm eğitim-karesi PSNR**: mevcut SfM\'de splat\'in kendi eğitim karelerini bile tutturamadığı \\"\\n        \\"bölümler (sabit isim listesi). Onarım işe yaradıysa bu sütun belirgin yükselir.\\",\\n        \\"* Her SfM kendi kameralarını kullanır; saklanan fotoğraflar isimle aynıdır ama her SfM\'in lens düzeltmesiyle açılır.\\",\\n        \\"* Yeniden projeksiyon hatası tek başına kalite ölçüsü değildir; daha çok/zor kareyi kaydeden SfM\'in hatası biraz artabilir.\\",\\n    ]\\n    if missing:\\n        lines.append(\\"* Kayıt edilemeyen saklanan fotoğraflar (ortalamaya girmez): \\" + \\", \\".join(missing))\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef strip_plot(fits: dict[str, dict[str, float]], order: list[str], bad_names: list[str], out: Path) -> Path:\\n    \\"\\"\\"One row per reconstruction: per-frame PSNR along the video (red = fit failure).\\"\\"\\"\\n    import matplotlib\\n\\n    matplotlib.use(\\"Agg\\")\\n    import matplotlib.pyplot as plt\\n\\n    fig, axes = plt.subplots(len(fits), 1, figsize=(16, 1.4 * len(fits) + 0.6), sharex=True, squeeze=False)\\n    x = np.arange(len(order))\\n    for ax, (name, fit) in zip(axes[:, 0], fits.items()):\\n        y = np.array([fit.get(n, np.nan) for n in order])\\n        ax.scatter(x, y, s=3, c=np.where(y < 18, \\"tab:red\\", \\"tab:blue\\"))\\n        ax.axhline(18, color=\\"0.6\\", lw=0.6)\\n        missing = np.isnan(y)\\n        if missing.any():\\n            ax.scatter(x[missing], np.full(missing.sum(), 10.5), s=3, c=\\"0.4\\", marker=\\"|\\")\\n        ax.set_ylim(10, 34)\\n        ax.set_ylabel(name, rotation=0, ha=\\"right\\", va=\\"center\\", fontsize=9)\\n        for n in bad_names:\\n            if n in order:\\n                ax.axvspan(order.index(n) - 0.5, order.index(n) + 0.5, color=\\"orange\\", alpha=0.15, lw=0)\\n    axes[-1, 0].set_xlabel(\\"video sırası (gri çizgi: o SfM\'de kayıtsız kare; turuncu: mevcut SfM\'in kötü bölümleri)\\")\\n    fig.suptitle(\\"Kare başına render PSNR (eğitim + saklanan)\\")\\n    fig.tight_layout()\\n    fig.savefig(out, dpi=110)\\n    plt.close(fig)\\n    return out\\n\\n\\ndef focus_sheets(render_root: Path, variants: list[str], photos: dict[str, np.ndarray], out_dir: Path) -> list[Path]:\\n    \\"\\"\\"photo | each reconstruction\'s render, for the fixed focus frames (by name).\\"\\"\\"\\n    paths = []\\n    for name, photo in photos.items():\\n        stem = Path(name).stem\\n        tiles = {\\"foto\\": photo}\\n        for v in variants:\\n            p = render_root / v / \\"focus\\" / f\\"{stem}.jpg\\"\\n            if p.exists():\\n                img = np.asarray(Image.open(p).convert(\\"RGB\\"))\\n                if img.shape == photo.shape:\\n                    tiles[v] = img\\n                else:  # different undistortion crop -> resize for display only\\n                    tiles[v] = np.asarray(Image.fromarray(img).resize((photo.shape[1], photo.shape[0]), Image.LANCZOS))\\n        if len(tiles) > 1:\\n            out = out_dir / f\\"focus_{stem}.jpg\\"\\n            comparison_image(tiles, out, tile_width=480, box_from=next(k for k in tiles if k != \\"foto\\"))\\n            paths.append(out)\\n    return paths\\n\\n\\ndef save_json(path: Path, obj) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    tmp = path.with_suffix(\\".tmp\\")\\n    tmp.write_text(json.dumps(obj, indent=1), encoding=\\"utf-8\\")\\n    tmp.replace(path)\\n", "lib/splat_restorer/restorer_model.py": "\\"\\"\\"Single-step diffusion restorer (Difix3D architecture).\\n\\nAdapted from NVIDIA Difix3D (https://github.com/nv-tlabs/Difix3D, src/model.py\\nand src/mv_unet.py) under the NVIDIA License in LICENSE-DIFIX3D.txt, which\\nallows non-commercial (research / evaluation) use only.\\n\\nChanges from the original:\\n* works with current diffusers / peft (no pinned 0.25 UNet copy): the\\n  reference-view attention is an attention *processor*, not a forked UNet;\\n* the text prompt is fixed and encoded once;\\n* x0 is computed directly from the scheduler\'s alphas (epsilon or v);\\n* checkpoints contain the full UNet so a saved model is self-contained.\\n\\nHow it works: the degraded image is VAE-encoded and treated as a latent at\\ntimestep ``t`` (default 199). The SD-Turbo UNet predicts the \\"noise\\", x0 is\\nrecovered in one step and decoded. VAE encoder activations are fed to the\\ndecoder through 1x1 skip convs so fine detail survives the round trip. With a\\nreference view, self-attention runs jointly over both views\' tokens.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport torch\\nimport torch.nn.functional as F\\n\\nBASE_MODEL = \\"stabilityai/sd-turbo\\"\\nPROMPT = \\"remove degradation\\"\\nVAE_LORA_TARGETS = (\\n    \\"conv1\\", \\"conv2\\", \\"conv_in\\", \\"conv_shortcut\\", \\"conv\\", \\"conv_out\\",\\n    \\"skip_conv_1\\", \\"skip_conv_2\\", \\"skip_conv_3\\", \\"skip_conv_4\\",\\n    \\"to_k\\", \\"to_q\\", \\"to_v\\", \\"to_out.0\\",\\n)\\n\\n\\ndef _vae_encoder_fwd(self, sample):\\n    sample = self.conv_in(sample)\\n    skips = []\\n    for down_block in self.down_blocks:\\n        skips.append(sample)\\n        sample = down_block(sample)\\n    sample = self.mid_block(sample)\\n    sample = self.conv_norm_out(sample)\\n    sample = self.conv_act(sample)\\n    sample = self.conv_out(sample)\\n    self.current_down_blocks = skips\\n    return sample\\n\\n\\ndef _vae_decoder_fwd(self, sample, latent_embeds=None):\\n    sample = self.conv_in(sample)\\n    upscale_dtype = next(iter(self.up_blocks.parameters())).dtype\\n    sample = self.mid_block(sample, latent_embeds)\\n    sample = sample.to(upscale_dtype)\\n    skip_convs = [self.skip_conv_1, self.skip_conv_2, self.skip_conv_3, self.skip_conv_4]\\n    skips = self.incoming_skip_acts[::-1]\\n    for idx, up_block in enumerate(self.up_blocks):\\n        if not self.ignore_skip:\\n            sample = sample + skip_convs[idx](skips[idx] * self.gamma)\\n        sample = up_block(sample, latent_embeds)\\n    if latent_embeds is None:\\n        sample = self.conv_norm_out(sample)\\n    else:\\n        sample = self.conv_norm_out(sample, latent_embeds)\\n    sample = self.conv_act(sample)\\n    return self.conv_out(sample)\\n\\n\\nclass MultiViewSelfAttnProcessor:\\n    \\"\\"\\"Self-attention over the tokens of all views of a sample.\\n\\n    The UNet sees a batch of (B * V) images; for attention the V views of each\\n    sample are concatenated along the token axis, which lets the degraded view\\n    borrow detail from the reference view.\\n    \\"\\"\\"\\n\\n    def __init__(self):\\n        self.num_views = 1\\n\\n    def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None, *args, **kwargs):\\n        input_ndim = hidden_states.ndim\\n        if input_ndim == 4:\\n            b0, c, h, w = hidden_states.shape\\n            hidden_states = hidden_states.view(b0, c, h * w).transpose(1, 2)\\n        bv, n, d = hidden_states.shape\\n        v = self.num_views\\n        x = hidden_states.reshape(bv // v, v * n, d) if v > 1 else hidden_states\\n        b = x.shape[0]\\n        q, k, val = attn.to_q(x), attn.to_k(x), attn.to_v(x)\\n        heads = attn.heads\\n        hd = q.shape[-1] // heads\\n        q, k, val = (t.view(b, -1, heads, hd).transpose(1, 2) for t in (q, k, val))\\n        out = F.scaled_dot_product_attention(q, k, val)\\n        out = out.transpose(1, 2).reshape(b, -1, heads * hd).to(q.dtype)\\n        out = attn.to_out[1](attn.to_out[0](out))\\n        if v > 1:\\n            out = out.reshape(bv, n, d)\\n        if input_ndim == 4:\\n            out = out.transpose(-1, -2).reshape(b0, c, h, w)\\n        if getattr(attn, \\"residual_connection\\", False):\\n            out = out + hidden_states\\n        return out / getattr(attn, \\"rescale_output_factor\\", 1.0)\\n\\n\\nclass Restorer(torch.nn.Module):\\n    def __init__(\\n        self,\\n        timestep: int = 199,\\n        lora_rank_vae: int = 4,\\n        device: str = \\"cuda\\",\\n        pretrained: bool = True,\\n        unet_overrides: dict | None = None,\\n        base_model: str = BASE_MODEL,\\n        pretrained_text_encoder: bool | None = None,\\n    ):\\n        \\"\\"\\"``pretrained=False`` builds random weights from the configs only, and\\n        ``unet_overrides`` can shrink that random UNet (tests on small GPUs).\\n        ``base_model`` is the Hugging Face repo the parts are loaded from.\\n        ``pretrained_text_encoder`` defaults to ``pretrained``.\\"\\"\\"\\n        super().__init__()\\n        from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel\\n        from diffusers.models.attention_processor import AttnProcessor2_0\\n        from peft import LoraConfig, inject_adapter_in_model\\n        from transformers import AutoTokenizer, CLIPTextConfig, CLIPTextModel\\n\\n        def load(cls, sub, overrides=None):\\n            if pretrained:\\n                return cls.from_pretrained(base_model, subfolder=sub)\\n            config = dict(cls.load_config(base_model, subfolder=sub))\\n            config.update(overrides or {})\\n            return cls.from_config(config)\\n\\n        tokenizer = AutoTokenizer.from_pretrained(base_model, subfolder=\\"tokenizer\\")\\n        if pretrained if pretrained_text_encoder is None else pretrained_text_encoder:\\n            text_encoder = CLIPTextModel.from_pretrained(base_model, subfolder=\\"text_encoder\\")\\n        else:\\n            text_encoder = CLIPTextModel(CLIPTextConfig.from_pretrained(base_model, subfolder=\\"text_encoder\\"))\\n        text_encoder = text_encoder.to(device)\\n        with torch.no_grad():\\n            ids = tokenizer(PROMPT, max_length=tokenizer.model_max_length, padding=\\"max_length\\",\\n                            truncation=True, return_tensors=\\"pt\\").input_ids.to(device)\\n            self.register_buffer(\\"prompt_embeds\\", text_encoder(ids)[0].float(), persistent=False)\\n        del text_encoder\\n\\n        sched = DDPMScheduler.from_pretrained(base_model, subfolder=\\"scheduler\\")\\n        self.prediction_type = sched.config.prediction_type\\n        self.register_buffer(\\"alphas_cumprod\\", sched.alphas_cumprod.float(), persistent=False)\\n        self.timestep = int(timestep)\\n\\n        vae = load(AutoencoderKL, \\"vae\\")\\n        vae.encoder.forward = _vae_encoder_fwd.__get__(vae.encoder, vae.encoder.__class__)\\n        vae.decoder.forward = _vae_decoder_fwd.__get__(vae.decoder, vae.decoder.__class__)\\n        dec = vae.decoder\\n        dec.skip_conv_1 = torch.nn.Conv2d(512, 512, 1, bias=False)\\n        dec.skip_conv_2 = torch.nn.Conv2d(256, 512, 1, bias=False)\\n        dec.skip_conv_3 = torch.nn.Conv2d(128, 512, 1, bias=False)\\n        dec.skip_conv_4 = torch.nn.Conv2d(128, 256, 1, bias=False)\\n        for conv in (dec.skip_conv_1, dec.skip_conv_2, dec.skip_conv_3, dec.skip_conv_4):\\n            torch.nn.init.constant_(conv.weight, 1e-5)\\n        dec.ignore_skip = False\\n        dec.gamma = 1.0\\n        targets = [name for name, _ in dec.named_modules() if name and any(name.endswith(t) for t in VAE_LORA_TARGETS)]\\n        inject_adapter_in_model(\\n            LoraConfig(r=lora_rank_vae, init_lora_weights=\\"gaussian\\", target_modules=targets), dec, adapter_name=\\"vae_skip\\"\\n        )\\n        vae.requires_grad_(False)\\n        self.vae = vae\\n\\n        unet = load(UNet2DConditionModel, \\"unet\\", unet_overrides)\\n        self.mv_processor = MultiViewSelfAttnProcessor()\\n        unet.set_attn_processor(\\n            {name: (self.mv_processor if name.endswith(\\"attn1.processor\\") else AttnProcessor2_0()) for name in unet.attn_processors}\\n        )\\n        self.unet = unet\\n        self.to(device)\\n\\n    @classmethod\\n    def from_difix(cls, repo: str = \\"nvidia/difix\\", device: str = \\"cuda\\") -> \\"Restorer\\":\\n        \\"\\"\\"NVIDIA\'s released Difix weights (\\"nvidia/difix\\" or \\"nvidia/difix_ref\\").\\n\\n        Their VAE (skip convs + \\"vae_skip\\" LoRA) and UNet have exactly this\\n        class\'s parameter names and shapes, so they load strictly.\\n        \\"\\"\\"\\n        from huggingface_hub import hf_hub_download\\n        from safetensors.torch import load_file\\n\\n        # Build VAE/UNet from the configs only: Difix\'s VAE file stores LoRA-wrapped\\n        # layers (\\".base_layer.\\"), which a plain AutoencoderKL.from_pretrained rejects.\\n        model = cls(device=device, base_model=repo, pretrained=False, pretrained_text_encoder=True)\\n        for part, module in ((\\"vae\\", model.vae), (\\"unet\\", model.unet)):\\n            path = hf_hub_download(repo, f\\"{part}/diffusion_pytorch_model.safetensors\\")\\n            module.load_state_dict(load_file(path), strict=True)\\n        model.set_eval()\\n        return model\\n\\n    # ----- trainable parameter groups -------------------------------------------------\\n    def vae_trainable_parameters(self):\\n        dec = self.vae.decoder\\n        params = [p for n, p in dec.named_parameters() if \\"lora_\\" in n]\\n        for conv in (dec.skip_conv_1, dec.skip_conv_2, dec.skip_conv_3, dec.skip_conv_4):\\n            params += [p for n, p in conv.named_parameters() if \\"lora_\\" not in n]\\n        return params\\n\\n    def set_train(self):\\n        self.unet.train()\\n        self.vae.train()\\n        self.unet.requires_grad_(True)\\n        self.vae.requires_grad_(False)\\n        for p in self.vae_trainable_parameters():\\n            p.requires_grad_(True)\\n\\n    def set_eval(self):\\n        self.unet.eval()\\n        self.vae.eval()\\n        self.requires_grad_(False)\\n\\n    # ----- forward ---------------------------------------------------------------------\\n    def forward(self, x: torch.Tensor, sample_latent: bool = True) -> torch.Tensor:\\n        \\"\\"\\"x: (B, V, 3, H, W) in [-1, 1]; view 0 is the image to restore,\\n        view 1 (optional) the reference. Returns (B, V, 3, H, W) in [-1, 1].\\"\\"\\"\\n        b, v = x.shape[:2]\\n        flat = x.flatten(0, 1)\\n        dist = self.vae.encode(flat).latent_dist\\n        z = (dist.sample() if sample_latent else dist.mode()) * self.vae.config.scaling_factor\\n        self.mv_processor.num_views = v\\n        t = torch.full((z.shape[0],), self.timestep, device=z.device, dtype=torch.long)\\n        cond = self.prompt_embeds.expand(z.shape[0], -1, -1).to(z.dtype)\\n        pred = self.unet(z, t, encoder_hidden_states=cond).sample\\n        a = self.alphas_cumprod[self.timestep].to(z.dtype)\\n        if self.prediction_type == \\"epsilon\\":\\n            z0 = (z - (1 - a).sqrt() * pred) / a.sqrt()\\n        elif self.prediction_type == \\"v_prediction\\":\\n            z0 = a.sqrt() * z - (1 - a).sqrt() * pred\\n        else:  # \\"sample\\"\\n            z0 = pred\\n        self.vae.decoder.incoming_skip_acts = self.vae.encoder.current_down_blocks\\n        out = self.vae.decode(z0 / self.vae.config.scaling_factor).sample.clamp(-1, 1)\\n        return out.view(b, v, *out.shape[1:])\\n\\n    @torch.no_grad()\\n    def restore(self, deg: torch.Tensor, ref: torch.Tensor | None = None) -> torch.Tensor:\\n        \\"\\"\\"deg/ref: (B, 3, H, W) in [0, 1], H and W multiples of 8. Returns [0, 1].\\"\\"\\"\\n        x = deg * 2 - 1\\n        if ref is not None:\\n            x = torch.stack([x, ref * 2 - 1], dim=1)\\n        else:\\n            x = x[:, None]\\n        # bf16 where the GPU supports it (A100/H100/L4); fp32 otherwise (T4), since\\n        # the SD VAE can overflow in fp16.\\n        # (is_bf16_supported() is also True on T4, where bf16 is only emulated and slow.)\\n        bf16 = x.is_cuda and torch.cuda.get_device_capability(x.device)[0] >= 8\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16, enabled=bf16):\\n            out = self.forward(x, sample_latent=False)[:, 0]\\n        return (out.float() * 0.5 + 0.5).clamp(0, 1)\\n\\n    # ----- persistence -----------------------------------------------------------------\\n    def trainable_state_dict(self) -> dict:\\n        vae_keys = {n for n, _ in self.vae.named_parameters() if \\"lora_\\" in n or \\"skip_conv\\" in n}\\n        return {\\n            \\"unet\\": self.unet.state_dict(),\\n            \\"vae\\": {k: v for k, v in self.vae.state_dict().items() if k in vae_keys},\\n            \\"timestep\\": self.timestep,\\n        }\\n\\n    def load_trainable_state_dict(self, sd: dict) -> None:\\n        self.unet.load_state_dict(sd[\\"unet\\"])\\n        missing = self.vae.load_state_dict(sd[\\"vae\\"], strict=False)\\n        bad = [k for k in missing.unexpected_keys]\\n        if bad:\\n            raise KeyError(f\\"Unexpected VAE keys in checkpoint: {bad[:5]}\\")\\n        self.timestep = int(sd.get(\\"timestep\\", self.timestep))\\n", "lib/splat_restorer/scene.py": "\\"\\"\\"Load a COLMAP scene (DL3DV gaussian_splat layout) into GPU-ready arrays.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom PIL import Image as PILImage\\n\\nfrom .colmap_io import read_sparse_model\\n\\n\\n@dataclass\\nclass Scene:\\n    name: str\\n    image_paths: list[Path]  # sorted by file name (= capture order for DL3DV)\\n    width: int\\n    height: int\\n    Ks: np.ndarray  # (F, 3, 3) at the loaded image resolution\\n    w2c: np.ndarray  # (F, 4, 4) world -> camera, normalised world\\n    points_xyz: np.ndarray  # (N, 3) normalised world\\n    points_rgb: np.ndarray  # (N, 3) float in [0, 1]\\n    scene_scale: float  # radius of the camera cloud after normalisation\\n\\n    @property\\n    def num_frames(self) -> int:\\n        return len(self.image_paths)\\n\\n    def camera_centers(self) -> np.ndarray:\\n        r = self.w2c[:, :3, :3]\\n        t = self.w2c[:, :3, 3]\\n        return -np.einsum(\\"fij,fi->fj\\", r, t)\\n\\n    def view_dirs(self) -> np.ndarray:\\n        # Camera +z (forward) in world coordinates = third row of R.\\n        return self.w2c[:, 2, :3]\\n\\n    def load_image(self, i: int) -> np.ndarray:\\n        \\"\\"\\"Return frame i as float32 HxWx3 in [0, 1] at the scene resolution.\\"\\"\\"\\n        img = PILImage.open(self.image_paths[i]).convert(\\"RGB\\")\\n        if img.size != (self.width, self.height):\\n            img = img.resize((self.width, self.height), PILImage.BICUBIC)\\n        return np.asarray(img, dtype=np.float32) / 255.0\\n\\n\\ndef _normalise(c2w_centers: np.ndarray) -> tuple[np.ndarray, float]:\\n    center = c2w_centers.mean(axis=0)\\n    radius = float(np.linalg.norm(c2w_centers - center, axis=1).max())\\n    scale = 1.0 / max(radius, 1e-8)\\n    return center, scale\\n\\n\\ndef load_colmap_scene(\\n    scene_dir: Path,\\n    images_subdir: str = \\"images_4\\",\\n    sparse_subdir: str = \\"sparse/0\\",\\n    max_points: int = 300_000,\\n    name: str | None = None,\\n) -> Scene:\\n    \\"\\"\\"Load a scene whose undistorted images live in ``scene_dir/images_subdir``.\\n\\n    COLMAP intrinsics are rescaled to the actual image size, so downsampled\\n    image folders (images_2/4/8) work with the full-resolution model.\\n    \\"\\"\\"\\n    scene_dir = Path(scene_dir)\\n    model = read_sparse_model(scene_dir / sparse_subdir)\\n    img_dir = scene_dir / images_subdir\\n    if not img_dir.is_dir():\\n        raise FileNotFoundError(f\\"Missing image folder {img_dir}\\")\\n\\n    by_name = {Path(im.name).name: im for im in model.images.values()}\\n    files = sorted(p for p in img_dir.iterdir() if p.suffix.lower() in (\\".png\\", \\".jpg\\", \\".jpeg\\"))\\n    files = [p for p in files if p.name in by_name]\\n    if len(files) < 10:\\n        raise ValueError(f\\"{scene_dir}: only {len(files)} registered images found in {images_subdir}\\")\\n\\n    with PILImage.open(files[0]) as probe:\\n        width, height = probe.size\\n\\n    Ks, w2cs = [], []\\n    for p in files:\\n        im = by_name[p.name]\\n        cam = model.cameras[im.camera_id]\\n        if cam.has_distortion():\\n            raise ValueError(f\\"{scene_dir}: camera {cam.id} is distorted ({cam.model}); use undistorted images\\")\\n        fx, fy, cx, cy = cam.intrinsics()\\n        sx, sy = width / cam.width, height / cam.height\\n        Ks.append(np.array([[fx * sx, 0, cx * sx], [0, fy * sy, cy * sy], [0, 0, 1]], np.float64))\\n        w2cs.append(im.world_to_camera())\\n    Ks = np.stack(Ks)\\n    w2c = np.stack(w2cs)\\n\\n    # Normalise the world so cameras sit inside the unit sphere.\\n    centers = -np.einsum(\\"fij,fi->fj\\", w2c[:, :3, :3], w2c[:, :3, 3])\\n    center, scale = _normalise(centers)\\n    c2w = np.linalg.inv(w2c)\\n    c2w[:, :3, 3] = (c2w[:, :3, 3] - center) * scale\\n    w2c = np.linalg.inv(c2w)\\n    xyz = (model.points_xyz - center) * scale\\n    rgb = model.points_rgb.astype(np.float32) / 255.0\\n\\n    # Drop far outliers and subsample very dense clouds.\\n    keep = np.linalg.norm(xyz, axis=1) < 20.0\\n    xyz, rgb = xyz[keep], rgb[keep]\\n    if len(xyz) > max_points:\\n        idx = np.random.default_rng(0).choice(len(xyz), max_points, replace=False)\\n        xyz, rgb = xyz[idx], rgb[idx]\\n    if len(xyz) < 100:\\n        raise ValueError(f\\"{scene_dir}: sparse cloud too small ({len(xyz)} points)\\")\\n\\n    return Scene(\\n        name=name or scene_dir.name,\\n        image_paths=files,\\n        width=width,\\n        height=height,\\n        Ks=Ks.astype(np.float32),\\n        w2c=w2c.astype(np.float32),\\n        points_xyz=xyz.astype(np.float32),\\n        points_rgb=rgb,\\n        scene_scale=1.0,\\n    )\\n", "lib/splat_restorer/spirula_quality.py": "\\"\\"\\"Spirula Studio training variants, scored on the same held-out photos as our other trainers.\\n\\nSpirula (v2026.9.24) holds out every 10th image of the name-sorted set with\\n``eval_mode=interval, eval_interval=10``, i.e. the same 93 frames our own\\ntrainers keep out. Its PLY stays in the input COLMAP frame, so it is rendered\\nwith our cameras through ``own_splat`` exactly like every other splat.\\n\\nTwo scores per frame:\\n\\n* ``raw``: the exported PLY as a viewer shows it (primary).\\n* ``cc``: after a per-image affine colour map fitted on one half of the image\\n  and scored on the other half (and swapped). Spirula trains per-image\\n  appearance (bilateral grid + PPISP) that is not exported into the PLY; this\\n  diagnostic shows how much of the raw gap is colour rather than geometry.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport math\\nfrom dataclasses import asdict, dataclass, field\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image\\n\\nfrom .gs_trainer import render, ssim\\nfrom .own_splat import SplatData, _to_u8, _w2c_t, comparison_image, load_ply\\n\\n# Frames where our in-house IMG_5966 splat cannot fit even its TRAINING photos\\n# (render PSNR < 18 dB, merged and padded by 2): misregistered close-ups. Fixed\\n# in advance from that splat\'s diagnostics, never from a candidate\'s scores.\\nPOSE_FAIL_SEGMENTS = [(485, 496), (518, 522), (557, 563), (702, 706), (777, 784), (789, 793)]\\n\\n\\n@dataclass\\nclass Variant:\\n    name: str\\n    quality: str = \\"medium\\"\\n    iterations: int = 30000\\n    cap: int = 1_000_000\\n    flags: dict = field(default_factory=dict)\\n    note: str = \\"\\"\\n    minutes_rtx6000: int = 15  # rough wall time on an RTX PRO 6000 (96 GB)\\n    min_gib: float = 14\\n\\n\\n# Same settings as the 28 Sep IMG_5966 Spirula run (medium, 4K, MoGe-2 normals and\\n# depth at 0.01) unless a variant says otherwise.\\nVARIANTS = {\\n    \\"medium_holdout\\": Variant(\\"medium_holdout\\", note=\\"28 Eylül ayarları + saklanan kareler\\"),\\n    \\"medium_no_appearance\\": Variant(\\n        \\"medium_no_appearance\\", flags={\\"use_bilateral_grid\\": \\"false\\", \\"use_ppisp\\": \\"false\\"},\\n        note=\\"kare başına renk telafisi kapalı (PLY renkleri dürüst)\\"),\\n    \\"medium_floater_mild\\": Variant(\\"medium_floater_mild\\", flags={\\"floater_suppression\\": \\"mild\\"},\\n                                   note=\\"floater bastırma: mild\\"),\\n    \\"medium_depth_x5\\": Variant(\\"medium_depth_x5\\", flags={\\"depth_supervision_weight\\": \\"0.05\\"},\\n                               note=\\"derinlik desteği 5x (0.01 -> 0.05)\\"),\\n    \\"high_native\\": Variant(\\"high_native\\", quality=\\"high\\", iterations=50000, cap=3_000_000,\\n                           note=\\"Spirula high: 3M / 50k\\", minutes_rtx6000=45, min_gib=22),\\n    \\"ultra\\": Variant(\\"ultra\\", quality=\\"ultra\\", iterations=80000, cap=10_000_000,\\n                     note=\\"Spirula ultra: 10M / 80k\\", minutes_rtx6000=150, min_gib=38),\\n}\\n\\n\\ndef train_args(binary: Path, dataset: Path, recon_dir: str, out_prefix: Path, v: Variant,\\n               device: str, depth_maps: bool = True) -> list[str]:\\n    \\"\\"\\"CLI for one variant; mirrors the app\'s Spirula notebook plus the held-out split.\\"\\"\\"\\n    args = [str(binary), \\"train\\", \\"3dgs\\", \\"--data\\", str(dataset), \\"--colmap-recon-dir\\", str(recon_dir),\\n            \\"--output-dir-prefix\\", str(out_prefix), \\"--output-dir-name\\", \\"training\\",\\n            \\"--quality\\", v.quality, \\"--num-iterations\\", str(v.iterations), \\"--max-steps\\", str(v.iterations),\\n            \\"--cap-max\\", str(v.cap), \\"--steps-per-save\\", str(v.iterations),\\n            \\"--save-full-checkpoint\\", \\"false\\", \\"--save-only-latest-checkpoint\\", \\"true\\",\\n            \\"--train-resolution-divisor\\", \\"1\\", \\"--sh-degree\\", \\"3\\", \\"--cache-images\\", \\"disk\\",\\n            \\"--disable-viewer\\", \\"true\\", \\"--keep-viewer-alive\\", \\"false\\",\\n            \\"--load-normals\\", \\"true\\", \\"--normal-supervision-weight\\", \\"0.01\\",\\n            \\"--load-depths\\", str(depth_maps).lower(), \\"--depth-supervision-weight\\", \\"0.01\\" if depth_maps else \\"0\\",\\n            \\"--eval-mode\\", \\"interval\\", \\"--eval-interval\\", \\"10\\", \\"--validation-fraction\\", \\"0\\",\\n            \\"--save-eval-images\\", \\"true\\",\\n            \\"--device\\", device, \\"--lang\\", \\"en\\"]\\n    for key, value in v.flags.items():\\n        flag = \\"--\\" + key.replace(\\"_\\", \\"-\\")\\n        if flag in args:  # a variant overrides a base value\\n            args[args.index(flag) + 1] = value\\n        else:\\n            args += [flag, value]\\n    return args\\n\\n\\ndef find_outputs(out_prefix: Path) -> dict:\\n    \\"\\"\\"splat.ply, metrics.json and eval images Spirula wrote under ``out_prefix``.\\"\\"\\"\\n    plys = sorted(out_prefix.rglob(\\"splat.ply\\"))\\n    metrics = sorted(out_prefix.rglob(\\"metrics.json\\"))\\n    configs = sorted(out_prefix.rglob(\\"config.json\\"))\\n    return {\\n        \\"ply\\": plys[-1] if plys else None,\\n        \\"metrics\\": metrics[-1] if metrics else None,\\n        \\"config\\": configs[-1] if configs else None,\\n        \\"eval_gt\\": sorted(out_prefix.rglob(\\"eval-gt-*.png\\")),\\n        \\"eval_render\\": sorted(out_prefix.rglob(\\"eval-render-*.png\\")),\\n    }\\n\\n\\ndef check_split(eval_gt: list[Path], data: SplatData, holdout: list[int], n_check: int = 5) -> list[float]:\\n    \\"\\"\\"PSNR between Spirula\'s held-out GT slots and our held-out photos (should be high if the sets match).\\"\\"\\"\\n    if len(eval_gt) != len(holdout):\\n        raise AssertionError(f\\"Spirula held out {len(eval_gt)} images, we hold out {len(holdout)}\\")\\n    out = []\\n    for k in np.linspace(0, len(holdout) - 1, n_check).round().astype(int):\\n        ours = np.asarray(Image.open(data.frames_dir / data.frames[holdout[k]]).convert(\\"L\\"), np.float32) / 255\\n        theirs = Image.open(eval_gt[k]).convert(\\"L\\").resize((data.width, data.height), Image.BILINEAR)\\n        theirs = np.asarray(theirs, np.float32) / 255\\n        out.append(float(10 * np.log10(1 / max(np.mean((ours - theirs) ** 2), 1e-12))))\\n    return out\\n\\n\\ndef native_metrics(path: Path | None) -> dict:\\n    \\"\\"\\"Spirula\'s own metrics.json, flattened to mean values (format may vary: keep numbers only).\\"\\"\\"\\n    if path is None or not Path(path).exists():\\n        return {}\\n    raw = json.loads(Path(path).read_text(encoding=\\"utf-8\\"))\\n    flat = {}\\n\\n    def walk(obj, prefix=\\"\\"):\\n        if isinstance(obj, dict):\\n            for k, v in obj.items():\\n                walk(v, f\\"{prefix}{k}.\\")\\n        elif isinstance(obj, (int, float)) and not isinstance(obj, bool):\\n            flat[prefix[:-1]] = float(obj)\\n        elif isinstance(obj, list) and obj and all(isinstance(x, (int, float)) for x in obj):\\n            flat[prefix[:-1] + \\"(mean)\\"] = float(np.mean(obj))\\n\\n    walk(raw)\\n    return flat\\n\\n\\n# ----------------------------------------------------------------------------- scoring\\n\\n\\ndef _psnr(a: torch.Tensor, b: torch.Tensor) -> float:\\n    return 10 * math.log10(1.0 / max(F.mse_loss(a, b).item(), 1e-12))\\n\\n\\ndef _affine_fit(src: torch.Tensor, dst: torch.Tensor) -> torch.Tensor:\\n    a = src.reshape(-1, 3).double()\\n    A = torch.cat([a, torch.ones_like(a[:, :1])], 1)\\n    return torch.linalg.lstsq(A, dst.reshape(-1, 3).double()).solution\\n\\n\\ndef _apply(img: torch.Tensor, M: torch.Tensor) -> torch.Tensor:\\n    a = img.reshape(-1, 3).double()\\n    return (torch.cat([a, torch.ones_like(a[:, :1])], 1) @ M).float().reshape(img.shape).clamp(0, 1)\\n\\n\\ndef colour_corrected(img: torch.Tensor, gt: torch.Tensor) -> torch.Tensor:\\n    \\"\\"\\"Half-image cross-fit: map for the left half is fitted on the right half and vice versa.\\"\\"\\"\\n    w = img.shape[1] // 2\\n    out = img.clone()\\n    out[:, :w] = _apply(img[:, :w], _affine_fit(img[:, w:], gt[:, w:]))\\n    out[:, w:] = _apply(img[:, w:], _affine_fit(img[:, :w], gt[:, :w]))\\n    return out\\n\\n\\ndef _scores(img, gt, net_lpips) -> dict:\\n    return {\\"psnr\\": _psnr(img, gt), \\"ssim\\": ssim(img, gt).item(),\\n            \\"lpips\\": net_lpips(img.permute(2, 0, 1)[None] * 2 - 1, gt.permute(2, 0, 1)[None] * 2 - 1).item()}\\n\\n\\n@torch.no_grad()\\ndef evaluate_ply(ply: Path, data: SplatData, holdout: list[int], net_lpips, render_dir: Path, device=\\"cuda\\") -> dict:\\n    params, deg = load_ply(Path(ply), device)\\n    w2c, K = _w2c_t(data, device)\\n    render_dir.mkdir(parents=True, exist_ok=True)\\n    rows = {}\\n    for i in holdout:\\n        img, _ = render(params, w2c[i], K, data.width, data.height, deg)\\n        gt = data.photo(i, device)\\n        rows[str(i)] = {\\"raw\\": _scores(img, gt, net_lpips), \\"cc\\": _scores(colour_corrected(img, gt), gt, net_lpips)}\\n        Image.fromarray(_to_u8(img)).save(render_dir / f\\"{i:06d}.jpg\\", quality=92)\\n    n = int(params[\\"means\\"].shape[0])\\n    del params\\n    torch.cuda.empty_cache()\\n    return {\\"frames\\": rows, \\"splats\\": n}\\n\\n\\ndef is_pose_fail(i: int) -> bool:\\n    return any(a <= i <= b for a, b in POSE_FAIL_SEGMENTS)\\n\\n\\ndef summarise(results: dict) -> dict:\\n    out = {}\\n    for name, r in results.items():\\n        frames = r[\\"frames\\"]\\n        keep = [k for k in frames if not is_pose_fail(int(k))]\\n        m = {\\"splats\\": r.get(\\"splats\\"), \\"minutes\\": r.get(\\"minutes\\"), \\"note\\": r.get(\\"note\\", \\"\\")}\\n        for kind in (\\"raw\\", \\"cc\\"):\\n            for metric in (\\"psnr\\", \\"ssim\\", \\"lpips\\"):\\n                m[f\\"{kind}_{metric}\\"] = float(np.mean([frames[k][kind][metric] for k in frames]))\\n                m[f\\"{kind}_{metric}_clean\\"] = float(np.mean([frames[k][kind][metric] for k in keep]))\\n        m[\\"native\\"] = r.get(\\"native\\", {})\\n        out[name] = m\\n    return out\\n\\n\\ndef markdown(summary: dict, n_frames: int, n_excluded: int, reference: str | None) -> str:\\n    base = summary.get(reference) if reference else None\\n    base = base or next(iter(summary.values()))\\n    lines = [\\n        f\\"## Spirula varyantları: {n_frames} saklanan kare (eğitimde hiç kullanılmadı)\\",\\n        \\"\\",\\n        \\"| varyant | PSNR (PLY) | Δ | LPIPS (PLY) | Δ | PSNR, poz hatası hariç | PSNR renk eşitlemeli | splat | dk | not |\\",\\n        \\"|---|---|---|---|---|---|---|---|---|---|\\",\\n    ]\\n    for name, m in summary.items():\\n        lines.append(\\n            f\\"| {name} | {m[\'raw_psnr\']:.2f} | {m[\'raw_psnr\'] - base[\'raw_psnr\']:+.2f} | {m[\'raw_lpips\']:.4f} | \\"\\n            f\\"{m[\'raw_lpips\'] - base[\'raw_lpips\']:+.4f} | {m[\'raw_psnr_clean\']:.2f} | {m[\'cc_psnr\']:.2f} | \\"\\n            f\\"{(m[\'splats\'] or 0):,} | {m[\'minutes\'] or 0:.0f} | {m[\'note\']} |\\")\\n    lines += [\\n        \\"\\",\\n        f\\"Δ: `{reference or next(iter(summary))}` satırına göre. **PLY** sütunları dışa aktarılan splat\'in görüntüleyicide \\"\\n        \\"görüneceği hali (asıl karar ölçütü).\\",\\n        f\\"**Poz hatası hariç:** kamera pozu bozuk {n_excluded} kare (sabit liste, adaylardan bağımsız) çıkarılmış ortalama.\\",\\n        \\"**Renk eşitlemeli:** her karede görüntünün bir yarısından öğrenilen renk dönüşümü diğer yarıda ölçülür. Spirula \\"\\n        \\"eğitimde kare başına renk telafisi (bilateral grid + PPISP) öğrenir ve bu PLY\'ye aktarılmaz; ham ile renk \\"\\n        \\"eşitlemeli arasındaki fark bunun etkisidir. Sadece teşhis içindir.\\",\\n    ]\\n    natives = {n: m[\\"native\\"] for n, m in summary.items() if m.get(\\"native\\")}\\n    if natives:\\n        lines += [\\"\\", \\"**Spirula\'nın kendi ölçümü** (4K, kendi render\'ı; ham sayılar):\\", \\"\\"]\\n        for n, nat in natives.items():\\n            keys = [k for k in nat if \\"psnr\\" in k.lower() or \\"ssim\\" in k.lower()][:6]\\n            lines.append(f\\"- `{n}`: \\" + \\", \\".join(f\\"{k} {nat[k]:.3f}\\" for k in keys))\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef comparison_sheets(results: dict, data: SplatData, out_dir: Path, render_root: Path, holdout: list[int],\\n                      n_worst: int = 3, n_even: int = 4) -> list[Path]:\\n    names = list(results)\\n    first = results[names[0]][\\"frames\\"]\\n    ids = [i for i in holdout if not is_pose_fail(i)]\\n    worst = sorted(ids, key=lambda i: first[str(i)][\\"raw\\"][\\"psnr\\"])[:n_worst]\\n    even = [ids[k] for k in np.linspace(0, len(ids) - 1, n_even + 2).round().astype(int)[1:-1]]\\n    paths = []\\n    for i in dict.fromkeys(worst + even):\\n        tiles = {\\"foto\\": _to_u8(data.photo(i, \\"cpu\\"))}\\n        for n in names:\\n            p = render_root / n / \\"renders\\" / f\\"{i:06d}.jpg\\"\\n            if p.exists():\\n                tiles[n] = np.asarray(Image.open(p).convert(\\"RGB\\"))\\n        p = out_dir / f\\"compare_{i:06d}.jpg\\"\\n        comparison_image(tiles, p, tile_width=480, box_from=names[0])\\n        paths.append(p)\\n    return paths\\n\\n\\ndef save_json(path: Path, obj) -> None:\\n    tmp = path.with_suffix(\\".tmp\\")\\n    tmp.write_text(json.dumps(obj, indent=1), encoding=\\"utf-8\\")\\n    tmp.replace(path)\\n\\n\\ndef variant_dict(v: Variant) -> dict:\\n    return asdict(v)\\n", "lib/splat_restorer/splits.py": "\\"\\"\\"Frame splits for degraded reconstructions.\\n\\nEvery scene reserves a fixed set of *eval* frames (every ``eval_every``-th\\nframe) that no recipe ever trains on, so the distill-back test can measure the\\nsplat on frames nobody has seen. Each recipe then picks its own sparse set of\\n*train* frames; everything else becomes a *pair* frame (render vs. photo).\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\n\\n\\n@dataclass(frozen=True)\\nclass Recipe:\\n    name: str\\n    train_stride: int  # keep every N-th non-eval frame for training\\n    steps: int  # optimisation steps for the degraded splat\\n\\n\\n# Two artefact regimes: very sparse (floaters, holes, smeared geometry) and\\n# moderately sparse (blur and view-dependent errors between views).\\nRECIPES: dict[str, Recipe] = {\\n    \\"sparse_hard\\": Recipe(\\"sparse_hard\\", train_stride=12, steps=4000),\\n    \\"sparse_mild\\": Recipe(\\"sparse_mild\\", train_stride=4, steps=4000),\\n}\\n\\n\\n@dataclass\\nclass FrameSplit:\\n    train: list[int]\\n    pair: list[int]\\n    eval: list[int]\\n\\n\\ndef eval_frames(num_frames: int, eval_every: int = 8) -> list[int]:\\n    return list(range(eval_every // 2, num_frames, eval_every))\\n\\n\\ndef make_split(num_frames: int, recipe: Recipe, eval_every: int = 8) -> FrameSplit:\\n    ev = eval_frames(num_frames, eval_every)\\n    ev_set = set(ev)\\n    candidates = [i for i in range(num_frames) if i not in ev_set]\\n    train = candidates[:: recipe.train_stride]\\n    # Always include the last candidate so the trajectory end is covered.\\n    if candidates and candidates[-1] not in train:\\n        train.append(candidates[-1])\\n    train_set = set(train)\\n    pair = [i for i in candidates if i not in train_set]\\n    return FrameSplit(train=sorted(train), pair=pair, eval=ev)\\n\\n\\ndef subsample(indices: list[int], max_count: int) -> list[int]:\\n    \\"\\"\\"Evenly spaced subset (deterministic) of at most max_count indices.\\"\\"\\"\\n    if len(indices) <= max_count:\\n        return list(indices)\\n    pos = np.linspace(0, len(indices) - 1, max_count).round().astype(int)\\n    return [indices[p] for p in sorted(set(pos.tolist()))]\\n\\n\\ndef nearest_reference(\\n    query: int,\\n    candidates: list[int],\\n    centers: np.ndarray,\\n    forwards: np.ndarray,\\n    angle_weight: float = 0.5,\\n) -> int:\\n    \\"\\"\\"Training frame whose camera is closest in position and viewing direction.\\"\\"\\"\\n    c = np.asarray(candidates)\\n    d_pos = np.linalg.norm(centers[c] - centers[query], axis=1)\\n    d_pos = d_pos / (d_pos.max() + 1e-8)\\n    cos = np.clip(forwards[c] @ forwards[query], -1.0, 1.0)\\n    d_ang = (1.0 - cos) / 2.0\\n    return int(c[np.argmin(d_pos + angle_weight * d_ang)])\\n", "lib/splat_restorer/stages.py": "\\"\\"\\"Notebook-facing stages. Every stage is resumable: re-running skips work\\nwhose durable output already exists on Drive.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport shutil\\nimport threading\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nfrom . import dl3dv\\nfrom .pairs import extract_pairs, generate_scene_pairs\\nfrom .workspace import Workspace, atomic_write_json\\n\\n\\ndef log(msg: str) -> None:\\n    print(time.strftime(\\"%H:%M:%S\\"), msg, flush=True)\\n\\n\\n# ----------------------------------------------------------------------------- preflight\\ndef preflight(ws: Workspace) -> dict:\\n    import torch\\n\\n    info = {}\\n    assert torch.cuda.is_available(), \\"No CUDA GPU. Runtime -> Change runtime type -> GPU.\\"\\n    p = torch.cuda.get_device_properties(0)\\n    info[\\"gpu\\"] = f\\"{p.name}, {p.total_memory / 2**30:.0f} GB, sm_{p.major}{p.minor}\\"\\n    info[\\"torch\\"] = f\\"{torch.__version__} (CUDA {torch.version.cuda})\\"\\n    info[\\"bf16\\"] = torch.cuda.is_bf16_supported()\\n\\n    import gsplat\\n\\n    from .gs_trainer import render\\n\\n    # Tiny render: forces the gsplat CUDA build now rather than mid-run.\\n    params = {\\n        \\"means\\": torch.randn(64, 3, device=\\"cuda\\") * 0.1 + torch.tensor([0, 0, 2.0], device=\\"cuda\\"),\\n        \\"scales\\": torch.full((64, 3), -3.0, device=\\"cuda\\"),\\n        \\"quats\\": torch.tensor([[1.0, 0, 0, 0]], device=\\"cuda\\").repeat(64, 1),\\n        \\"opacities\\": torch.zeros(64, device=\\"cuda\\"),\\n        \\"sh0\\": torch.rand(64, 1, 3, device=\\"cuda\\"),\\n        \\"shN\\": torch.zeros(64, 15, 3, device=\\"cuda\\"),\\n    }\\n    K = torch.tensor([[100.0, 0, 64], [0, 100.0, 64], [0, 0, 1]], device=\\"cuda\\")\\n    img, _ = render(params, torch.eye(4, device=\\"cuda\\"), K, 128, 128, 3)\\n    assert img.shape == (128, 128, 3) and torch.isfinite(img).all()\\n    info[\\"gsplat\\"] = f\\"{gsplat.__version__} (render OK)\\"\\n\\n    import diffusers, peft, transformers\\n\\n    info[\\"diffusers\\"] = diffusers.__version__\\n    info[\\"peft\\"] = peft.__version__\\n    info[\\"transformers\\"] = transformers.__version__\\n\\n    from huggingface_hub import HfApi, auth_check\\n\\n    info[\\"hf_user\\"] = HfApi().whoami()[\\"name\\"]\\n    auth_check(dl3dv.REPO, repo_type=\\"dataset\\")\\n    info[\\"dl3dv_access\\"] = \\"OK\\"\\n\\n    info[\\"local_free_gb\\"] = round(shutil.disk_usage(ws.local).free / 2**30, 1)\\n    info[\\"drive_free_gb\\"] = round(shutil.disk_usage(ws.root).free / 2**30, 1)\\n    for k, v in info.items():\\n        log(f\\"  {k:14s} {v}\\")\\n    if info[\\"drive_free_gb\\"] < 30:\\n        log(\\"  WARNING: under 30 GB free on Drive; pairs + checkpoints peak around 25-30 GB.\\")\\n    return info\\n\\n\\n# ----------------------------------------------------------------------------- split\\ndef prepare_split(ws: Workspace, n_test: int, seed: int = 1234) -> dict:\\n    if ws.split_path.exists():\\n        split = json.loads(ws.split_path.read_text(encoding=\\"utf-8\\"))\\n        log(f\\"split: reusing {ws.split_path} ({len(split[\'test\'])} test, {len(split[\'train_pool\'])} train pool)\\")\\n        return split\\n    scenes = dl3dv.list_scenes()\\n    labels = dl3dv.scene_labels()\\n    split = dl3dv.make_split(scenes, n_test=n_test, seed=seed, labels=labels)\\n    atomic_write_json(ws.split_path, split)\\n    log(f\\"split: {len(scenes)} scenes -> {len(split[\'test\'])} test, {len(split[\'train_pool\'])} train pool\\")\\n    return split\\n\\n\\ndef done_scenes(ws: Workspace, scenes: list[str]) -> list[str]:\\n    return [s for s in scenes if ws.pair_tar(s).exists()]\\n\\n\\ndef train_scenes(ws: Workspace, split: dict, n: int) -> list[str]:\\n    \\"\\"\\"First n train-pool scenes whose pairs exist (failed scenes are skipped).\\"\\"\\"\\n    failed = set(_failures(ws))\\n    pool = [s for s in split[\\"train_pool\\"] if s not in failed]\\n    return done_scenes(ws, pool)[:n]\\n\\n\\ndef _failures(ws: Workspace) -> dict:\\n    p = ws.root / \\"failed_scenes.json\\"\\n    return json.loads(p.read_text(encoding=\\"utf-8\\")) if p.exists() else {}\\n\\n\\n# ----------------------------------------------------------------------------- pairs\\ndef generate_pairs(\\n    ws: Workspace,\\n    split: dict,\\n    n_train: int,\\n    recipes: list[str],\\n    max_pairs_per_recipe: int = 100,\\n    only_test: bool = False,\\n    time_budget_h: float | None = None,\\n) -> None:\\n    \\"\\"\\"Make pairs for all test scenes, then train-pool scenes until n_train succeed.\\n\\n    Downloads the next scene in a background thread while the GPU trains on\\n    the current one.\\n    \\"\\"\\"\\n    t_start = time.time()\\n    test = set(split[\\"test\\"])\\n    failed = _failures(ws)\\n    queue = [s for s in split[\\"test\\"] if s not in failed]\\n    if not only_test:\\n        queue += [s for s in split[\\"train_pool\\"] if s not in failed]\\n\\n    def need_more_train() -> bool:\\n        return len(done_scenes(ws, [s for s in split[\\"train_pool\\"] if s not in failed])) < n_train\\n\\n    def wanted(s: str) -> bool:\\n        return not ws.pair_tar(s).exists() and (s in test or need_more_train())\\n\\n    pending = [s for s in queue if wanted(s)]\\n    log(f\\"pairs: {len(done_scenes(ws, list(test)))}/{len(test)} test done, \\"\\n        f\\"{len(done_scenes(ws, split[\'train_pool\']))} train done (target {n_train}), {len(pending)} candidates queued\\")\\n\\n    prefetch: dict[str, dict] = {}\\n\\n    def start_download(s: str):\\n        slot = {\\"done\\": threading.Event(), \\"dir\\": None, \\"error\\": None}\\n\\n        def run():\\n            try:\\n                slot[\\"dir\\"] = dl3dv.download_scene(s, ws.raw_dir(s))\\n            except Exception as e:  # noqa: BLE001 - reported below\\n                slot[\\"error\\"] = e\\n            finally:\\n                slot[\\"done\\"].set()\\n\\n        threading.Thread(target=run, daemon=True).start()\\n        prefetch[s] = slot\\n\\n    order = [s for s in pending]\\n    for k, scene in enumerate(order):\\n        if not wanted(scene):\\n            continue\\n        if time_budget_h and (time.time() - t_start) / 3600 > time_budget_h:\\n            log(f\\"pairs: time budget of {time_budget_h} h reached, stopping (re-run to continue)\\")\\n            break\\n        if scene not in prefetch:\\n            start_download(scene)\\n        nxt = next((s for s in order[k + 1 :] if wanted(s)), None)\\n        if nxt and nxt not in prefetch:\\n            start_download(nxt)\\n        slot = prefetch.pop(scene)\\n        slot[\\"done\\"].wait()\\n        is_test = scene in test\\n        try:\\n            if slot[\\"error\\"]:\\n                raise slot[\\"error\\"]\\n            generate_scene_pairs(\\n                ws, scene, slot[\\"dir\\"], recipes, is_test=is_test, max_pairs_per_recipe=max_pairs_per_recipe, log=log\\n            )\\n        except Exception as e:  # noqa: BLE001 - one bad scene must not stop the run\\n            failed[scene] = f\\"{type(e).__name__}: {e}\\"\\n            atomic_write_json(ws.root / \\"failed_scenes.json\\", failed)\\n            log(f\\"  {scene[:12]} FAILED: {failed[scene]}\\")\\n            traceback.print_exc()\\n        finally:\\n            dl3dv.remove_download(ws.raw_dir(scene))\\n        done_train = len(done_scenes(ws, split[\\"train_pool\\"]))\\n        log(f\\"pairs progress: test {len(done_scenes(ws, list(test)))}/{len(test)}, train {done_train}/{n_train}, \\"\\n            f\\"elapsed {(time.time() - t_start) / 60:.0f} min\\")\\n        if not only_test and not need_more_train() and all(ws.pair_tar(s).exists() or s in failed for s in test):\\n            break\\n    # Do not leave half-downloaded prefetches around.\\n    for s in list(prefetch):\\n        prefetch[s][\\"done\\"].wait()\\n        dl3dv.remove_download(ws.raw_dir(s))\\n\\n\\ndef generate_one(ws: Workspace, scene: str, recipes: list[str], is_test: bool, max_pairs_per_recipe: int = 100) -> Path:\\n    if ws.pair_tar(scene).exists():\\n        log(f\\"pairs for {scene[:12]} already on Drive\\")\\n        return ws.pair_tar(scene)\\n    try:\\n        scene_dir = dl3dv.download_scene(scene, ws.raw_dir(scene))\\n        return generate_scene_pairs(ws, scene, scene_dir, recipes, is_test=is_test,\\n                                    max_pairs_per_recipe=max_pairs_per_recipe, log=log)\\n    finally:\\n        dl3dv.remove_download(ws.raw_dir(scene))\\n\\n\\ndef sanity_overfit(ws: Workspace, scene: str, steps: int = 300) -> bool:\\n    \\"\\"\\"Train the restorer briefly on ONE scene and check it beats its input there.\\n\\n    This proves the plumbing (data, model, loss, optimiser) works before any\\n    long run. Outputs stay on local disk.\\n    \\"\\"\\"\\n    import csv as _csv\\n\\n    from .train_restorer import RestorerConfig, train_restorer\\n\\n    root = extract(ws, [scene])\\n    local_ws = Workspace(root=ws.local / \\"sanity\\", local=ws.local / \\"sanity_local\\")\\n    shutil.rmtree(local_ws.runs_dir / \\"sanity\\", ignore_errors=True)\\n    cfg = RestorerConfig(\\n        run_name=\\"sanity\\", scenes=[scene], steps=steps, sample_every=100, log_every=10,\\n        local_ckpt_every=10**9, gram_warmup_steps=10**9, warmup_steps=50, num_workers=4,\\n    )\\n    train_restorer(local_ws, cfg, root, log=log)\\n    rows = [r for r in _csv.DictReader(open(local_ws.runs_dir / \\"sanity\\" / \\"log.csv\\")) if r[\\"val_psnr\\"]]\\n    last = rows[-1]\\n    gain = float(last[\\"val_psnr\\"]) - float(last[\\"val_psnr_input\\"])\\n    ok = gain > 0.5\\n    log(f\\"SANITY {\'PASSED\' if ok else \'FAILED\'}: val PSNR {float(last[\'val_psnr\']):.2f} vs input \\"\\n        f\\"{float(last[\'val_psnr_input\']):.2f} ({gain:+.2f} dB) after {steps} steps\\")\\n    (local_ws.runs_dir / \\"sanity\\" / \\"model_final.pt\\").unlink(missing_ok=True)\\n    return ok\\n\\n\\ndef extract(ws: Workspace, scenes: list[str]) -> Path:\\n    t = time.time()\\n    root = extract_pairs(ws, scenes)\\n    log(f\\"extracted {len(scenes)} scenes to {root} in {time.time() - t:.0f}s\\")\\n    return root\\n\\n\\n# ----------------------------------------------------------------------------- reports\\ndef show_markdown(path: Path) -> None:\\n    try:\\n        from IPython.display import Markdown, display\\n\\n        display(Markdown(path.read_text(encoding=\\"utf-8\\")))\\n    except Exception:  # noqa: BLE001 - plain text fallback outside notebooks\\n        print(path.read_text(encoding=\\"utf-8\\"))\\n\\n", "lib/splat_restorer/train_ab.py": "\\"\\"\\"A/B training variants of the app\'s static trainer on one captured scene.\\n\\nEvery variant trains the same Trainer4DGS setup the hybrid notebook uses\\n(\\"Spirula dataset + our trainer\\"), on the same frames and seed, and is scored\\non the same held-out photos. The trainer classes are passed in, so this module\\ndoes not depend on the app\'s source layout.\\n\\nVariants change one thing at a time:\\n\\n* ``lr_decay``: exponential decay of the Gaussian position learning rate\\n  (standard 3DGS; the app\'s trainer keeps it constant). Applied from the\\n  trainer\'s progress callback, so the trainer code is unchanged.\\n* ``clean``: training frames are undistorted at full resolution, area-averaged\\n  down (2x2 averaging halves sensor noise) and lightly denoised. Held-out\\n  photos are never trained on; they are scored both as captured and cleaned.\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport time\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom PIL import Image\\n\\nfrom .colmap_io import read_sparse_model\\nfrom .own_splat import SplatData, _k_and_dist, _scores, _to_u8, _w2c_t, comparison_image, load_ply\\nfrom .gs_trainer import render\\n\\n\\n@dataclass\\nclass Variant:\\n    name: str\\n    steps: int = 30000\\n    cap: int = 1_000_000\\n    lr_decay: bool = False\\n    lr_final_frac: float = 0.01\\n    clean: bool = False\\n    note: str = \\"\\"\\n\\n\\nVARIANTS = {\\n    \\"baseline\\": Variant(\\"baseline\\", note=\\"28 Eylül eğitimiyle aynı ayarlar\\"),\\n    \\"lr_decay\\": Variant(\\"lr_decay\\", lr_decay=True, note=\\"konum öğrenme oranı 100x azalır\\"),\\n    \\"lr_decay_clean\\": Variant(\\"lr_decay_clean\\", lr_decay=True, clean=True, note=\\"+ gürültüsü azaltılmış eğitim kareleri\\"),\\n    \\"lr_decay_60k\\": Variant(\\"lr_decay_60k\\", steps=60000, lr_decay=True, note=\\"+ 60k adım\\"),\\n}\\n\\n\\n# ----------------------------------------------------------------------------- clean frames\\n\\n\\ndef clean_frames(source: Path, data: SplatData, out: Path, model_name: str = \\"0\\", nlm_h: float = 3.0,\\n                 workers: int = 8, log=print) -> Path:\\n    \\"\\"\\"Frames on the same cameras as ``data``, area-downsampled from full resolution and denoised.\\"\\"\\"\\n    import cv2\\n\\n    out = Path(out)\\n    frames = out / \\"frames\\"\\n    frames.mkdir(parents=True, exist_ok=True)\\n    model = read_sparse_model(Path(source) / \\"sparse\\" / model_name)\\n    cam = next(iter(model.cameras.values()))\\n    K, dist = _k_and_dist(cam)\\n    w, h = cam.width, cam.height\\n    sx, sy = w / data.width, h / data.height\\n    # Same output camera as data.K, expressed at full resolution (pixel-centre convention).\\n    k_full = data.K.copy()\\n    k_full[0, 0] *= sx; k_full[1, 1] *= sy\\n    k_full[0, 2] = (data.K[0, 2] + 0.5) * sx - 0.5\\n    k_full[1, 2] = (data.K[1, 2] + 0.5) * sy - 0.5\\n    mx, my = cv2.initUndistortRectifyMap(K, dist, None, k_full, (w, h), cv2.CV_32FC1)\\n\\n    def convert(i):\\n        dest = frames / data.frames[i]\\n        if dest.exists():\\n            return\\n        img = cv2.imread(str(Path(source) / \\"images\\" / data.names[i]))\\n        full = cv2.remap(img, mx, my, cv2.INTER_LINEAR)\\n        small = cv2.resize(full, (data.width, data.height), interpolation=cv2.INTER_AREA)\\n        if nlm_h > 0:\\n            small = cv2.fastNlMeansDenoisingColored(small, None, nlm_h, nlm_h, 7, 21)\\n        tmp = dest.with_suffix(\\".tmp.png\\")\\n        cv2.imwrite(str(tmp), small, [cv2.IMWRITE_PNG_COMPRESSION, 1])\\n        tmp.replace(dest)\\n\\n    todo = [i for i in range(data.num_frames) if not (frames / data.frames[i]).exists()]\\n    if todo:\\n        log(f\\"[clean] {len(todo)} frames: full-res undistort -> area downsample -> NLM h={nlm_h}\\")\\n        with ThreadPoolExecutor(workers) as pool:\\n            for n, _ in enumerate(pool.map(convert, todo), 1):\\n                if n % 200 == 0 or n == len(todo):\\n                    log(f\\"[clean] {n}/{len(todo)}\\")\\n    return frames\\n\\n\\n# ----------------------------------------------------------------------------- training\\n\\n\\ndef init_points(source: Path, model_name: str, seed: int, max_points: int = 150_000) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\\n    \\"\\"\\"COLMAP points, subsampled exactly like the hybrid notebook.\\"\\"\\"\\n    model = read_sparse_model(Path(source) / \\"sparse\\" / model_name)\\n    xyz, rgb = model.points_xyz, model.points_rgb\\n    ids = np.arange(len(xyz))\\n    if len(ids) > max_points:\\n        ids = np.sort(np.random.default_rng(seed).choice(len(xyz), max_points, replace=False))\\n    return xyz, rgb, ids\\n\\n\\ndef train_variant(\\n    v: Variant,\\n    data: SplatData,\\n    frames_dir: Path,\\n    xyz: np.ndarray,\\n    rgb: np.ndarray,\\n    init_ids: np.ndarray,\\n    holdout: list[int],\\n    work: Path,\\n    app,  # namespace with GaussianModel, DeformationField, Trainer4DGS, export_to_ply, scene_extent\\n    seed: int = 42,\\n    log_interval: int = 200,\\n    log=print,\\n) -> tuple[Path, dict]:\\n    \\"\\"\\"Train one variant with the app\'s trainer; returns (exported PLY, info).\\"\\"\\"\\n    torch.manual_seed(seed)\\n    np.random.seed(seed)\\n    gs = app.GaussianModel(torch.from_numpy(xyz[init_ids]).float(),\\n                           torch.from_numpy(rgb[init_ids].astype(\\"float32\\") / 255), fourier_K=0)\\n    deform = app.DeformationField(resolution=8, feat_dim=4, mlp_width=32, mlp_depth=1, num_time_freqs=2)\\n    for p in deform.parameters():\\n        p.requires_grad_(False)\\n    trainer = app.Trainer4DGS(\\n        gs, deform, scene_extent=app.scene_extent(xyz), max_gaussians=v.cap,\\n        density_end_iter=min(25000, v.steps // 2), deform_pos_mode=\\"mlp\\",\\n        lambda_deform_reg=0, lambda_smoothness=0, lambda_rigidity=0, lambda_fourier_reg=0, lambda_mask_motion=0,\\n    )\\n    lr0 = trainer.lr_specs[\\"means\\"]\\n    curve = []\\n\\n    def progress(it, total, loss, psnr, n):\\n        curve.append({\\"iter\\": it, \\"loss\\": loss, \\"train_psnr\\": psnr, \\"splats\\": n})\\n        if v.lr_decay:\\n            lr = lr0 * v.lr_final_frac ** (min(it, v.steps) / v.steps)\\n            for g in trainer.optimizer.param_groups:\\n                if any(p is trainer.gs.means for p in g[\\"params\\"]):\\n                    g[\\"lr\\"] = lr\\n\\n    t0 = time.time()\\n    trainer.train(\\n        [frames_dir / f for f in data.frames], torch.from_numpy(data.K).float(),\\n        [torch.from_numpy(w).float() for w in data.w2c],\\n        n_iters=v.steps, image_size=(data.width, data.height), static_mode=True, preload_to_ram=True,\\n        log_interval=log_interval, ckpt_dir=work / \\"ckpt\\", ckpt_interval=v.steps,\\n        sh_progressive_schedule=True, holdout_indices=holdout, progress_callback=progress,\\n    )\\n    torch.cuda.synchronize()\\n    seconds = time.time() - t0\\n    ply = app.export_to_ply(gs, None, work / \\"export\\", num_timestamps=1)[0]\\n    info = {\\"variant\\": asdict(v), \\"train_seconds\\": seconds, \\"splats\\": int(gs.num_points), \\"curve\\": curve}\\n    del trainer, gs, deform\\n    torch.cuda.empty_cache()\\n    return Path(ply), info\\n\\n\\n# ----------------------------------------------------------------------------- evaluation\\n\\n\\n@torch.no_grad()\\ndef evaluate_params(params, deg: int, data: SplatData, holdout: list[int], clean_dir: Path | None, net_lpips,\\n                    render_dir: Path, rasterize_mode: str = \\"classic\\", device=\\"cuda\\", **raster_kwargs) -> dict:\\n    \\"\\"\\"Score held-out frames against the captured photo and (if given) its cleaned version.\\"\\"\\"\\n    w2c, K = _w2c_t(data, device)\\n    render_dir.mkdir(parents=True, exist_ok=True)\\n    rows = {}\\n    for i in holdout:\\n        img, _ = render(params, w2c[i], K, data.width, data.height, deg, rasterize_mode, **raster_kwargs)\\n        row = _scores(img, data.photo(i, device), net_lpips)\\n        if clean_dir is not None:\\n            clean = torch.from_numpy(np.asarray(Image.open(clean_dir / data.frames[i]).convert(\\"RGB\\")).copy())\\n            row[\\"clean\\"] = _scores(img, clean.to(device).float() / 255, net_lpips)\\n        rows[str(i)] = row\\n        Image.fromarray(_to_u8(img)).save(render_dir / f\\"{i:06d}.jpg\\", quality=92)\\n    return rows\\n\\n\\ndef evaluate_ply(ply: Path, data: SplatData, holdout: list[int], clean_dir: Path | None, net_lpips,\\n                 render_dir: Path, device=\\"cuda\\") -> dict:\\n    params, deg = load_ply(ply, device)\\n    rows = evaluate_params(params, deg, data, holdout, clean_dir, net_lpips, render_dir, device=device)\\n    del params\\n    torch.cuda.empty_cache()\\n    return rows\\n\\n\\ndef summarise(results: dict) -> dict:\\n    out = {}\\n    for name, r in results.items():\\n        rows = list(r[\\"frames\\"].values())\\n        m = {k: float(np.mean([x[k] for x in rows])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")}\\n        if rows and \\"clean\\" in rows[0]:\\n            m.update({f\\"clean_{k}\\": float(np.mean([x[\\"clean\\"][k] for x in rows])) for k in (\\"psnr\\", \\"ssim\\", \\"lpips\\")})\\n        m.update(splats=r[\\"info\\"][\\"splats\\"], minutes=r[\\"info\\"][\\"train_seconds\\"] / 60)\\n        out[name] = m\\n    return out\\n\\n\\ndef markdown(summary: dict, n_frames: int) -> str:\\n    base = summary.get(\\"baseline\\") or next(iter(summary.values()))\\n    has_clean = \\"clean_psnr\\" in base\\n    head = \\"| varyant | PSNR ↑ | Δ | SSIM ↑ | LPIPS ↓ | Δ |\\" + (\\" PSNR (temiz foto) | Δ |\\" if has_clean else \\"\\") + \\" splat | dk |\\"\\n    sep = \\"|---\\" * (head.count(\\"|\\") - 1) + \\"|\\"\\n    lines = [f\\"## Eğitim varyantları: {n_frames} saklanan kare (hiçbiri eğitimde kullanılmadı)\\", \\"\\", head, sep]\\n    for name, m in summary.items():\\n        row = (f\\"| {name} | {m[\'psnr\']:.2f} | {m[\'psnr\'] - base[\'psnr\']:+.2f} | {m[\'ssim\']:.4f} | \\"\\n               f\\"{m[\'lpips\']:.4f} | {m[\'lpips\'] - base[\'lpips\']:+.4f} |\\")\\n        if has_clean:\\n            row += f\\" {m[\'clean_psnr\']:.2f} | {m[\'clean_psnr\'] - base[\'clean_psnr\']:+.2f} |\\"\\n        row += f\\" {m[\'splats\']:,} | {m[\'minutes\']:.0f} |\\"\\n        lines.append(row)\\n    lines += [\\"\\", \\"Δ: ilk satıra (baseline) göre. \'Temiz foto\' sütunu, sensör gürültüsü azaltılmış fotoğrafa karşı ölçer; \\"\\n              \\"gürültüsüz eğitilen varyant orijinal fotoğrafın gürültüsünü taklit etmediği için orada daha adil karşılaştırılır.\\"]\\n    return \\"\\\\n\\".join(lines)\\n\\n\\ndef comparison_sheets(results: dict, data: SplatData, out_dir: Path, render_root: Path, n_worst: int = 4, n_even: int = 4) -> list[Path]:\\n    names = list(results)\\n    first = results[names[0]][\\"frames\\"]\\n    ids = sorted(int(i) for i in first)\\n    worst = sorted(ids, key=lambda i: first[str(i)][\\"psnr\\"])[:n_worst]\\n    even = [ids[k] for k in np.linspace(0, len(ids) - 1, n_even + 2).round().astype(int)[1:-1]]\\n    picks = list(dict.fromkeys(worst + even))\\n    paths = []\\n    for i in picks:\\n        tiles = {\\"foto\\": _to_u8(data.photo(i, \\"cpu\\"))}\\n        for n in names:\\n            tiles[n] = np.asarray(Image.open(render_root / n / \\"renders\\" / f\\"{i:06d}.jpg\\").convert(\\"RGB\\"))\\n        p = out_dir / f\\"compare_{i:06d}.jpg\\"\\n        comparison_image(tiles, p, tile_width=560, box_from=names[0])\\n        paths.append(p)\\n    return paths\\n\\n\\ndef save_json(path: Path, obj) -> None:\\n    tmp = path.with_suffix(\\".tmp\\")\\n    tmp.write_text(json.dumps(obj, indent=1), encoding=\\"utf-8\\")\\n    tmp.replace(path)\\n", "lib/splat_restorer/train_restorer.py": "\\"\\"\\"Train the diffusion restorer on extracted pairs. Resumable from Drive.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport csv\\nimport json\\nimport math\\nimport random\\nimport time\\nfrom dataclasses import asdict, dataclass, field\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\n\\nfrom .data import PairDataset, collect_items, split_holdout\\nfrom .workspace import Workspace, atomic_copy, atomic_write_json\\n\\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\\nIMAGENET_STD = (0.229, 0.224, 0.225)\\nGRAM_LAYERS = {3: 1.0 / 2.6, 8: 1.0 / 4.8, 15: 1.0 / 3.7, 22: 1.0 / 5.6, 29: 10.0 / 1.5}\\n\\n\\n@dataclass\\nclass RestorerConfig:\\n    run_name: str\\n    scenes: list[str]\\n    use_ref: bool = True\\n    steps: int = 10_000\\n    batch_size: int = 4\\n    lr: float = 2e-5\\n    warmup_steps: int = 500\\n    weight_decay: float = 1e-2\\n    lambda_l2: float = 1.0\\n    lambda_lpips: float = 1.0\\n    lambda_gram: float = 1.0\\n    gram_warmup_steps: int = 2000\\n    crop: int = 512\\n    timestep: int = 199\\n    lora_rank_vae: int = 4\\n    grad_checkpointing: bool = False\\n    num_workers: int = 8\\n    log_every: int = 50\\n    sample_every: int = 500\\n    local_ckpt_every: int = 500\\n    drive_ckpt_every: int = 2000\\n    val_items: int = 64\\n    seed: int = 0\\n    recipes: list[str] = field(default_factory=list)\\n\\n\\ndef gram_loss(pred: torch.Tensor, target: torch.Tensor, vgg: torch.nn.Module) -> torch.Tensor:\\n    \\"\\"\\"Difix-style style loss on VGG16 feature Gram matrices (inputs in [0, 1]).\\"\\"\\"\\n    mean = torch.tensor(IMAGENET_MEAN, device=pred.device).view(1, 3, 1, 1)\\n    std = torch.tensor(IMAGENET_STD, device=pred.device).view(1, 3, 1, 1)\\n    x, y = (pred - mean) / std, (target - mean) / std\\n    total = pred.new_zeros(())\\n    for i, layer in enumerate(vgg):\\n        x, y = layer(x), layer(y)\\n        if i in GRAM_LAYERS:\\n            b, d, h, w = x.shape\\n            fx, fy = x.reshape(b, d, h * w).float(), y.reshape(b, d, h * w).float()\\n            gx = fx @ fx.transpose(1, 2)\\n            gy = fy @ fy.transpose(1, 2)\\n            total = total + GRAM_LAYERS[i] * ((gx - gy) ** 2).mean() / (d * h * w)\\n        if i >= max(GRAM_LAYERS):\\n            break\\n    return total\\n\\n\\ndef _infinite(loader):\\n    while True:\\n        for batch in loader:\\n            yield batch\\n\\n\\ndef _to_views(batch, use_ref: bool, device) -> tuple[torch.Tensor, torch.Tensor]:\\n    deg, gt, ref = (batch[k].to(device, non_blocking=True) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n    if use_ref:\\n        x = torch.stack([deg, ref], dim=1)\\n        y = torch.stack([gt, ref], dim=1)\\n    else:\\n        x, y = deg[:, None], gt[:, None]\\n    return x * 2 - 1, y * 2 - 1\\n\\n\\ndef _save_grid(path: Path, rows: list[list[torch.Tensor]]) -> None:\\n    from PIL import Image\\n\\n    lines = [torch.cat([t.clamp(0, 1) for t in row], dim=2) for row in rows]\\n    grid = torch.cat(lines, dim=1)\\n    arr = (grid.permute(1, 2, 0).cpu().numpy() * 255 + 0.5).astype(np.uint8)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    Image.fromarray(arr).save(path, quality=90)\\n\\n\\n@torch.no_grad()\\ndef validate(model, val_ds, use_ref: bool, net_lpips, device, max_items: int) -> dict:\\n    model.set_eval()\\n    l2s, lps, psnrs, base_psnrs = [], [], [], []\\n    for i in range(min(len(val_ds), max_items)):\\n        s = val_ds[i]\\n        deg, gt, ref = (s[k][None].to(device) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n        out = model.restore(deg, ref if use_ref else None)\\n        mse = F.mse_loss(out, gt).item()\\n        l2s.append(mse)\\n        psnrs.append(10 * math.log10(1 / max(mse, 1e-12)))\\n        base_psnrs.append(10 * math.log10(1 / max(F.mse_loss(deg, gt).item(), 1e-12)))\\n        lps.append(net_lpips(out * 2 - 1, gt * 2 - 1).mean().item())\\n    model.set_train()\\n    return {\\n        \\"val_psnr\\": float(np.mean(psnrs)),\\n        \\"val_psnr_input\\": float(np.mean(base_psnrs)),\\n        \\"val_lpips\\": float(np.mean(lps)),\\n        \\"val_l2\\": float(np.mean(l2s)),\\n    }\\n\\n\\ndef train_restorer(ws: Workspace, cfg: RestorerConfig, pairs_root: Path, device: str = \\"cuda\\", log=print) -> Path:\\n    import lpips\\n    import torchvision\\n\\n    from .restorer_model import Restorer\\n\\n    run_dir = ws.runs_dir / cfg.run_name\\n    run_dir.mkdir(parents=True, exist_ok=True)\\n    final_path = run_dir / \\"model_final.pt\\"\\n    if final_path.exists():\\n        log(f\\"[{cfg.run_name}] already finished -> {final_path}\\")\\n        return final_path\\n    local_dir = ws.local / \\"runs\\" / cfg.run_name\\n    local_dir.mkdir(parents=True, exist_ok=True)\\n    atomic_write_json(run_dir / \\"config.json\\", asdict(cfg))\\n\\n    random.seed(cfg.seed)\\n    np.random.seed(cfg.seed)\\n    torch.manual_seed(cfg.seed)\\n    torch.backends.cuda.matmul.allow_tf32 = True\\n    torch.backends.cudnn.allow_tf32 = True\\n\\n    items = collect_items(pairs_root, cfg.scenes, cfg.recipes or None)\\n    train_items, val_items = split_holdout(items, frac=0.02, seed=cfg.seed)\\n    log(f\\"[{cfg.run_name}] {len(cfg.scenes)} scenes, {len(train_items)} train pairs, {len(val_items)} val pairs, ref={cfg.use_ref}\\")\\n    train_ds = PairDataset(train_items, train=True, crop=cfg.crop)\\n    val_ds = PairDataset(val_items, train=False)\\n    loader = torch.utils.data.DataLoader(\\n        train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers,\\n        pin_memory=True, drop_last=True, persistent_workers=cfg.num_workers > 0,\\n    )\\n\\n    model = Restorer(timestep=cfg.timestep, lora_rank_vae=cfg.lora_rank_vae, device=device)\\n    model.set_train()\\n    if cfg.grad_checkpointing:\\n        model.unet.enable_gradient_checkpointing()\\n    params = list(model.unet.parameters()) + model.vae_trainable_parameters()\\n    opt = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)\\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(cfg.warmup_steps, 1)))\\n\\n    net_lpips = lpips.LPIPS(net=\\"vgg\\", verbose=False).to(device).eval().requires_grad_(False)\\n    vgg = torchvision.models.vgg16(weights=\\"IMAGENET1K_V1\\").features[:30].to(device).eval().requires_grad_(False)\\n\\n    # Resume: prefer the newer of local / Drive checkpoints.\\n    step = 0\\n    candidates = [p for p in (local_dir / \\"ckpt.pt\\", run_dir / \\"ckpt.pt\\") if p.exists()]\\n    if candidates:\\n        ck_path = max(candidates, key=lambda p: torch.load(p, map_location=\\"cpu\\", weights_only=False, mmap=True)[\\"step\\"])\\n        ck = torch.load(ck_path, map_location=\\"cpu\\", weights_only=False)\\n        model.load_trainable_state_dict(ck[\\"model\\"])\\n        opt.load_state_dict(ck[\\"opt\\"])\\n        sched.load_state_dict(ck[\\"sched\\"])\\n        step = int(ck[\\"step\\"])\\n        log(f\\"[{cfg.run_name}] resumed from {ck_path} at step {step}\\")\\n        del ck\\n\\n    log_path = run_dir / \\"log.csv\\"\\n    new_log = not log_path.exists()\\n    log_f = open(log_path, \\"a\\", newline=\\"\\")\\n    writer = csv.writer(log_f)\\n    if new_log:\\n        writer.writerow([\\"step\\", \\"loss\\", \\"l2\\", \\"lpips\\", \\"gram\\", \\"lr\\", \\"sec_per_step\\", \\"val_psnr\\", \\"val_psnr_input\\", \\"val_lpips\\"])\\n\\n    def save_ckpt(to_drive: bool):\\n        state = {\\"model\\": model.trainable_state_dict(), \\"opt\\": opt.state_dict(), \\"sched\\": sched.state_dict(), \\"step\\": step}\\n        tmp = local_dir / \\"ckpt.pt.partial\\"\\n        torch.save(state, tmp)\\n        tmp.replace(local_dir / \\"ckpt.pt\\")\\n        if to_drive:\\n            atomic_copy(local_dir / \\"ckpt.pt\\", run_dir / \\"ckpt.pt\\")\\n\\n    it = _infinite(loader)\\n    t_last = time.time()\\n    while step < cfg.steps:\\n        batch = next(it)\\n        x, y = _to_views(batch, cfg.use_ref, device)\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n            pred = model(x)\\n        pred_f = pred.float().flatten(0, 1)\\n        y_f = y.flatten(0, 1)\\n        l2 = F.mse_loss(pred_f, y_f) * cfg.lambda_l2\\n        with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n            lp = net_lpips(pred_f, y_f).float().mean() * cfg.lambda_lpips\\n        loss = l2 + lp\\n        gram = torch.zeros((), device=device)\\n        if cfg.lambda_gram > 0 and step >= cfg.gram_warmup_steps:\\n            h, w = pred_f.shape[-2:]\\n            c = min(400, h, w)\\n            top, left = random.randint(0, h - c), random.randint(0, w - c)\\n            p_c = pred_f[..., top : top + c, left : left + c] * 0.5 + 0.5\\n            y_c = y_f[..., top : top + c, left : left + c] * 0.5 + 0.5\\n            with torch.autocast(\\"cuda\\", dtype=torch.bfloat16):\\n                gram = gram_loss(p_c, y_c, vgg) * cfg.lambda_gram\\n            loss = loss + gram\\n        opt.zero_grad(set_to_none=True)\\n        loss.backward()\\n        torch.nn.utils.clip_grad_norm_(params, 1.0)\\n        opt.step()\\n        sched.step()\\n        step += 1\\n\\n        if not math.isfinite(loss.item()):\\n            raise FloatingPointError(f\\"[{cfg.run_name}] non-finite loss at step {step}\\")\\n\\n        val = {}\\n        if step % cfg.sample_every == 0 or step == cfg.steps:\\n            val = validate(model, val_ds, cfg.use_ref, net_lpips, device, cfg.val_items)\\n            rows = []\\n            for i in range(min(4, len(val_ds))):\\n                s = val_ds[i]\\n                deg, gt, ref = (s[k][None].to(device) for k in (\\"deg\\", \\"gt\\", \\"ref\\"))\\n                model.set_eval()\\n                out = model.restore(deg, ref if cfg.use_ref else None)\\n                model.set_train()\\n                rows.append([deg[0], out[0], gt[0], ref[0]])\\n            _save_grid(run_dir / \\"samples\\" / f\\"step_{step:06d}.jpg\\", rows)\\n            log(\\n                f\\"[{cfg.run_name}] step {step}: val PSNR {val[\'val_psnr\']:.2f} \\"\\n                f\\"(input {val[\'val_psnr_input\']:.2f}), LPIPS {val[\'val_lpips\']:.4f}\\"\\n            )\\n        if step % cfg.log_every == 0:\\n            dt = (time.time() - t_last) / cfg.log_every\\n            t_last = time.time()\\n            writer.writerow([\\n                step, f\\"{loss.item():.5f}\\", f\\"{l2.item():.5f}\\", f\\"{lp.item():.5f}\\", f\\"{gram.item():.5f}\\",\\n                f\\"{sched.get_last_lr()[0]:.2e}\\", f\\"{dt:.3f}\\",\\n                val.get(\\"val_psnr\\", \\"\\"), val.get(\\"val_psnr_input\\", \\"\\"), val.get(\\"val_lpips\\", \\"\\"),\\n            ])\\n            log_f.flush()\\n            if step % (cfg.log_every * 10) == 0:\\n                log(f\\"[{cfg.run_name}] step {step}/{cfg.steps} loss {loss.item():.4f} ({dt:.2f}s/step)\\")\\n        if step % cfg.local_ckpt_every == 0 and step < cfg.steps:\\n            save_ckpt(to_drive=step % cfg.drive_ckpt_every == 0)\\n\\n    log_f.close()\\n    final = model.trainable_state_dict()\\n    final[\\"unet\\"] = {k: v.half() for k, v in final[\\"unet\\"].items()}\\n    final[\\"use_ref\\"] = cfg.use_ref\\n    final[\\"lora_rank_vae\\"] = cfg.lora_rank_vae\\n    tmp = local_dir / \\"model_final.pt\\"\\n    torch.save(final, tmp)\\n    atomic_copy(tmp, final_path)\\n    # The resume checkpoint is large; drop it once the final weights are safe.\\n    for p in (run_dir / \\"ckpt.pt\\", local_dir / \\"ckpt.pt\\"):\\n        p.unlink(missing_ok=True)\\n    atomic_write_json(run_dir / \\"DONE.json\\", {\\"steps\\": step, \\"finished\\": time.strftime(\\"%Y-%m-%d %H:%M:%S\\")})\\n    log(f\\"[{cfg.run_name}] finished -> {final_path}\\")\\n    return final_path\\n\\n\\ndef load_restorer(path: Path, device: str = \\"cuda\\"):\\n    from .restorer_model import Restorer\\n\\n    sd = torch.load(path, map_location=\\"cpu\\", weights_only=False)\\n    model = Restorer(timestep=int(sd.get(\\"timestep\\", 199)), lora_rank_vae=int(sd.get(\\"lora_rank_vae\\", 4)), device=device)\\n    sd[\\"unet\\"] = {k: v.float() for k, v in sd[\\"unet\\"].items()}\\n    model.load_trainable_state_dict(sd)\\n    model.set_eval()\\n    return model, bool(sd.get(\\"use_ref\\", False))\\n", "lib/splat_restorer/video.py": "\\"\\"\\"Run a restorer over every frame of a splat render video.\\n\\nThere is no ground-truth photo for a render video, so the outputs are for\\nlooking at (side-by-side video, zoomed comparison sheet) plus three rough\\nnumbers: how much the model changed each frame, how much sharper it got, and\\nwhether frame-to-frame flicker went up (per-frame restoration is not\\ntemporally consistent).\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport shutil\\nimport subprocess\\nimport time\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom PIL import Image, ImageDraw\\n\\n\\ndef probe(video: Path) -> dict:\\n    out = subprocess.run(\\n        [\\"ffprobe\\", \\"-v\\", \\"error\\", \\"-select_streams\\", \\"v:0\\", \\"-show_entries\\",\\n         \\"stream=width,height,r_frame_rate,nb_frames:format=duration\\", \\"-of\\", \\"json\\", str(video)],\\n        check=True, capture_output=True, text=True,\\n    ).stdout\\n    info = json.loads(out)\\n    s = info[\\"streams\\"][0]\\n    num, den = (int(x) for x in s[\\"r_frame_rate\\"].split(\\"/\\"))\\n    return {\\n        \\"width\\": int(s[\\"width\\"]),\\n        \\"height\\": int(s[\\"height\\"]),\\n        \\"fps\\": num / den if den else 30.0,\\n        \\"duration\\": float(info.get(\\"format\\", {}).get(\\"duration\\", 0) or 0),\\n    }\\n\\n\\ndef target_size(width: int, height: int, long_edge: int) -> tuple[int, int]:\\n    \\"\\"\\"Scale so the long edge is ``long_edge`` (never upscale); both sides multiples of 8.\\"\\"\\"\\n    s = min(1.0, long_edge / max(width, height))\\n    return max(8, round(width * s / 8) * 8), max(8, round(height * s / 8) * 8)\\n\\n\\ndef extract_frames(video: Path, out_dir: Path, size: tuple[int, int], stride: int = 1, max_seconds: float = 0) -> int:\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    vf = f\\"scale={size[0]}:{size[1]}:flags=lanczos\\"\\n    if stride > 1:\\n        vf = f\\"select=\'not(mod(n\\\\\\\\,{stride}))\',{vf}\\"\\n    cmd = [\\"ffmpeg\\", \\"-v\\", \\"error\\", \\"-y\\", \\"-i\\", str(video)]\\n    if max_seconds > 0:\\n        cmd += [\\"-t\\", str(max_seconds)]\\n    out = str(out_dir / \\"%05d.png\\")\\n    # -fps_mode needs ffmpeg 5.1+; Colab ships 4.4, which only knows -vsync.\\n    try:\\n        subprocess.run(cmd + [\\"-vf\\", vf, \\"-fps_mode\\", \\"vfr\\", out], check=True, capture_output=True)\\n    except subprocess.CalledProcessError:\\n        subprocess.run(cmd + [\\"-vf\\", vf, \\"-vsync\\", \\"vfr\\", out], check=True)\\n    return len(list(out_dir.glob(\\"*.png\\")))\\n\\n\\ndef _load(path: Path, device) -> torch.Tensor:\\n    arr = np.asarray(Image.open(path).convert(\\"RGB\\"), dtype=np.float32) / 255.0\\n    return torch.from_numpy(arr).permute(2, 0, 1)[None].to(device)\\n\\n\\ndef _save(t: torch.Tensor, path: Path) -> None:\\n    arr = (t[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy() * 255 + 0.5).astype(np.uint8)\\n    Image.fromarray(arr).save(path)\\n\\n\\ndef load_ref(path: Path, size: tuple[int, int], device) -> torch.Tensor:\\n    \\"\\"\\"Reference photo, centre-cropped to the frame aspect and resized to the frame size.\\"\\"\\"\\n    img = Image.open(path).convert(\\"RGB\\")\\n    w, h = size\\n    s = max(w / img.width, h / img.height)\\n    img = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)\\n    left, top = (img.width - w) // 2, (img.height - h) // 2\\n    img = img.crop((left, top, left + w, top + h))\\n    return torch.from_numpy(np.asarray(img, dtype=np.float32) / 255.0).permute(2, 0, 1)[None].to(device)\\n\\n\\ndef restore_frames(model, in_dir: Path, out_dir: Path, ref: torch.Tensor | None = None, device=\\"cuda\\", log=print) -> float:\\n    \\"\\"\\"Restore every frame not already in ``out_dir``. Returns seconds per frame.\\"\\"\\"\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    todo = [p for p in sorted(in_dir.glob(\\"*.png\\")) if not (out_dir / p.name).exists()]\\n    if not todo:\\n        return 0.0\\n    log(f\\"[video] restoring {len(todo)} frames\\")\\n    t0, n = time.time(), 0\\n    for i, p in enumerate(todo):\\n        x = _load(p, device)\\n        h, w = x.shape[-2:]\\n        # The UNet wants latent sides divisible by 8, i.e. image sides divisible by 64.\\n        ph, pw = (-h) % 64, (-w) % 64\\n        xp = F.pad(x, (0, pw, 0, ph), mode=\\"reflect\\") if ph or pw else x\\n        rp = F.pad(ref, (0, pw, 0, ph), mode=\\"reflect\\") if ref is not None and (ph or pw) else ref\\n        y = model.restore(xp, rp)[..., :h, :w]\\n        _save(y, out_dir / p.name)\\n        n += 1\\n        if i == 0 or (i + 1) % 25 == 0 or i + 1 == len(todo):\\n            log(f\\"[video] {i + 1}/{len(todo)}  {(time.time() - t0) / n:.2f} s/frame\\")\\n    return (time.time() - t0) / max(n, 1)\\n\\n\\ndef encode(frames_dir: Path, fps: float, out: Path) -> None:\\n    subprocess.run(\\n        [\\"ffmpeg\\", \\"-v\\", \\"error\\", \\"-y\\", \\"-framerate\\", f\\"{fps:.6f}\\", \\"-i\\", str(frames_dir / \\"%05d.png\\"),\\n         \\"-c:v\\", \\"libx264\\", \\"-crf\\", \\"16\\", \\"-preset\\", \\"medium\\", \\"-pix_fmt\\", \\"yuv420p\\", str(out)],\\n        check=True,\\n    )\\n\\n\\ndef _label(img: Image.Image, text: str) -> Image.Image:\\n    d = ImageDraw.Draw(img)\\n    pad = max(4, img.height // 120)\\n    box = d.textbbox((0, 0), text)\\n    d.rectangle((0, 0, box[2] + 2 * pad, box[3] + 2 * pad), fill=(0, 0, 0))\\n    d.text((pad, pad), text, fill=(255, 255, 255))\\n    return img\\n\\n\\ndef side_by_side(before_dir: Path, after_dir: Path, out_dir: Path) -> None:\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    for p in sorted(after_dir.glob(\\"*.png\\")):\\n        if (out_dir / p.name).exists():\\n            continue\\n        a = _label(Image.open(before_dir / p.name).convert(\\"RGB\\"), \\"ONCE (render)\\")\\n        b = _label(Image.open(p).convert(\\"RGB\\"), \\"SONRA (Difix)\\")\\n        canvas = Image.new(\\"RGB\\", (a.width + b.width, a.height))\\n        canvas.paste(a, (0, 0))\\n        canvas.paste(b, (a.width, 0))\\n        canvas.save(out_dir / p.name)\\n\\n\\ndef _most_changed_box(a: np.ndarray, b: np.ndarray, size: int) -> tuple[int, int]:\\n    diff = torch.from_numpy(np.abs(a - b).mean(-1))[None, None]\\n    pooled = F.avg_pool2d(diff, size, stride=max(1, size // 4))\\n    idx = int(pooled.flatten().argmax())\\n    ny, nx = pooled.shape[-2:]\\n    step = max(1, size // 4)\\n    return (idx % nx) * step, (idx // nx) * step\\n\\n\\ndef comparison_sheet(before_dir: Path, after_dir: Path, out: Path, rows: int = 4) -> None:\\n    \\"\\"\\"Rows of: before | after | zoomed before | zoomed after, zoomed where the model changed most.\\"\\"\\"\\n    names = sorted(p.name for p in after_dir.glob(\\"*.png\\"))\\n    picks = [names[i] for i in np.linspace(0, len(names) - 1, min(rows, len(names))).round().astype(int)]\\n    tiles = []\\n    for name in picks:\\n        a = Image.open(before_dir / name).convert(\\"RGB\\")\\n        b = Image.open(after_dir / name).convert(\\"RGB\\")\\n        crop = max(32, min(a.width, a.height) // 3)\\n        x, y = _most_changed_box(np.asarray(a, np.float32), np.asarray(b, np.float32), crop)\\n        za = a.crop((x, y, x + crop, y + crop)).resize((a.height, a.height), Image.NEAREST)\\n        zb = b.crop((x, y, x + crop, y + crop)).resize((a.height, a.height), Image.NEAREST)\\n        d = ImageDraw.Draw(a)\\n        d.rectangle((x, y, x + crop, y + crop), outline=(255, 0, 0), width=max(2, a.height // 200))\\n        row = [_label(a, \\"once\\"), _label(b, \\"sonra\\"), _label(za, \\"once (zoom)\\"), _label(zb, \\"sonra (zoom)\\")]\\n        canvas = Image.new(\\"RGB\\", (sum(t.width for t in row), a.height))\\n        xo = 0\\n        for t in row:\\n            canvas.paste(t, (xo, 0))\\n            xo += t.width\\n        tiles.append(canvas)\\n    sheet = Image.new(\\"RGB\\", (tiles[0].width, sum(t.height for t in tiles)))\\n    yo = 0\\n    for t in tiles:\\n        sheet.paste(t, (0, yo))\\n        yo += t.height\\n    if sheet.width > 3200:\\n        sheet = sheet.resize((3200, round(sheet.height * 3200 / sheet.width)), Image.LANCZOS)\\n    sheet.save(out, quality=92)\\n\\n\\ndef _gray(path: Path) -> np.ndarray:\\n    return np.asarray(Image.open(path).convert(\\"L\\"), dtype=np.float32) / 255.0\\n\\n\\ndef _laplacian_var(g: np.ndarray) -> float:\\n    lap = -4 * g[1:-1, 1:-1] + g[:-2, 1:-1] + g[2:, 1:-1] + g[1:-1, :-2] + g[1:-1, 2:]\\n    return float(lap.var())\\n\\n\\ndef metrics(before_dir: Path, after_dir: Path) -> dict:\\n    names = sorted(p.name for p in after_dir.glob(\\"*.png\\"))\\n    change_psnr, sharp_ratio, flick_b, flick_a = [], [], [], []\\n    prev_b = prev_a = None\\n    for name in names:\\n        gb, ga = _gray(before_dir / name), _gray(after_dir / name)\\n        mse = float(((gb - ga) ** 2).mean())\\n        change_psnr.append(10 * np.log10(1.0 / max(mse, 1e-10)))\\n        sharp_ratio.append(_laplacian_var(ga) / max(_laplacian_var(gb), 1e-10))\\n        if prev_b is not None:\\n            flick_b.append(float(np.abs(gb - prev_b).mean()))\\n            flick_a.append(float(np.abs(ga - prev_a).mean()))\\n        prev_b, prev_a = gb, ga\\n    fb, fa = float(np.mean(flick_b)) if flick_b else 0.0, float(np.mean(flick_a)) if flick_a else 0.0\\n    return {\\n        \\"frames\\": len(names),\\n        \\"change_psnr_db\\": float(np.mean(change_psnr)),\\n        \\"sharpness_ratio\\": float(np.median(sharp_ratio)),\\n        \\"flicker_before\\": fb,\\n        \\"flicker_after\\": fa,\\n        \\"flicker_ratio\\": fa / fb if fb > 0 else float(\\"nan\\"),\\n    }\\n\\n\\ndef summary_markdown(m: dict, info: dict) -> str:\\n    fr = m[\\"flicker_ratio\\"]\\n    return \\"\\\\n\\".join([\\n        f\\"## Difix video testi: {info[\'video\']}\\",\\n        \\"\\",\\n        f\\"* Model: `{info[\'repo\']}`\\" + (f\\", referans: `{info[\'ref\']}`\\" if info.get(\\"ref\\") else \\"\\"),\\n        f\\"* İşlenen kare: {m[\'frames\']} ({info[\'size\'][0]}x{info[\'size\'][1]}), {info[\'sec_per_frame\']:.2f} s/kare\\",\\n        \\"\\",\\n        \\"| ölçüm | değer | nasıl okunur |\\",\\n        \\"|---|---|---|\\",\\n        f\\"| değişim (önce↔sonra PSNR) | {m[\'change_psnr_db\']:.1f} dB | 40+ dB: neredeyse dokunmamış, 25-35: belirgin düzeltme, <22: çok şey uydurmuş olabilir |\\",\\n        f\\"| keskinlik oranı | {m[\'sharpness_ratio\']:.2f}x | >1 daha keskin; tek başına kalite demek değil (gürültü de keskinlik sayılır) |\\",\\n        f\\"| titreme oranı | {fr:.2f}x | ~1 iyi; 1.2+ ise kareler arası titreme artmış (kare kare işlemenin bilinen yan etkisi) |\\",\\n        \\"\\",\\n        \\"Gerçek fotoğraf olmadığı için bu sayılar doğruluk ölçmez; asıl karar `side_by_side.mp4` ve \\"\\n        \\"`compare.jpg`\'deki zoom\'lara bakarak verilir (kırmızı kutu: modelin en çok değiştirdiği bölge).\\",\\n    ])\\n\\n\\ndef run(\\n    video: Path,\\n    out_dir: Path,\\n    model,\\n    repo: str,\\n    work_dir: Path = Path(\\"/content/difix_video\\"),\\n    ref_image: Path | None = None,\\n    long_edge: int = 1024,\\n    stride: int = 1,\\n    max_seconds: float = 0,\\n    device: str = \\"cuda\\",\\n    log=print,\\n) -> dict:\\n    video, out_dir = Path(video), Path(out_dir)\\n    info = probe(video)\\n    size = target_size(info[\\"width\\"], info[\\"height\\"], long_edge)\\n    work = Path(work_dir) / f\\"{video.stem}_{size[0]}x{size[1]}_s{stride}_t{max_seconds:g}\\"\\n    before, after = work / \\"before\\", work / f\\"after_{repo.split(\'/\')[-1]}\\"\\n    if not any(before.glob(\\"*.png\\")):\\n        n = extract_frames(video, before, size, stride, max_seconds)\\n        log(f\\"[video] {info[\'width\']}x{info[\'height\']} @ {info[\'fps\']:.2f} fps -> {n} frames at {size[0]}x{size[1]}\\")\\n    ref = load_ref(Path(ref_image), size, device) if ref_image else None\\n    spf = restore_frames(model, before, after, ref=ref, device=device, log=log)\\n    timing = after.parent / f\\"{after.name}_timing.json\\"\\n    if spf:\\n        timing.write_text(json.dumps({\\"sec_per_frame\\": spf}), encoding=\\"utf-8\\")\\n    elif timing.exists():\\n        spf = json.loads(timing.read_text(encoding=\\"utf-8\\"))[\\"sec_per_frame\\"]\\n\\n    out_dir.mkdir(parents=True, exist_ok=True)\\n    fps = info[\\"fps\\"] / stride\\n    encode(after, fps, out_dir / \\"restored.mp4\\")\\n    sbs = work / \\"side_by_side\\"\\n    side_by_side(before, after, sbs)\\n    encode(sbs, fps, out_dir / \\"side_by_side.mp4\\")\\n    comparison_sheet(before, after, out_dir / \\"compare.jpg\\")\\n    m = metrics(before, after)\\n    meta = {\\"video\\": video.name, \\"repo\\": repo, \\"ref\\": Path(ref_image).name if ref_image else None,\\n            \\"size\\": size, \\"stride\\": stride, \\"sec_per_frame\\": spf, **m}\\n    (out_dir / \\"metrics.json\\").write_text(json.dumps(meta, indent=2), encoding=\\"utf-8\\")\\n    md = summary_markdown(m, meta)\\n    (out_dir / \\"summary.md\\").write_text(md, encoding=\\"utf-8\\")\\n    log(md)\\n    return {\\"out_dir\\": out_dir, \\"work\\": work, \\"metrics\\": meta}\\n\\n\\ndef cleanup(work: Path) -> None:\\n    shutil.rmtree(work, ignore_errors=True)\\n", "lib/splat_restorer/workspace.py": "\\"\\"\\"Folder layout. Durable outputs live on Drive; scratch work on local disk.\\n\\nDrive (``root``)::\\n\\n    split.json                    scene split (written once, then reused)\\n    pairs/<scene>.tar             all pairs of one scene (atomic, = done marker)\\n    splats/<scene>/<recipe>.pt    degraded splats of *test* scenes\\n    runs/<run>/...                restorer / baseline checkpoints and logs\\n    results/...                   evaluation tables, contact sheets\\n\\nLocal (``local``)::\\n\\n    raw/<scene>/                  downloaded scene, deleted after use\\n    pairs/                        extracted pair tars for training / eval\\n\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport json\\nimport os\\nimport shutil\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\n\\n\\n@dataclass\\nclass Workspace:\\n    root: Path\\n    local: Path\\n\\n    def __post_init__(self):\\n        self.root = Path(self.root)\\n        self.local = Path(self.local)\\n        for d in (self.root, self.pairs_dir, self.splats_dir, self.runs_dir, self.results_dir, self.local):\\n            d.mkdir(parents=True, exist_ok=True)\\n\\n    @property\\n    def split_path(self) -> Path:\\n        return self.root / \\"split.json\\"\\n\\n    @property\\n    def pairs_dir(self) -> Path:\\n        return self.root / \\"pairs\\"\\n\\n    @property\\n    def splats_dir(self) -> Path:\\n        return self.root / \\"splats\\"\\n\\n    @property\\n    def runs_dir(self) -> Path:\\n        return self.root / \\"runs\\"\\n\\n    @property\\n    def results_dir(self) -> Path:\\n        return self.root / \\"results\\"\\n\\n    def pair_tar(self, scene: str) -> Path:\\n        return self.pairs_dir / f\\"{scene}.tar\\"\\n\\n    def raw_dir(self, scene: str) -> Path:\\n        return self.local / \\"raw\\" / scene\\n\\n    @property\\n    def local_pairs(self) -> Path:\\n        return self.local / \\"pairs\\"\\n\\n    def splat_path(self, scene: str, recipe: str) -> Path:\\n        return self.splats_dir / scene / f\\"{recipe}.pt\\"\\n\\n\\ndef atomic_copy(src: Path, dst: Path) -> None:\\n    \\"\\"\\"Copy to ``dst`` via a temp name so a crash never leaves a half file.\\"\\"\\"\\n    dst.parent.mkdir(parents=True, exist_ok=True)\\n    tmp = dst.with_name(dst.name + \\".partial\\")\\n    shutil.copyfile(src, tmp)\\n    os.replace(tmp, dst)\\n\\n\\ndef atomic_write_json(path: Path, data) -> None:\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    tmp = path.with_name(path.name + \\".partial\\")\\n    tmp.write_text(json.dumps(data, indent=2), encoding=\\"utf-8\\")\\n    os.replace(tmp, path)\\n"}')
root = Path("/content/tb_src")
for name, text in SOURCES.items():
    p = root / name
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
if str(root / "lib") not in sys.path:
    sys.path.insert(0, str(root / "lib"))
from splat_restorer import own_splat, train_ab, gsplat_testbed as tb
# Colab uyumluluğu: HF datasets paketiyle isim çakışması + pycolmap fork'unun numpy-2 hatası.
tb.fix_gsplat_install(Path("/content/gsplat_src/examples"))
print(f"{len(SOURCES)} dosya yazıldı")

In [ ]:
# @title Drive + veri
import shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
out_dir = Path(OUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)

raw = Path("/content/own_dataset")
if not (raw / ".unzipped").exists():
    print("Zip açılıyor...")
    shutil.rmtree(raw, ignore_errors=True)
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(raw)
    (raw / ".unzipped").touch()
cams_bin = next(raw.rglob("sparse/*/cameras.bin"))
src, model_name = cams_bin.parents[2], cams_bin.parent.name
assert model_name == "0", "gsplat eğiticisi sparse/0 bekliyor"
# Ölçüm için gerçek fotoğraflar: önceki testlerle aynı hazırlık (1920 uzun kenar).
data = own_splat.prepare_dataset(src, Path("/content/own_prepared_1920"), long_edge=1920,
                                 model_name=model_name, workers=os.cpu_count() or 4)
holdout = list(range(0, data.num_frames, TEST_EVERY))   # gsplat'in val kümesiyle aynı (sıralı isimler, i % N == 0)
train_ids = [i for i in range(data.num_frames) if i not in set(holdout)]
factors = sorted({FACTOR_OVERRIDE or tb.VARIANTS[v].factor for v in VARIANTS})
for f in factors:
    tb.prepare_factor_images(src, f, workers=os.cpu_count() or 4)
print(f"{data.num_frames} kare, saklanan {len(holdout)}; gsplat çözünürlük bölücüleri {factors}")

In [ ]:
# @title Eğit + ölç (varyant başına ~15-90 dk)
import dataclasses, time, lpips, torch
from IPython.display import Markdown, display
assert torch.cuda.is_available(), "GPU yok: Runtime → Change runtime type → GPU"
net_lpips = lpips.LPIPS(net="alex", verbose=False).cuda().eval()
flags_path = out_dir / "pose_flags.json"
results, failed = {}, []
for name in VARIANTS:
    v = tb.VARIANTS[name]
    res_path = out_dir / name / "result.json"
    if res_path.exists():
        results[name] = json.loads(res_path.read_text())
        portable = out_dir / name / "result_klasik.json"
        if portable.exists():
            results[name + " (klasik)"] = json.loads(portable.read_text())
        print(f"{name}: daha önce bitmiş, atlandı")
        continue
    if FACTOR_OVERRIDE:
        v = dataclasses.replace(v, factor=FACTOR_OVERRIDE)
    exclude = None
    if v.exclude_bad_frames:
        assert flags_path.exists(), "exclude_bad_frames için önce poz taraması gerekli: VARIANTS'ta başa 'default' koy."
        exclude = [data.names[i] for i in json.loads(flags_path.read_text())["bad_train"]]
        print(f"Eğitimden çıkarılan {len(exclude)} kare (saklanan 93 kare aynı kalır)")
    print(f"\n===== {name}: {v.note}")
    run_dir = Path("/content/gs_runs") / name
    shutil.rmtree(run_dir, ignore_errors=True)
    try:
        seconds = tb.run_variant(v, src, Path("/content/gsplat_src/examples"), run_dir, STEPS, TEST_EVERY, exclude)
    except RuntimeError as e:   # bir varyant çökerse diğerleri yine koşsun
        print(f"!!! {name} BAŞARISIZ, atlanıyor:\n{e}")
        (out_dir / name).mkdir(parents=True, exist_ok=True)
        (out_dir / name / "failed.txt").write_text(str(e), encoding="utf-8")
        failed.append(name)
        continue
    params, deg = tb.load_checkpoint(run_dir)
    (out_dir / name).mkdir(parents=True, exist_ok=True)
    frames = train_ab.evaluate_params(params, deg, data, holdout, None, net_lpips, out_dir / name / "renders",
                                      rasterize_mode=v.rasterize_mode, **v.render)
    if not flags_path.exists():
        print("Eğitim karelerinde poz hatası taranıyor...")
        tp = tb.frame_psnr(params, deg, data, train_ids, v.rasterize_mode, **v.render)
        flagged, bad_train, thr = tb.pose_failure_holdouts(tp, holdout)
        tb.save_json(flags_path, {"reference_variant": name, "threshold_db": thr, "flagged_holdout": flagged,
                                  "bad_train": bad_train, "train_psnr": {str(k): x for k, x in tp.items()}})
        print(f"Çöken eğitim kareleri ({len(bad_train)}): {bad_train}")
        print(f"Hariç tutulacak saklanan kareler: {flagged}")
    if SAVE_PLY:
        tb.save_ply(params, out_dir / name / "final.ply")
    for stats in (run_dir / "stats").glob("val_step*.json"):
        shutil.copy2(stats, out_dir / name / ("gsplat_" + stats.name))
    info = {"variant": dataclasses.asdict(v), "train_seconds": seconds, "splats": int(params["means"].shape[0]),
            "excluded_train_images": len(exclude or [])}
    results[name] = {"source": "gsplat", "info": info, "frames": frames, "gsplat_val": tb.gsplat_val(run_dir / "stats")}
    if v.portable_eval:   # standart 3DGS görüntüleyicide (ek render ayarı olmadan) nasıl göründüğü
        klasik = train_ab.evaluate_params(params, deg, data, holdout, None, net_lpips,
                                          out_dir / name / "renders_klasik", rasterize_mode="classic")
        results[name + " (klasik)"] = {"source": "gsplat", "info": info, "frames": klasik}
        tb.save_json(out_dir / name / "result_klasik.json", results[name + " (klasik)"])
    tb.save_json(res_path, results[name])
    del params
    torch.cuda.empty_cache()
    m = tb.summarise({name: results[name]}, holdout)[name]
    print(f"{name}: PSNR {m['psnr']:.2f}  LPIPS {m['lpips']:.4f}  {m['splats']:,} splat  {m['minutes']:.0f} dk")

In [ ]:
# @title Sonuçlar
from IPython.display import Image as IPImage
flags = json.loads(flags_path.read_text())
keep = [h for h in holdout if h not in set(flags["flagged_holdout"])]
for name, r in results.items():   # önceki koşulardan kalan sonuçlara gsplat'in kendi metriğini ekle
    if r.get("source") == "gsplat" and "gsplat_val" not in r and "(" not in name:
        r["gsplat_val"] = tb.gsplat_val(out_dir / name)
table = dict(results)
if APP_AB_DIR and Path(APP_AB_DIR).exists():   # app eğiticisinin A/B sonuçları, aynı kareler
    for p in sorted(Path(APP_AB_DIR).glob("*/result.json")):
        table["app:" + p.parent.name] = json.loads(p.read_text())
if failed:
    print("Başarısız varyantlar (ayrıntı Drive'da <varyant>/failed.txt):", failed)
summary = tb.summarise(table, keep)
md_text = tb.markdown(summary, len(holdout), flags["flagged_holdout"], flags["threshold_db"])
(out_dir / "summary.md").write_text(md_text, encoding="utf-8")
tb.save_json(out_dir / "summary.json", summary)
display(Markdown(md_text))
shown = {n: r for n, r in results.items() if "(" not in n}
sheets = train_ab.comparison_sheets(shown, data, out_dir, out_dir)
print("Sütunlar: foto | " + " | ".join(shown) + "; alt sıra: kırmızı kutunun zoom'u")
for p in sheets[:6]:
    display(IPImage(filename=str(p), width=1600))

## Nasıl okunur?

* **PSNR / LPIPS:** saklanan fotoğraflara karşı; **Δ** `default`'a göre. ~0,3 dB altındaki farklar koşudan koşuya oynayabilir.
* **Poz hatasız sütunlar:** yanlış kamera pozu yüzünden hiçbir yöntemin düzeltemeyeceği kareler çıkarılmış hâli;
  yöntemleri karşılaştırmak için daha güvenilir.
* `app:` satırları app'in kendi eğiticisi (önceki A/B testi). gsplat `default` ile `app:lr_decay` yakınsa, app eğiticisi
  öğrenme oranı düzeltmesiyle referans seviyesine gelmiş demektir.

Bana `summary.md` ve birkaç `compare_*.jpg` getirmen yeterli (Drive'da `OUT_DIR` altında).